# 06. Улучшение модели Semantic News Novelty

В этом ноутбуке дорабатывается уже представленная модель финального проекта.  
Основная идея: не переписывать весь пайплайн заново, а последовательно проверить несколько направлений улучшения и для каждого посчитать вклад в качество.

Задача проекта состоит не только в классификации отдельной новости. Полный pipeline выглядит так:

```text
clean-like news dataset
        ↓
семантическая векторизация
        ↓
кластеризация новостей в сюжеты
        ↓
previous-only признаки
        ↓
модель significant / not significant
        ↓
eval-like prediction CSV
```

Поэтому в этом ноутбуке эксперименты сгруппированы по трём уровням:

1. улучшение семантической векторизации;
2. улучшение кластеризации сюжетов;
3. улучшение финального шага novelty detection.


## 1. Что было сделано ранее

На предыдущем этапе была построена текущая основная модель, которая используется здесь как точка отсчёта.

Ключевые решения предыдущего этапа:

1. Зафиксирован единый входной формат модели, совместимый с `lenta_clean_news.csv`:
   `news_id`, `url`, `title`, `text`, `topic`, `tags`, `published_at`, `text_length`, `text_num_words`, `title_length`, `title_num_words`, `model_text`, `model_length`, `model_num_words`.
2. Зафиксирован единый выходной формат предсказаний, совместимый с golden/human-разметкой:
   `news_id`, `published_at`, `topic`, `title`, `text`, `cluster_id`, `novelty_label`, `comment`, `needs_review`.
3. Baseline был построен на `BAAI/bge-m3`, графовой кластеризации и rule-based novelty logic.
4. Затем baseline был заменён на supervised significance model: бинарная модель `significant / not significant` на previous-only признаках.
5. При обучении был исправлен leakage: строки, входящие в human golden subset, были исключены из silver train.
6. Для финальной модели использовался CatBoost с threshold `0.42`.

Ниже — сводка результатов из прошлого этапа. В этом ноутбуке эти значения используются только как историческая справка; актуальные метрики будут пересчитаны заново через единый evaluator.


In [2]:
import pandas as pd

previous_results = pd.DataFrame([
    {
        "stage": "Rule-based baseline",
        "accuracy": 0.4943,
        "significant_precision": 0.8222,
        "significant_recall": 0.5068,
        "significant_f1": 0.6271,
        "comment": "BGE-M3 + threshold graph + rule-based novelty; recall важных обновлений низковат",
    },
    {
        "stage": "CatBoost significance model, leakage fixed",
        "accuracy": 0.6322,
        "significant_precision": 0.8361,
        "significant_recall": 0.6986,
        "significant_f1": 0.7612,
        "comment": "Текущая основная модель; threshold=0.42, previous-only признаки",
    },
])
previous_results


,stage,accuracy,significant_precision,significant_recall,significant_f1,comment
0,Rule-based baseline,0.4943,0.8222,0.5068,0.6271,BGE-M3 + threshold graph + rule-based novelty;...
1,"CatBoost significance model, leakage fixed",0.6322,0.8361,0.6986,0.7612,"Текущая основная модель; threshold=0.42, previ..."


## 2. Imports и настройка путей

Текущая модель и вспомогательные функции вынесены в модуль `semantic_news_novelty`, чтобы не тащить в ноутбук большой класс с кодом.

Если структура проекта отличается, нужно поправить `PROJECT_ROOT` и пути ниже.


In [3]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd


PROJECT_ROOT = Path("..").resolve()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

%load_ext autoreload
%autoreload 2

from model.config import ExperimentPaths, SemanticNewsBaselineConfig, SignificanceModelConfig
from model.data import (
    load_clean_news,
    load_annotation,
    annotation_to_clean_like,
    remove_train_eval_leakage,
    save_prediction_csv,
)
from model.embeddings import SentenceTransformerEncoder, load_cached_embeddings, save_cached_embeddings
from model.clustering import (
    ThresholdGraphClusterer,
    MutualKnnTemporalClusterer,
    TemporalDecayGraphClusterer,
    OnlineLifecycleClusterer,
)
from model.features import build_previous_only_features, DEFAULT_FEATURE_COLUMNS
from model.significance_model import CatBoostSignificanceModel
from model.evaluation import evaluate_predictions, add_experiment_result, compact_metrics_table
from model.experiments import run_and_evaluate_experiment, attach_clusters_from_prediction


In [4]:
paths = ExperimentPaths(project_root=PROJECT_ROOT)

# При необходимости поправить под фактическую структуру проекта.
paths.clean_news_path, paths.golden_path, paths.silver_path


(WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/prepared/lenta_clean_news.csv'),
 WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/prepared/lenta_golden_set_human.csv'),
 WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/prepared/lenta_silver_set_llm.csv'))

## 3. Загрузка данных

Используются четыре набора:

- `clean_news` — подготовленные новости в clean-like формате;
- `candidate_news` — полный candidate pool, на котором надо строить кластеры и predictions;
- `golden` — human-разметка, используется только для evaluation;
- `silver` — LLM/silver-разметка без пересечений с golden, используется для обучения projection/final-step моделей.

Важно: для воспроизведения прошлого ноутбука нельзя кластеризовать только `golden` из 121 строки. Кластеры строятся на полном `lenta_golden_candidate_pool.csv`, а затем evaluator сравнивает predictions с `golden` по пересечению `news_id`.


In [5]:
clean_news = load_clean_news(paths.clean_news_path)
golden = load_annotation(paths.golden_path)
silver_raw = load_annotation(paths.silver_path)

candidate_pool_path = paths.prepared_dir / "lenta_golden_candidate_pool.csv"
if not candidate_pool_path.exists():
    raise FileNotFoundError(
        f"Не найден candidate pool: {candidate_pool_path}. "
        "Для воспроизведения старого пайплайна нужен файл lenta_golden_candidate_pool.csv"
    )

candidate_pool_raw = pd.read_csv(candidate_pool_path)

silver = remove_train_eval_leakage(silver_raw, golden, id_column="news_id")

print("clean_news:", clean_news.shape)
print("candidate pool raw:", candidate_pool_raw.shape)
print("golden:", golden.shape)
print("silver raw:", silver_raw.shape)
print("silver without golden leakage:", silver.shape)


clean_news: (99555, 14)
candidate pool raw: (3176, 16)
golden: (121, 10)
silver raw: (2036, 10)
silver without golden leakage: (1915, 10)


In [6]:
def prepare_candidate_news(
    candidate_pool_df: pd.DataFrame,
    source_clean_df: pd.DataFrame,
    id_column: str = "news_id",
) -> pd.DataFrame:
    """Prepare full candidate pool for end-to-end prediction.

    The old baseline notebook clustered the full `lenta_golden_candidate_pool.csv`,
    not only the human golden subset. This helper reconstructs a clean-like
    dataframe for the candidate pool, preferring rows from `clean_news` by
    `news_id` so that all technical columns are consistent.
    """
    candidate = candidate_pool_df.copy()
    source = source_clean_df.copy()

    candidate[id_column] = candidate[id_column].astype(str)
    source[id_column] = source[id_column].astype(str)

    candidate_ids = candidate[[id_column]].drop_duplicates()
    source_ids = set(source[id_column])

    if set(candidate_ids[id_column]).issubset(source_ids):
        prepared = source[source[id_column].isin(candidate_ids[id_column])].copy()
    else:
        # Fallback for annotation-like candidate pools that are not fully present in clean_news.
        # annotation_to_clean_like handles files with news_id/published_at/topic/title/text.
        prepared = annotation_to_clean_like(candidate, source_clean_df=source)

    prepared[id_column] = prepared[id_column].astype(str)
    prepared["published_at"] = pd.to_datetime(prepared["published_at"], errors="coerce")
    prepared["title"] = prepared["title"].fillna("").astype(str)
    prepared["text"] = prepared["text"].fillna("").astype(str)
    prepared["topic"] = prepared["topic"].fillna("unknown").astype(str)

    if "url" not in prepared.columns:
        prepared["url"] = ""
    if "tags" not in prepared.columns:
        prepared["tags"] = ""

    if "model_text" not in prepared.columns:
        prepared["model_text"] = (prepared["title"] + "\n\n" + prepared["text"]).str.strip()
    else:
        prepared["model_text"] = prepared["model_text"].fillna("").astype(str)
        empty_model_text_mask = prepared["model_text"].str.strip().eq("")
        prepared.loc[empty_model_text_mask, "model_text"] = (
            prepared.loc[empty_model_text_mask, "title"].fillna("").astype(str)
            + "\n\n"
            + prepared.loc[empty_model_text_mask, "text"].fillna("").astype(str)
        ).str.strip()

    # Recompute length columns to make annotation-like and clean-like inputs consistent.
    prepared["text_length"] = prepared["text"].str.len()
    prepared["text_num_words"] = prepared["text"].str.split().str.len()
    prepared["title_length"] = prepared["title"].str.len()
    prepared["title_num_words"] = prepared["title"].str.split().str.len()
    prepared["model_length"] = prepared["model_text"].str.len()
    prepared["model_num_words"] = prepared["model_text"].str.split().str.len()

    # Keep the old notebook behavior: stable chronological order.
    prepared = prepared.sort_values(["published_at", id_column], kind="mergesort").reset_index(drop=True)

    # Keep only columns that exist in clean_news, if possible.
    clean_like_columns = [col for col in source_clean_df.columns if col in prepared.columns]
    return prepared[clean_like_columns].copy()


golden_clean = annotation_to_clean_like(golden, source_clean_df=clean_news)
silver_clean = annotation_to_clean_like(silver, source_clean_df=clean_news)
candidate_news = prepare_candidate_news(candidate_pool_raw, source_clean_df=clean_news)

# Evaluation remains on golden; prediction/clustering runs on full candidate pool.
eval_news = candidate_news.copy()

print("golden_clean:", golden_clean.shape)
print("silver_clean:", silver_clean.shape)
print("candidate_news / eval_news:", eval_news.shape)
print("golden news_id covered by candidate pool:", golden["news_id"].astype(str).isin(eval_news["news_id"].astype(str)).mean())

eval_news.head(3)


golden_clean: (121, 14)
silver_clean: (1915, 14)
candidate_news / eval_news: (3176, 14)
golden news_id covered by candidate pool: 1.0


,news_id,url,title,text,topic,tags,published_at,text_length,text_num_words,title_length,title_num_words,model_text,model_length,model_num_words
0,88522,https://lenta.ru/news/2004/03/01/rouble/,Центробанк предлагает отвязать рубль от доллара,Россия может изменить нынешний механизм опреде...,Экономика,Все,2004-03-01,1172,168,47,6,Центробанк предлагает отвязать рубль от доллар...,1221,174
1,88525,https://lenta.ru/news/2004/03/01/katar/,Задержанные в Москве катарские борцы оказались...,Членами сборной Катара по греко-римской (класс...,Мир,Все,2004-03-01,1474,187,58,7,Задержанные в Москве катарские борцы оказались...,1534,194
2,88528,https://lenta.ru/news/2004/03/01/effects/,Американская киноакадемия признала лучшими спе...,"Джим Ригел, Джо Леттери, Рэндалл Уильям Кук и ...",Культура,Все,2004-03-01,731,108,76,8,Американская киноакадемия признала лучшими спе...,809,116


## 4. Единый evaluator и таблица экспериментов

Каждый эксперимент должен вернуть prediction CSV в простой eval-схеме. После этого одна и та же функция `evaluate_predictions` считает:

- coverage;
- pairwise cluster metrics;
- false merge rate;
- binary significance metrics.


In [7]:
results_table = None
paths.predictions_dir.mkdir(parents=True, exist_ok=True)
paths.artifacts_dir.mkdir(parents=True, exist_ok=True)

KEY_COLUMNS = [
    "experiment",
    "pairwise_f1",
    "false_merge_rate",
    "significance_accuracy",
    "significant_precision",
    "significant_recall",
    "significant_f1",
    "comment",
]


## 5. Embeddings cache

Текущая основная модель использовала `BAAI/bge-m3`.

Важно: embeddings ниже считаются для полного `candidate_news`, а не только для `golden`. Это нужно, чтобы кластеризация повторяла прошлый end-to-end пайплайн.


In [8]:
baseline_config = SemanticNewsBaselineConfig()

encoder = SentenceTransformerEncoder(
    model_name=baseline_config.embedding_model_name,
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
)

candidate_pool_embeddings_path = paths.significance_model_dir / "bge_m3_candidate_pool_embeddings.npz"
eval_embeddings_path = paths.significance_model_dir / "bge_m3_candidate_pool_embeddings_by_id.npz"
"""
eval_embeddings = encoder.encode_dataframe(
    eval_news,
    text_column="model_text",
    id_column="news_id",
    cache_path=eval_embeddings_path,
    force_recompute=False,
)

print(eval_embeddings.shape)

"""
eval_embeddings = encoder.encode_dataframe(
    eval_news,
    text_column="model_text",
    id_column="news_id",
    cache_path=candidate_pool_embeddings_path,
    force_recompute=not candidate_pool_embeddings_path.exists(),
)

print("eval_embeddings:", eval_embeddings.shape)
print("Candidate pool rows:", eval_news.shape)
print("Candidate pool embeddings path:", candidate_pool_embeddings_path)

if len(eval_news) != len(eval_embeddings):
    raise ValueError(
        f"eval_news and eval_embeddings length mismatch: "
        f"{len(eval_news)} != {len(eval_embeddings)}"
    )


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

eval_embeddings: (3176, 1024)
Candidate pool rows: (3176, 14)
Candidate pool embeddings path: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\significance_model\bge_m3_candidate_pool_embeddings.npz


## 6. Experiment 0 — воспроизведение текущей основной модели

Это контрольная точка. Она нужна, чтобы новый ноутбук начинался с воспроизводимого состояния.

Важное исправление относительно первой версии этого ноутбука: `exp_00_current_model` строит кластеры и predictions на полном `candidate_news`, как в прошлом ноутбуке. `golden` используется только для оценки. Если кластеризовать только 121 строки human golden, граф похожести разваливается почти на одни singleton-кластеры, и `pairwise_f1` становится бессмысленным.


In [9]:
from model.features import LEGACY_SIGNIFICANCE_FEATURE_COLUMNS

current_clusterer = ThresholdGraphClusterer(
    similarity_threshold=baseline_config.story_threshold,
    window_days=baseline_config.story_window_days,
)

current_model_config = SignificanceModelConfig(
    threshold=0.42,
    duplicate_threshold=0.90,
    review_margin=0.10,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
)

current_model = CatBoostSignificanceModel.load(
    model_path=paths.current_catboost_model_path,
    config_path=None,
    config=current_model_config,
)


In [10]:
# Альтернативный вариант, если сохранённого .cbm ещё нет: переобучить текущую CatBoost-модель на silver.
# Для финального отчёта лучше не смешивать это с exp_00_current_model, а явно назвать exp_00_retrained_current_model.

# silver_embeddings_path = paths.artifacts_dir / "silver_bge_m3_embeddings.npz"
# silver_embeddings = encoder.encode_dataframe(
#     silver_clean,
#     text_column="model_text",
#     id_column="news_id",
#     cache_path=silver_embeddings_path,
#     force_recompute=False,
# )
#
# silver_train = silver_clean.merge(silver[["news_id", "cluster_id"]], on="news_id", how="left")
# silver_features = build_previous_only_features(silver_train, silver_embeddings)
#
# current_model = CatBoostSignificanceModel(config=current_model_config)
# current_model.fit(silver_features, silver)
# current_model.save(paths.current_catboost_model_path, paths.current_catboost_config_path)


In [11]:
# ---------------------------------------------------------------------
# Experiment 00: текущая основная модель из предыдущего этапа
# ---------------------------------------------------------------------
# Важно:
# exp_00 должен воспроизводить старый pipeline.
# Поэтому cluster_id берём из старого baseline prediction-файла,
# а не пересобираем кластеры заново новым ThresholdGraphClusterer.
#
# Схема:
#   candidate pool
#     + saved baseline cluster_id
#     + legacy 18 previous-only features
#     + saved CatBoost model
#     -> predictions
#     -> evaluation on human golden subset
# ---------------------------------------------------------------------

from model.features import LEGACY_SIGNIFICANCE_FEATURE_COLUMNS

# На случай повторного запуска ячейки — удаляем старую строку exp_00 из таблицы.
if "results_table" not in globals() or results_table is None:
    results_table = pd.DataFrame()
elif not results_table.empty and "experiment" in results_table.columns:
    results_table = (
        results_table[results_table["experiment"] != "exp_00_current_model"]
        .copy()
        .reset_index(drop=True)
    )

# Убеждаемся, что wrapper модели ждёт legacy 18 признаков.
current_model.config = SignificanceModelConfig(
    threshold=0.42,
    duplicate_threshold=0.90,
    review_margin=0.10,
    feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
)
current_model.feature_columns = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)

print("Model feature names:")
print(current_model.model.feature_names_)

print("\nWrapper feature count:", len(current_model.feature_columns))
print("Wrapper feature columns:")
print(current_model.feature_columns)

# Старые baseline predictions из предыдущего ноутбука.
baseline_pred_path = paths.prepared_dir / "lenta_baseline_predictions.csv"

if not baseline_pred_path.exists():
    raise FileNotFoundError(
        f"Не найден старый baseline prediction file: {baseline_pred_path}"
    )

old_baseline_pred = pd.read_csv(baseline_pred_path)
old_baseline_pred["news_id"] = old_baseline_pred["news_id"].astype(str)

print("\nOld baseline predictions:", old_baseline_pred.shape)
print("Old baseline clusters:", old_baseline_pred["cluster_id"].nunique())

# Candidate pool / eval_news должен быть тем же пулом, что использовался
# для baseline prediction, а не только human golden subset.
eval_news_for_exp00 = eval_news.copy()
eval_news_for_exp00["news_id"] = eval_news_for_exp00["news_id"].astype(str)

# Подтягиваем cluster_id из старого baseline prediction-файла.
baseline_clusters = (
    old_baseline_pred[["news_id", "cluster_id"]]
    .drop_duplicates(subset=["news_id"], keep="first")
)

baseline_clustered_news = (
    eval_news_for_exp00
    .drop(columns=["cluster_id"], errors="ignore")
    .merge(baseline_clusters, on="news_id", how="left")
)

missing_clusters = baseline_clustered_news["cluster_id"].isna().sum()
if missing_clusters:
    raise ValueError(
        f"Для {missing_clusters} строк candidate pool не найден cluster_id "
        f"в {baseline_pred_path}. Проверь, что eval_news построен из того же "
        "lenta_golden_candidate_pool.csv."
    )

print("Baseline clustered news:", baseline_clustered_news.shape)
print("Baseline clustered news clusters:", baseline_clustered_news["cluster_id"].nunique())

# Строим те самые legacy 18 признаков, на которых обучалась сохранённая CatBoost-модель.
baseline_features = build_previous_only_features(
    news_df=baseline_clustered_news,
    embeddings=eval_embeddings,
)

missing_feature_columns = [
    col for col in current_model.feature_columns
    if col not in baseline_features.columns
]

if missing_feature_columns:
    raise ValueError(f"Не хватает feature columns: {missing_feature_columns}")

print("Baseline features:", baseline_features.shape)
print("Feature columns used:", len(current_model.feature_columns))

# Применяем сохранённую CatBoost-модель к старым baseline-кластерам.
pred_00 = current_model.predict_clustered_with_fallback(
    news_df=baseline_clustered_news,
    embeddings=eval_embeddings,
)

paths.predictions_dir.mkdir(parents=True, exist_ok=True)
pred_00_path = paths.predictions_dir / "exp_00_current_model.csv"
save_prediction_csv(pred_00, pred_00_path)

# Оцениваем только на human golden subset.
metrics_00 = evaluate_predictions(golden, pred_00)

"""
results_table = add_experiment_result(
    results_table,
    experiment_name="exp_00_current_model",
    metrics=metrics_00,
    prediction_path=str(pred_00_path),
    embedding_variant="BAAI/bge-m3",
    clustering="saved_baseline_predictions",
    novelty_variant="CatBoost previous-only",
    comment=(
        "Текущая основная модель из предыдущего этапа: "
        "старые baseline-кластеры + сохранённая CatBoost-модель"
    ),
)
"""

# Sanity-check: теперь на golden-пересечении не должно быть 119 кластеров из 121 строки.
print("\npred_00:", pred_00.shape)
print("pred_00 clusters:", pred_00["cluster_id"].nunique())

merged_00 = golden[["news_id", "cluster_id"]].copy()
merged_00["news_id"] = merged_00["news_id"].astype(str)

merged_00 = merged_00.merge(
    pred_00[["news_id", "cluster_id"]],
    on="news_id",
    suffixes=("_true", "_pred"),
)

print("\ngolden/pred merge:", merged_00.shape)
print("true clusters on golden:", merged_00["cluster_id_true"].nunique())
print("pred clusters on golden:", merged_00["cluster_id_pred"].nunique())

print("\nTop pred clusters on golden:")
print(merged_00["cluster_id_pred"].value_counts().head(10))

compact_metrics_table(results_table)

Model feature names:
['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17']

Wrapper feature count: 18
Wrapper feature columns:
['position_in_cluster', 'cluster_size_so_far', 'days_since_previous', 'days_since_cluster_start', 'max_prev_similarity', 'mean_prev_similarity', 'min_prev_similarity', 'top2_mean_similarity', 'top3_mean_similarity', 'last_prev_similarity', 'previous_centroid_similarity', 'previous_centroid_distance', 'title_jaccard_max', 'text_jaccard_max', 'shared_numbers_count', 'new_numbers_count', 'title_length', 'text_length']

Old baseline predictions: (3176, 9)
Old baseline clusters: 2607
Baseline clustered news: (3176, 15)
Baseline clustered news clusters: 2607
Baseline features: (3176, 22)
Feature columns used: 18

pred_00: (3176, 10)
pred_00 clusters: 2607

golden/pred merge: (121, 3)
true clusters on golden: 34
pred clusters on golden: 58

Top pred clusters on golden:
cluster_id_pred
baseline_cluster_00573    8
baseline_cl

""


In [12]:
from model.legacy_pipeline import (
    LegacyNewsNoveltyPipeline,
    LegacyNewsPipelineConfig,
)

raw_candidate_pool = pd.read_csv(paths.prepared_dir / "lenta_golden_candidate_pool.csv")

legacy_pipeline = LegacyNewsNoveltyPipeline(
    encoder=encoder,
    novelty_model=current_model,
    config=LegacyNewsPipelineConfig(
        embeddings_cache_path=(
            paths.artifacts_dir
            / "embeddings"
            / "bge_m3_candidate_pool_id_aligned.npz"
        ),
        story_threshold=0.82,
        story_window_days=14,
        use_topic_blocking=True,
        normalize_embeddings_for_clustering=True,
    ),
)

legacy_result = legacy_pipeline.predict(raw_candidate_pool)

pred_00b = legacy_result.predictions
old_pipeline_clustered_news = legacy_result.clustered_news
old_pipeline_embeddings = legacy_result.embeddings
legacy_clusterer = legacy_result.clusterer

print("Old pipeline predictions:", pred_00b.shape)
print("Old pipeline clusters:", pred_00b["cluster_id"].nunique())
print("Old pipeline graph edges:", legacy_clusterer.last_graph_edges_)
print("Old pipeline cluster count:", legacy_clusterer.last_cluster_count_)

pred_00b_path = paths.predictions_dir / "exp_00b_reproduced_old_pipeline.csv"
save_prediction_csv(pred_00b, pred_00b_path)

metrics_00b = evaluate_predictions(golden, pred_00b)

results_table = add_experiment_result(
    results_table,
    experiment_name="exp_00b_reproduced_old_pipeline",
    metrics=metrics_00b,
    prediction_path=str(pred_00b_path),
    embedding_variant="BAAI/bge-m3 id-aligned",
    clustering="LegacyBaselineGraphClusterer",
    novelty_variant="CatBoost previous-only + first-item fallback",
    comment=(
        "Полностью воспроизводимый pipeline: legacy input preparation, "
        "id-aware BGE-M3 embeddings, legacy graph clustering, CatBoost fallback"
    ),
)

compact_metrics_table(results_table)

Рёбер в графе похожести: 914
Количество кластеров: 2607
Old pipeline predictions: (3176, 10)
Old pipeline clusters: 2607
Old pipeline graph edges: 914
Old pipeline cluster count: 2607


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.0,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...


In [13]:
from model.legacy_clustering import compare_saved_and_reproduced_clusters
cluster_reproduction_report = compare_saved_and_reproduced_clusters(
    saved_pred=old_baseline_pred,
    reproduced_pred=pred_00b,
)

cluster_reproduction_report

{'rows': 3176,
 'saved_clusters': 2607,
 'reproduced_clusters': 2607,
 'tp_same_pairs': 1946,
 'fp_false_merge_pairs': 0,
 'fn_missed_pairs': 0,
 'tn_diff_pairs': 5039954,
 'pairwise_precision': 1.0,
 'pairwise_recall': 1.0,
 'pairwise_f1': 1.0}

## 7. Experiment 1 — улучшение семантической векторизации

Идея: не делать тяжёлый full fine-tuning `BAAI/bge-m3`, а обучить лёгкий projection/adaptor поверх уже готовых эмбеддингов.

Положительные пары:

- новости из одного размеченного кластера;
- особенно `minor`/`duplicate`, если такие пары есть;
- при необходимости — парафразы.

Hard negative пары:

- один topic;
- близкая семантика;
- разные `cluster_id`.

После projection/adaptor пересчитываем кластеризацию и previous-only признаки. Так можно проверить, улучшилась ли именно семантическая геометрия.


In [14]:
from model.embedding_tuning import (
    PairDatasetConfig,
    build_contrastive_pairs,
    train_projection_adapter,
    apply_projection_adapter,
)

# Для обучения projection используем silver без golden leakage.
silver_embeddings_path = paths.artifacts_dir / "silver_bge_m3_embeddings.npz"
silver_embeddings = encoder.encode_dataframe(
    silver_clean,
    text_column="model_text",
    id_column="news_id",
    cache_path=silver_embeddings_path,
    force_recompute=False,
)

pair_config = PairDatasetConfig(
    max_positive_pairs_per_cluster=20,
    max_negative_pairs_per_topic=50,
    hard_negative_min_similarity=0.70,
    random_state=42,
)

pairs_df = build_contrastive_pairs(
    labels_df=silver,
    embeddings=silver_embeddings,
    id_order=silver_clean["news_id"].astype(str),
    config=pair_config,
)

pairs_df["target"].value_counts(dropna=False), pairs_df.head()


Building positive pairs by cluster:   0%|          | 0/1736 [00:00<?, ?it/s]

Building hard negative pairs by topic:   0%|          | 0/3 [00:00<?, ?it/s]

(target
  1    226
 -1    111
 Name: count, dtype: int64,
   news_id_a news_id_b  target  base_similarity topic cluster_id_a cluster_id_b
 0     88525     88617       1         0.271778   Мир  prelim_0002  prelim_0002
 1     88534     88547       1         0.239546   Мир  prelim_0004  prelim_0004
 2     88551     88692       1         0.271153   Мир  prelim_0007  prelim_0007
 3     88563     88594       1         0.250214   Мир  prelim_0010  prelim_0010
 4     88593     88628       1         0.371193   Мир  prelim_0017  prelim_0017)

In [15]:
projection_model, projection_history = train_projection_adapter(
    embeddings=silver_embeddings,
    id_order=silver_clean["news_id"].astype(str),
    pairs_df=pairs_df,
    hidden_dim=512,
    epochs=10,
    batch_size=256,
    learning_rate=1e-3,
    margin=0.2,
    random_state=42,
)

projection_history.tail()


,epoch,loss
5,6,0.264750
6,7,0.281542
7,8,0.264610
8,9,0.257797
9,10,0.254372


In [16]:
eval_embeddings_projected = apply_projection_adapter(projection_model, eval_embeddings)

pred_01, results_table, metrics_01 = run_and_evaluate_experiment(
    experiment_name="exp_01_embedding_projection",
    news_df=eval_news,
    embeddings=eval_embeddings_projected,
    clusterer=current_clusterer,
    novelty_model=current_model,
    golden_df=golden,
    predictions_dir=paths.predictions_dir,
    results_table=results_table,
    embedding_variant="BGE-M3 + projection adapter",
    clustering="threshold_graph",
    novelty_variant="current CatBoost",
    comment="Проверка, улучшает ли contrastive projection геометрию эмбеддингов",
)

compact_metrics_table(results_table)


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."


### 7.1. Вариант с парафразами

Если есть отдельный файл с парафразами, его можно подключить как дополнительные positive pairs.

Ожидаемый формат:

```text
news_id_a,news_id_b,target
123,456,1
```

Либо можно сгенерировать парафразы для части silver/golden train и затем считать `оригинал ↔ парафраз` положительной парой. Важно не использовать golden evaluation строки для обучения projection, иначе снова появится leakage.


In [17]:
# paraphrase_pairs_path = paths.prepared_dir / "paraphrase_pairs.csv"
# if paraphrase_pairs_path.exists():
#     paraphrase_pairs = pd.read_csv(paraphrase_pairs_path)
#     pairs_with_paraphrases = pd.concat([pairs_df, paraphrase_pairs], ignore_index=True)
# else:
#     pairs_with_paraphrases = pairs_df.copy()
#
# projection_model_para, projection_history_para = train_projection_adapter(
#     embeddings=silver_embeddings,
#     id_order=silver_clean["news_id"].astype(str),
#     pairs_df=pairs_with_paraphrases,
#     hidden_dim=512,
#     epochs=10,
#     batch_size=256,
#     learning_rate=1e-3,
#     margin=0.2,
#     random_state=42,
# )
#
# eval_embeddings_projected_para = apply_projection_adapter(projection_model_para, eval_embeddings)


## 8. Experiment 2 — улучшение кластеризации

Текущая кластеризация строит similarity graph и берёт connected components. Это простой и понятный baseline, но у него есть риск chaining-effect:

```text
A похожа на B
B похожа на C
A уже не очень похожа на C
но все три новости оказываются в одном компоненте
```

Для новостных сюжетов это опасно: можно случайно склеить соседние, но разные события. Поэтому проверяем более строгие варианты.


In [18]:
cluster_experiments = [
    (
        "exp_02a_mutual_knn_clustering",
        MutualKnnTemporalClusterer(similarity_threshold=0.80, window_days=14, k=5),
        "mutual_kNN temporal graph",
        "Снижает chaining-effect: связь засчитывается только при mutual top-k",
    ),
    (
        "exp_02b_temporal_decay_clustering",
        TemporalDecayGraphClusterer(similarity_threshold=0.82, window_days=30, decay_per_day=0.002),
        "temporal decay graph",
        "Похожесть штрафуется за временную дистанцию",
    ),
    (
        "exp_02c_online_lifecycle_clustering",
        OnlineLifecycleClusterer(similarity_threshold=0.82, lifecycle_days=14),
        "online lifecycle clustering",
        "Старые кластеры перестают принимать новые новости после периода неактивности",
    ),
]

for exp_name, clusterer, clustering_name, comment in cluster_experiments:
    _, results_table, _ = run_and_evaluate_experiment(
        experiment_name=exp_name,
        news_df=eval_news,
        embeddings=eval_embeddings,
        clusterer=clusterer,
        novelty_model=current_model,
        golden_df=golden,
        predictions_dir=paths.predictions_dir,
        results_table=results_table,
        embedding_variant="BAAI/bge-m3",
        clustering=clustering_name,
        novelty_variant="current CatBoost",
        comment=comment,
    )

compact_metrics_table(results_table)


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...


## 9. Experiment 3 — признаки относительно центра и top-k previous similarities

В модуле `features.py` уже добавлены признаки, которые раньше были только идеей:

- `prev_centroid_sim`;
- `prev_centroid_distance`;
- `top3_prev_sim_mean`;
- `top5_prev_sim_mean`;
- `cluster_density_last_3_days`;
- `cluster_density_last_7_days`;
- fallback-признаки по предыдущим новостям того же topic за 30 дней.

Этот эксперимент честнее делать с переобучением CatBoost, потому что feature set изменился.


In [19]:
# Готовим train features на silver-разметке.
# Здесь используем cluster_id из silver-разметки: для обучения финальной модели это нормально,
# потому что задача финального шага — научиться significant/not significant внутри уже заданного сюжета.

silver_train = silver_clean.merge(silver[["news_id", "cluster_id"]], on="news_id", how="left")
silver_features = build_previous_only_features(silver_train, silver_embeddings)

# Eval/prediction features строим на полном candidate pool после кластеризации текущим clusterer.
# Evaluation ниже всё равно считается только на пересечении с golden по news_id.
eval_clustered = eval_news.copy()
eval_clustered["cluster_id"] = current_clusterer.fit_predict(eval_news, eval_embeddings)
eval_features = build_previous_only_features(eval_clustered, eval_embeddings)

expanded_catboost = CatBoostSignificanceModel(config=current_model_config)
expanded_catboost.fit(
    silver_features,
    silver,
    catboost_params={
        "iterations": 600,
        "depth": 5,
        "learning_rate": 0.04,
        "l2_leaf_reg": 6.0,
        "verbose": 100,
    },
)

pred_03 = expanded_catboost.predict_eval_schema(eval_clustered, eval_features)
save_prediction_csv(pred_03, paths.predictions_dir / "exp_03_expanded_features_catboost.csv")
metrics_03 = evaluate_predictions(golden, pred_03)
results_table = add_experiment_result(
    results_table,
    experiment_name="exp_03_expanded_features_catboost",
    metrics=metrics_03,
    prediction_path=str(paths.predictions_dir / "exp_03_expanded_features_catboost.csv"),
    embedding_variant="BAAI/bge-m3",
    clustering="threshold_graph",
    novelty_variant="CatBoost retrained previous-only features",
    comment="Переобучение CatBoost на том же feature pipeline; prediction на полном candidate pool",
)

compact_metrics_table(results_table)


0:	learn: 0.8686869	total: 138ms	remaining: 1m 22s
100:	learn: 0.9078498	total: 239ms	remaining: 1.18s
200:	learn: 0.9851852	total: 344ms	remaining: 684ms
300:	learn: 1.0000000	total: 455ms	remaining: 452ms
400:	learn: 1.0000000	total: 563ms	remaining: 279ms
500:	learn: 1.0000000	total: 675ms	remaining: 133ms
599:	learn: 1.0000000	total: 780ms	remaining: 0us


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Переобучение CatBoost на том же feature pipeli...


## 10. Experiment 4 — нейросетевая модель для финального шага

CatBoost уже достаточно хорошо подходит для табличных previous-only признаков, поэтому нейросетка здесь не гарантирует прирост. Но как ветка из лекционной схемы «замена модели» это корректный эксперимент.

Сравнение будет честным: MLP получает те же признаки, что и CatBoost в эксперименте 3.


In [20]:
from model.neural_model import TorchTabularSignificanceMLP, MLPConfig

mlp_model = TorchTabularSignificanceMLP(
    model_config=current_model_config,
    mlp_config=MLPConfig(
        hidden_dim=128,
        dropout=0.10,
        epochs=40,
        batch_size=128,
        learning_rate=1e-3,
        weight_decay=1e-4,
        random_state=42,
    ),
)

mlp_history = mlp_model.fit(silver_features, silver)
mlp_history.tail()


,epoch,loss
35,36,0.469841
36,37,0.435082
37,38,0.474040
38,39,0.468476
39,40,0.436836


In [21]:
mlp_labelled = mlp_model.predict_labels(eval_features)

pred_04 = eval_clustered[["news_id", "published_at", "topic", "title", "text"]].merge(
    mlp_labelled[["news_id", "cluster_id", "novelty_label", "needs_review"]],
    on="news_id",
    how="left",
)
pred_04["comment"] = ""

save_prediction_csv(pred_04, paths.predictions_dir / "exp_04_mlp_final_step.csv")
metrics_04 = evaluate_predictions(golden, pred_04)
results_table = add_experiment_result(
    results_table,
    experiment_name="exp_04_mlp_final_step",
    metrics=metrics_04,
    prediction_path=str(paths.predictions_dir / "exp_04_mlp_final_step.csv"),
    embedding_variant="BAAI/bge-m3",
    clustering="threshold_graph",
    novelty_variant="MLP expanded previous-only features",
    comment="Нейросетевая замена финальной CatBoost-модели; prediction на полном candidate pool",
)

compact_metrics_table(results_table)


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Переобучение CatBoost на том же feature pipeli...
6,exp_04_mlp_final_step,0.000000,1.000000,0.793103,0.839506,0.931507,0.883117,Нейросетевая замена финальной CatBoost-модели;...


In [22]:
# ---------------------------------------------------------------------
# Prepare training data on reproduced old clustering
# ---------------------------------------------------------------------
# Используем ту же кластеризацию и embeddings, что в exp_00b.
# Это делает exp_03b / exp_04b честными: меняется только novelty model.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import GroupShuffleSplit

from model.config import SignificanceModelConfig
from model.features import (
    FEATURE_COLUMNS,
    LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    build_legacy_significance_features,
)
from model.significance_model import CatBoostSignificanceModel


def normalize_news_id_for_join(series: pd.Series) -> pd.Series:
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def remove_same_experiment_row(
    results_table: pd.DataFrame,
    experiment_name: str,
) -> pd.DataFrame:
    if results_table is None or results_table.empty:
        return pd.DataFrame()

    if "experiment" not in results_table.columns:
        return results_table.copy()

    return (
        results_table[results_table["experiment"] != experiment_name]
        .copy()
        .reset_index(drop=True)
    )


def find_best_threshold(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    thresholds: np.ndarray | None = None,
) -> tuple[float, float]:
    if thresholds is None:
        thresholds = np.linspace(0.20, 0.80, 61)

    best_threshold = 0.42
    best_f1 = -1.0

    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)
        score = f1_score(y_true, y_pred, zero_division=0)

        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)

    return best_threshold, float(best_f1)


if "old_pipeline_clustered_news" not in globals():
    raise RuntimeError("Сначала выполни exp_00b и получи old_pipeline_clustered_news.")

if "old_pipeline_embeddings" not in globals():
    raise RuntimeError("Сначала выполни exp_00b и получи old_pipeline_embeddings.")

if len(old_pipeline_clustered_news) != len(old_pipeline_embeddings):
    raise ValueError(
        f"old_pipeline_clustered_news / old_pipeline_embeddings length mismatch: "
        f"{len(old_pipeline_clustered_news)} != {len(old_pipeline_embeddings)}"
    )

print("Fixed clustered news:", old_pipeline_clustered_news.shape)
print("Fixed embeddings:", old_pipeline_embeddings.shape)
print("Fixed clusters:", old_pipeline_clustered_news["cluster_id"].nunique())

# Берём silver labels из текущего ноутбука, если они уже загружены.
if "silver_df" in globals():
    silver_labels = silver_df.copy()
    print("Using silver_df from notebook state:", silver_labels.shape)
else:
    # Запасной поиск частых имён.
    candidate_silver_paths = [
        paths.prepared_dir / "lenta_silver_set_llm.csv",
        paths.prepared_dir / "lenta_silver_set.csv",
        paths.prepared_dir / "lenta_silver_annotations.csv",
        paths.prepared_dir / "lenta_golden_set_llm.csv",
    ]

    existing_silver_paths = [p for p in candidate_silver_paths if p.exists()]

    if not existing_silver_paths:
        raise FileNotFoundError(
            "Не найден silver_df в памяти и не найден silver label file. "
            "Загрузи silver_df или укажи путь к silver-разметке вручную."
        )

    silver_labels = pd.read_csv(existing_silver_paths[0])
    print("Loaded silver labels:", existing_silver_paths[0], silver_labels.shape)

silver_labels["news_id"] = normalize_news_id_for_join(silver_labels["news_id"])

if "novelty_label" in silver_labels.columns:
    label_column = "novelty_label"
elif "source_novelty_label" in silver_labels.columns:
    label_column = "source_novelty_label"
else:
    raise ValueError("silver labels must contain novelty_label or source_novelty_label")

silver_labels[label_column] = (
    silver_labels[label_column]
    .fillna("")
    .astype(str)
    .str.strip()
)

# Строим признаки именно на фиксированной старой кластеризации.
fixed_features = build_legacy_significance_features(
    news_df=old_pipeline_clustered_news,
    embeddings=old_pipeline_embeddings,
)

fixed_features["news_id"] = normalize_news_id_for_join(fixed_features["news_id"])

if "cluster_id" not in fixed_features.columns:
    fixed_features = fixed_features.merge(
        old_pipeline_clustered_news[["news_id", "cluster_id"]],
        on="news_id",
        how="left",
    )

train_frame = fixed_features.merge(
    silver_labels[["news_id", label_column]],
    on="news_id",
    how="inner",
)

train_frame[label_column] = (
    train_frame[label_column]
    .fillna("")
    .astype(str)
    .str.strip()
)

trainable_labels = {"significant", "minor", "duplicate"}

train_frame = train_frame[
    train_frame[label_column].isin(trainable_labels)
].copy()

# Убираем golden из train, чтобы не было leakage.
golden_ids = set(normalize_news_id_for_join(golden["news_id"]))

before_leakage_filter = len(train_frame)

train_frame = train_frame[
    ~train_frame["news_id"].isin(golden_ids)
].copy()

after_leakage_filter = len(train_frame)

train_frame["is_significant"] = (
    train_frame[label_column].eq("significant").astype(int)
)

missing_features = [
    column
    for column in LEGACY_SIGNIFICANCE_FEATURE_COLUMNS
    if column not in train_frame.columns
]

if missing_features:
    raise ValueError(f"Missing feature columns: {missing_features}")

print("Trainable rows before golden filter:", before_leakage_filter)
print("Trainable rows after golden filter:", after_leakage_filter)
print("Label distribution:")
print(train_frame[label_column].value_counts())
print("\nBinary target distribution:")
print(train_frame["is_significant"].value_counts())

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, val_idx = next(
    splitter.split(
        train_frame,
        train_frame["is_significant"],
        groups=train_frame["cluster_id"].astype(str),
    )
)

train_part = train_frame.iloc[train_idx].copy()
val_part = train_frame.iloc[val_idx].copy()

X_train = train_part[FEATURE_COLUMNS]
y_train = train_part["is_significant"].astype(int).to_numpy()

X_val = val_part[FEATURE_COLUMNS]
y_val = val_part["is_significant"].astype(int).to_numpy()

print("\nTrain:", X_train.shape)
print(pd.Series(y_train).value_counts())

print("\nValidation:", X_val.shape)
print(pd.Series(y_val).value_counts())

Fixed clustered news: (3176, 17)
Fixed embeddings: (3176, 1024)
Fixed clusters: 2607
Loaded silver labels: E:\ML\Projects\Git\news-flow-analysis\data\prepared\lenta_silver_set_llm.csv (2036, 9)
Trainable rows before golden filter: 266
Trainable rows after golden filter: 179
Label distribution:
novelty_label
significant    133
minor           42
duplicate        4
Name: count, dtype: int64

Binary target distribution:
is_significant
1    133
0     46
Name: count, dtype: int64

Train: (147, 18)
1    111
0     36
Name: count, dtype: int64

Validation: (32, 18)
1    22
0    10
Name: count, dtype: int64


In [23]:
print("CatBoost-like variables:")
print([name for name in globals() if "catboost" in name.lower() or "boost" in name.lower()])

print("\nMLP-like variables:")
print([name for name in globals() if "mlp" in name.lower()])

print("\nModel-like variables with predict_proba:")
print([
    name
    for name, value in globals().items()
    if hasattr(value, "predict_proba")
])

CatBoost-like variables:
['CatBoostSignificanceModel', 'expanded_catboost']

MLP-like variables:
['TorchTabularSignificanceMLP', 'MLPConfig', 'mlp_model', 'mlp_history', 'mlp_labelled']

Model-like variables with predict_proba:
['CatBoostSignificanceModel', 'current_model', 'expanded_catboost', 'TorchTabularSignificanceMLP', 'mlp_model']


In [24]:
# ---------------------------------------------------------------------
# Experiment 03b: retrained CatBoost on fixed old clustering
# ---------------------------------------------------------------------
# Обучаем CatBoost с нуля на legacy features, построенных поверх
# воспроизведённой baseline-кластеризации exp_00b.
# ---------------------------------------------------------------------

from catboost import CatBoostClassifier

experiment_name = "exp_03b_catboost_retrained_fixed_old_clustering"

catboost_params = {
    "iterations": 500,
    "depth": 5,
    "learning_rate": 0.05,
    "l2_leaf_reg": 5.0,
    "loss_function": "Logloss",
    "eval_metric": "F1",
    "random_seed": 42,
    "verbose": 100,
}

catboost_candidate = CatBoostClassifier(**catboost_params)

catboost_candidate.fit(
    X_train,
    y_train,
    eval_set=(X_val, y_val),
    use_best_model=True,
)

val_proba = catboost_candidate.predict_proba(X_val)[:, 1]

best_threshold_03b, best_val_f1_03b = find_best_threshold(
    y_true=y_val,
    y_proba=val_proba,
)

print("Best validation threshold:", best_threshold_03b)
print("Best validation F1:", best_val_f1_03b)

val_pred = (val_proba >= best_threshold_03b).astype(int)

print("\nValidation report:")
print(
    classification_report(
        y_val,
        val_pred,
        target_names=["not_significant", "significant"],
        zero_division=0,
    )
)

# Финальная модель обучается на всём train_frame после выбора threshold.
feature_columns = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)
X_full = train_frame[feature_columns]
y_full = train_frame["is_significant"].astype(int).to_numpy()

catboost_final = CatBoostClassifier(**catboost_params)
catboost_final.fit(X_full, y_full)

catboost_03b_model = CatBoostSignificanceModel(
    config=SignificanceModelConfig(
        threshold=best_threshold_03b,
        duplicate_threshold=0.90,
        review_margin=0.10,
        feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    ),
    model=catboost_final,
)
catboost_03b_model.feature_columns = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)

pred_03b = catboost_03b_model.predict_clustered_with_fallback(
    news_df=old_pipeline_clustered_news,
    embeddings=old_pipeline_embeddings,
)

pred_03b_path = paths.predictions_dir / f"{experiment_name}.csv"
save_prediction_csv(pred_03b, pred_03b_path)

metrics_03b = evaluate_predictions(golden, pred_03b)

results_table = remove_same_experiment_row(results_table, experiment_name)

results_table = add_experiment_result(
    results_table,
    experiment_name=experiment_name,
    metrics=metrics_03b,
    prediction_path=str(pred_03b_path),
    embedding_variant="BAAI/bge-m3 id-aligned",
    clustering="fixed LegacyBaselineGraphClusterer from exp_00b",
    novelty_variant="Retrained CatBoost + first-item fallback",
    comment=(
        "CatBoost обучен с нуля на legacy features, построенных поверх "
        "воспроизведённой baseline-кластеризации exp_00b"
    ),
)

print("pred_03b:", pred_03b.shape)
print("pred_03b clusters:", pred_03b["cluster_id"].nunique())
print("pred_03b novelty distribution:")
print(pred_03b["novelty_label"].replace("", "<first_item>").value_counts(dropna=False))

compact_metrics_table(results_table)

0:	learn: 0.9391304	test: 0.8627451	best: 0.8627451 (0)	total: 1.51ms	remaining: 754ms
100:	learn: 0.9779736	test: 0.8400000	best: 0.8627451 (0)	total: 107ms	remaining: 423ms
200:	learn: 1.0000000	test: 0.8400000	best: 0.8627451 (0)	total: 212ms	remaining: 316ms
300:	learn: 1.0000000	test: 0.8400000	best: 0.8627451 (0)	total: 317ms	remaining: 210ms
400:	learn: 1.0000000	test: 0.7916667	best: 0.8627451 (0)	total: 423ms	remaining: 104ms
499:	learn: 1.0000000	test: 0.7916667	best: 0.8627451 (0)	total: 534ms	remaining: 0us

bestTest = 0.862745098
bestIteration = 0

Shrink model to first 1 iterations.
Best validation threshold: 0.48000000000000004
Best validation F1: 0.8627450980392157

Validation report:
                 precision    recall  f1-score   support

not_significant       1.00      0.30      0.46        10
    significant       0.76      1.00      0.86        22

       accuracy                           0.78        32
      macro avg       0.88      0.65      0.66        32
   

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Переобучение CatBoost на том же feature pipeli...
6,exp_04_mlp_final_step,0.000000,1.000000,0.793103,0.839506,0.931507,0.883117,Нейросетевая замена финальной CatBoost-модели;...
7,exp_03b_catboost_retrained_fixed_old_clustering,0.769737,0.000000,0.712644,0.875000,0.767123,0.817518,"CatBoost обучен с нуля на legacy features, пос..."


In [25]:
# ---------------------------------------------------------------------
# Experiment 04b: retrained MLP on fixed old clustering
# ---------------------------------------------------------------------
# Обучаем MLP с нуля на тех же legacy features.
# Кластеризация фиксирована и берётся из exp_00b.
# ---------------------------------------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier

experiment_name = "exp_04b_mlp_retrained_fixed_old_clustering"

mlp_candidate = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPClassifier(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                alpha=1e-3,
                learning_rate_init=1e-3,
                max_iter=500,
                early_stopping=True,
                validation_fraction=0.15,
                random_state=42,
            ),
        ),
    ]
)

mlp_candidate.fit(X_train, y_train)

val_proba = mlp_candidate.predict_proba(X_val)[:, 1]

best_threshold_04b, best_val_f1_04b = find_best_threshold(
    y_true=y_val,
    y_proba=val_proba,
)

print("Best validation threshold:", best_threshold_04b)
print("Best validation F1:", best_val_f1_04b)

val_pred = (val_proba >= best_threshold_04b).astype(int)

print("\nValidation report:")
print(
    classification_report(
        y_val,
        val_pred,
        target_names=["not_significant", "significant"],
        zero_division=0,
    )
)

# Финальная модель обучается на всём train_frame после выбора threshold.
feature_columns = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)
X_full = train_frame[feature_columns]
y_full = train_frame["is_significant"].astype(int).to_numpy()

mlp_final = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "mlp",
            MLPClassifier(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                alpha=1e-3,
                learning_rate_init=1e-3,
                max_iter=500,
                early_stopping=True,
                validation_fraction=0.15,
                random_state=42,
            ),
        ),
    ]
)

mlp_final.fit(X_full, y_full)

mlp_04b_model = CatBoostSignificanceModel(
    config=SignificanceModelConfig(
        threshold=best_threshold_04b,
        duplicate_threshold=0.90,
        review_margin=0.10,
        feature_columns=LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    ),
    model=mlp_final,
)
mlp_04b_model.feature_columns = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)

pred_04b = mlp_04b_model.predict_clustered_with_fallback(
    news_df=old_pipeline_clustered_news,
    embeddings=old_pipeline_embeddings,
)

pred_04b_path = paths.predictions_dir / f"{experiment_name}.csv"
save_prediction_csv(pred_04b, pred_04b_path)

metrics_04b = evaluate_predictions(golden, pred_04b)

results_table = remove_same_experiment_row(results_table, experiment_name)

results_table = add_experiment_result(
    results_table,
    experiment_name=experiment_name,
    metrics=metrics_04b,
    prediction_path=str(pred_04b_path),
    embedding_variant="BAAI/bge-m3 id-aligned",
    clustering="fixed LegacyBaselineGraphClusterer from exp_00b",
    novelty_variant="Retrained MLP + first-item fallback",
    comment=(
        "MLP обучен с нуля на legacy features, построенных поверх "
        "воспроизведённой baseline-кластеризации exp_00b"
    ),
)

print("pred_04b:", pred_04b.shape)
print("pred_04b clusters:", pred_04b["cluster_id"].nunique())
print("pred_04b novelty distribution:")
print(pred_04b["novelty_label"].replace("", "<first_item>").value_counts(dropna=False))

compact_metrics_table(results_table)

Best validation threshold: 0.36000000000000004
Best validation F1: 0.8301886792452831

Validation report:
                 precision    recall  f1-score   support

not_significant       1.00      0.10      0.18        10
    significant       0.71      1.00      0.83        22

       accuracy                           0.72        32
      macro avg       0.85      0.55      0.51        32
   weighted avg       0.80      0.72      0.63        32

pred_04b: (3176, 10)
pred_04b clusters: 2607
pred_04b novelty distribution:
novelty_label
<first_item>    2273
significant      903
Name: count, dtype: int64


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Переобучение CatBoost на том же feature pipeli...
6,exp_04_mlp_final_step,0.000000,1.000000,0.793103,0.839506,0.931507,0.883117,Нейросетевая замена финальной CatBoost-модели;...
7,exp_03b_catboost_retrained_fixed_old_clustering,0.769737,0.000000,0.712644,0.875000,0.767123,0.817518,"CatBoost обучен с нуля на legacy features, пос..."
8,exp_04b_mlp_retrained_fixed_old_clustering,0.769737,0.000000,0.724138,0.826667,0.849315,0.837838,"MLP обучен с нуля на legacy features, построен..."


In [26]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

# ---------------------------------------------------------------------
# Experiment 04c: MLP with constrained threshold on fixed old clustering
# ---------------------------------------------------------------------

experiment_name = "exp_04c_mlp_constrained_threshold_fixed_old_clustering"

FEATURE_COLUMNS = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)
best_threshold_04c = 0.48

mlp_04c_model = CatBoostSignificanceModel(
    config=SignificanceModelConfig(
        threshold=best_threshold_04c,
        duplicate_threshold=0.90,
        review_margin=0.10,
        feature_columns=FEATURE_COLUMNS,
    ),
    model=mlp_final,
)
mlp_04c_model.feature_columns = FEATURE_COLUMNS

pred_04c = mlp_04c_model.predict_clustered_with_fallback(
    news_df=old_pipeline_clustered_news,
    embeddings=old_pipeline_embeddings,
)

pred_04c_path = paths.predictions_dir / f"{experiment_name}.csv"
save_prediction_csv(pred_04c, pred_04c_path)

metrics_04c = evaluate_predictions(golden, pred_04c)

results_table = remove_same_experiment_row(results_table, experiment_name)

results_table = add_experiment_result(
    results_table,
    experiment_name=experiment_name,
    metrics=metrics_04c,
    prediction_path=str(pred_04c_path),
    embedding_variant="BAAI/bge-m3 id-aligned",
    clustering="fixed LegacyBaselineGraphClusterer from exp_00b",
    novelty_variant="Same MLP as exp_04b + constrained threshold",
    comment=(
        "Порог MLP выбран по validation как компромисс между F1 и долей predicted significant; "
        "кластеризация фиксирована от exp_00b"
    ),
)

compact_metrics_table(results_table)

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Переобучение CatBoost на том же feature pipeli...
6,exp_04_mlp_final_step,0.000000,1.000000,0.793103,0.839506,0.931507,0.883117,Нейросетевая замена финальной CatBoost-модели;...
7,exp_03b_catboost_retrained_fixed_old_clustering,0.769737,0.000000,0.712644,0.875000,0.767123,0.817518,"CatBoost обучен с нуля на legacy features, пос..."
8,exp_04b_mlp_retrained_fixed_old_clustering,0.769737,0.000000,0.724138,0.826667,0.849315,0.837838,"MLP обучен с нуля на legacy features, построен..."
9,exp_04c_mlp_constrained_threshold_fixed_old_cl...,0.769737,0.000000,0.586207,0.824561,0.643836,0.723077,Порог MLP выбран по validation как компромисс ...


In [27]:
# ---------------------------------------------------------------------
# Safe optional experiment: one Logistic Regression with liblinear
# ---------------------------------------------------------------------

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

FEATURE_COLUMNS = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)

X_train = train_part[FEATURE_COLUMNS]
y_train = train_part["is_significant"].astype(int).to_numpy()

X_val = val_part[FEATURE_COLUMNS]
y_val = val_part["is_significant"].astype(int).to_numpy()

safe_logreg = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "logreg",
            LogisticRegression(
                solver="liblinear",
                C=1.0,
                class_weight=None,
                max_iter=100,
                random_state=42,
            ),
        ),
    ]
)

safe_logreg.fit(X_train, y_train)

val_proba = safe_logreg.predict_proba(X_val)[:, 1]

threshold_rows = []

for threshold in np.linspace(0.20, 0.80, 31):
    val_pred = (val_proba >= threshold).astype(int)

    threshold_rows.append({
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_val, val_pred),
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred, zero_division=0),
        "f1": f1_score(y_val, val_pred, zero_division=0),
        "positive_rate_pred": float(np.mean(val_pred)),
        "positive_rate_true": float(np.mean(y_val)),
    })

safe_logreg_thresholds = pd.DataFrame(threshold_rows).sort_values(
    ["f1", "recall", "precision"],
    ascending=False,
)

safe_logreg_thresholds.head(10)

,threshold,accuracy,precision,recall,f1,positive_rate_pred,positive_rate_true
3,0.26,0.78125,0.758621,1.0,0.862745,0.90625,0.6875
4,0.28,0.78125,0.758621,1.0,0.862745,0.90625,0.6875
5,0.30,0.78125,0.758621,1.0,0.862745,0.90625,0.6875
6,0.32,0.78125,0.758621,1.0,0.862745,0.90625,0.6875
7,0.34,0.78125,0.758621,1.0,0.862745,0.90625,0.6875
8,0.36,0.78125,0.758621,1.0,0.862745,0.90625,0.6875
9,0.38,0.78125,0.758621,1.0,0.862745,0.90625,0.6875
10,0.40,0.78125,0.758621,1.0,0.862745,0.90625,0.6875
11,0.42,0.78125,0.758621,1.0,0.862745,0.90625,0.6875
12,0.44,0.78125,0.758621,1.0,0.862745,0.90625,0.6875


In [28]:
# ---------------------------------------------------------------------
# Experiment 05: safe Logistic Regression on fixed old clustering
# ---------------------------------------------------------------------

experiment_name = "exp_05_logreg_fixed_old_clustering"

FEATURE_COLUMNS = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)
best_threshold_05 = 0.44

# Дообучаем на всём train_frame, а не только train_part
X_full = train_frame[FEATURE_COLUMNS]
y_full = train_frame["is_significant"].astype(int).to_numpy()

logreg_final = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "logreg",
            LogisticRegression(
                solver="liblinear",
                C=1.0,
                class_weight=None,
                max_iter=100,
                random_state=42,
            ),
        ),
    ]
)

logreg_final.fit(X_full, y_full)

logreg_05_model = CatBoostSignificanceModel(
    config=SignificanceModelConfig(
        threshold=best_threshold_05,
        duplicate_threshold=0.90,
        review_margin=0.10,
        feature_columns=FEATURE_COLUMNS,
    ),
    model=logreg_final,
)
logreg_05_model.feature_columns = FEATURE_COLUMNS

pred_05 = logreg_05_model.predict_clustered_with_fallback(
    news_df=old_pipeline_clustered_news,
    embeddings=old_pipeline_embeddings,
)

pred_05_path = paths.predictions_dir / f"{experiment_name}.csv"
save_prediction_csv(pred_05, pred_05_path)

metrics_05 = evaluate_predictions(golden, pred_05)

results_table = remove_same_experiment_row(results_table, experiment_name)

results_table = add_experiment_result(
    results_table,
    experiment_name=experiment_name,
    metrics=metrics_05,
    prediction_path=str(pred_05_path),
    embedding_variant="BAAI/bge-m3 id-aligned",
    clustering="fixed LegacyBaselineGraphClusterer from exp_00b",
    novelty_variant="LogisticRegression liblinear + first-item fallback",
    comment=(
        "Линейная модель на legacy features; threshold выбран по validation plateau; "
        "кластеризация фиксирована от exp_00b"
    ),
)

compact_metrics_table(results_table)

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Переобучение CatBoost на том же feature pipeli...
6,exp_04_mlp_final_step,0.000000,1.000000,0.793103,0.839506,0.931507,0.883117,Нейросетевая замена финальной CatBoost-модели;...
7,exp_03b_catboost_retrained_fixed_old_clustering,0.769737,0.000000,0.712644,0.875000,0.767123,0.817518,"CatBoost обучен с нуля на legacy features, пос..."
8,exp_04b_mlp_retrained_fixed_old_clustering,0.769737,0.000000,0.724138,0.826667,0.849315,0.837838,"MLP обучен с нуля на legacy features, построен..."
9,exp_04c_mlp_constrained_threshold_fixed_old_cl...,0.769737,0.000000,0.586207,0.824561,0.643836,0.723077,Порог MLP выбран по validation как компромисс ...


In [29]:
# ---------------------------------------------------------------------
# Derived novelty features
# ---------------------------------------------------------------------
# Эти признаки строятся поверх legacy features.
# Идея: дать модели не только "похожесть", но и разрывы, временной контекст,
# lifecycle кластера и числовые обновления.
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


def add_derived_novelty_features(features_df: pd.DataFrame) -> pd.DataFrame:
    out = features_df.copy()

    eps = 1e-6

    # Берём базовые признаки с безопасным fillna.
    max_sim = out["max_prev_similarity"].fillna(0.0)
    mean_sim = out["mean_prev_similarity"].fillna(0.0)
    last_sim = out["last_prev_similarity"].fillna(0.0)
    centroid_sim = out["previous_centroid_similarity"].fillna(0.0)

    title_jaccard = out["title_jaccard_max"].fillna(0.0)
    text_jaccard = out["text_jaccard_max"].fillna(0.0)

    days_since_previous = out["days_since_previous"].fillna(9999.0)
    days_since_cluster_start = out["days_since_cluster_start"].fillna(0.0)

    cluster_size = out["cluster_size_so_far"].fillna(0.0)
    position = out["position_in_cluster"].fillna(0.0)

    shared_numbers = out["shared_numbers_count"].fillna(0.0)
    new_numbers = out["new_numbers_count"].fillna(0.0)
    total_numbers = shared_numbers + new_numbers

    # Similarity / distance.
    out["max_prev_distance"] = 1.0 - max_sim
    out["last_prev_distance"] = 1.0 - last_sim
    out["centroid_similarity_gap"] = max_sim - centroid_sim
    out["max_vs_mean_similarity_gap"] = max_sim - mean_sim
    out["max_vs_centroid_similarity_gap"] = max_sim - centroid_sim

    # Semantic vs lexical overlap.
    out["semantic_lexical_gap"] = max_sim - text_jaccard
    out["title_text_jaccard_gap"] = title_jaccard - text_jaccard

    # Time-aware interactions.
    out["similarity_per_day"] = max_sim / (1.0 + days_since_previous)
    out["distance_per_day"] = (1.0 - max_sim) / (1.0 + days_since_previous)

    out["is_recent_high_similarity"] = (
        (max_sim >= 0.90) & (days_since_previous <= 2)
    ).astype(int)

    out["is_old_high_similarity"] = (
        (max_sim >= 0.90) & (days_since_previous > 14)
    ).astype(int)

    # Cluster lifecycle.
    out["log_cluster_size_so_far"] = np.log1p(cluster_size.clip(lower=0))
    out["log_position_in_cluster"] = np.log1p(position.clip(lower=0))
    out["cluster_velocity"] = cluster_size / (1.0 + days_since_cluster_start)

    out["is_early_cluster_item"] = (position <= 2).astype(int)
    out["is_late_cluster_item"] = (position >= 5).astype(int)

    # Numbers / factual updates.
    out["has_new_numbers"] = (new_numbers > 0).astype(int)
    out["new_numbers_ratio"] = np.where(
        total_numbers > 0,
        new_numbers / (total_numbers + eps),
        0.0,
    )

    # Защита от inf/nan.
    derived_columns = [
        "max_prev_distance",
        "last_prev_distance",
        "centroid_similarity_gap",
        "max_vs_mean_similarity_gap",
        "max_vs_centroid_similarity_gap",
        "semantic_lexical_gap",
        "title_text_jaccard_gap",
        "similarity_per_day",
        "distance_per_day",
        "is_recent_high_similarity",
        "is_old_high_similarity",
        "log_cluster_size_so_far",
        "log_position_in_cluster",
        "cluster_velocity",
        "is_early_cluster_item",
        "is_late_cluster_item",
        "has_new_numbers",
        "new_numbers_ratio",
    ]

    out[derived_columns] = (
        out[derived_columns]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return out


DERIVED_FEATURE_COLUMNS = [
    "max_prev_distance",
    "last_prev_distance",
    "centroid_similarity_gap",
    "max_vs_mean_similarity_gap",
    "max_vs_centroid_similarity_gap",
    "semantic_lexical_gap",
    "title_text_jaccard_gap",
    "similarity_per_day",
    "distance_per_day",
    "is_recent_high_similarity",
    "is_old_high_similarity",
    "log_cluster_size_so_far",
    "log_position_in_cluster",
    "cluster_velocity",
    "is_early_cluster_item",
    "is_late_cluster_item",
    "has_new_numbers",
    "new_numbers_ratio",
]

FEATURE_COLUMNS = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)
EXPANDED_FEATURE_COLUMNS = FEATURE_COLUMNS + DERIVED_FEATURE_COLUMNS

print("Legacy features:", len(FEATURE_COLUMNS))
print("Derived features:", len(DERIVED_FEATURE_COLUMNS))
print("Expanded features:", len(EXPANDED_FEATURE_COLUMNS))

Legacy features: 18
Derived features: 18
Expanded features: 36


In [30]:
# ---------------------------------------------------------------------
# Prepare expanded train/validation frames
# ---------------------------------------------------------------------

train_frame_expanded = add_derived_novelty_features(train_frame)
train_part_expanded = add_derived_novelty_features(train_part)
val_part_expanded = add_derived_novelty_features(val_part)

missing_expanded_features = [
    column
    for column in EXPANDED_FEATURE_COLUMNS
    if column not in train_frame_expanded.columns
]

if missing_expanded_features:
    raise ValueError(f"Missing expanded feature columns: {missing_expanded_features}")

X_train_expanded = train_part_expanded[EXPANDED_FEATURE_COLUMNS]
y_train_expanded = train_part_expanded["is_significant"].astype(int).to_numpy()

X_val_expanded = val_part_expanded[EXPANDED_FEATURE_COLUMNS]
y_val_expanded = val_part_expanded["is_significant"].astype(int).to_numpy()

X_full_expanded = train_frame_expanded[EXPANDED_FEATURE_COLUMNS]
y_full_expanded = train_frame_expanded["is_significant"].astype(int).to_numpy()

print("X_train_expanded:", X_train_expanded.shape)
print("X_val_expanded:", X_val_expanded.shape)
print("X_full_expanded:", X_full_expanded.shape)

X_train_expanded: (147, 36)
X_val_expanded: (32, 36)
X_full_expanded: (179, 36)


In [31]:
# ---------------------------------------------------------------------
# Experiment 06: CatBoost with derived features on fixed old clustering
# ---------------------------------------------------------------------

from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

from model.config import SignificanceModelConfig
from model.significance_model import CatBoostSignificanceModel


def remove_same_experiment_row(
    results_table: pd.DataFrame,
    experiment_name: str,
) -> pd.DataFrame:
    if results_table is None or results_table.empty:
        return pd.DataFrame()

    if "experiment" not in results_table.columns:
        return results_table.copy()

    return (
        results_table[results_table["experiment"] != experiment_name]
        .copy()
        .reset_index(drop=True)
    )


experiment_name = "exp_06_catboost_derived_features_fixed_old_clustering"

catboost_06_params = {
    "iterations": 500,
    "depth": 5,
    "learning_rate": 0.05,
    "l2_leaf_reg": 5.0,
    "loss_function": "Logloss",
    "eval_metric": "F1",
    "random_seed": 42,
    "verbose": 100,
}

catboost_06_candidate = CatBoostClassifier(**catboost_06_params)

catboost_06_candidate.fit(
    X_train_expanded,
    y_train_expanded,
    eval_set=(X_val_expanded, y_val_expanded),
    use_best_model=True,
)

val_proba_06 = catboost_06_candidate.predict_proba(X_val_expanded)[:, 1]

threshold_rows_06 = []

for threshold in np.linspace(0.20, 0.80, 61):
    val_pred = (val_proba_06 >= threshold).astype(int)

    threshold_rows_06.append({
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_val_expanded, val_pred),
        "precision": precision_score(y_val_expanded, val_pred, zero_division=0),
        "recall": recall_score(y_val_expanded, val_pred, zero_division=0),
        "f1": f1_score(y_val_expanded, val_pred, zero_division=0),
        "positive_rate_pred": float(np.mean(val_pred)),
        "positive_rate_true": float(np.mean(y_val_expanded)),
    })

threshold_table_06 = pd.DataFrame(threshold_rows_06).sort_values(
    ["f1", "recall", "precision"],
    ascending=False,
).reset_index(drop=True)

display(threshold_table_06.head(15))

best_threshold_06 = float(threshold_table_06.iloc[0]["threshold"])

print("Best threshold:", best_threshold_06)
print("\nValidation report:")

val_pred_06 = (val_proba_06 >= best_threshold_06).astype(int)

print(
    classification_report(
        y_val_expanded,
        val_pred_06,
        target_names=["not_significant", "significant"],
        zero_division=0,
    )
)

0:	learn: 0.9339207	test: 0.8163265	best: 0.8163265 (0)	total: 1.66ms	remaining: 827ms
100:	learn: 0.9779736	test: 0.8400000	best: 0.8627451 (2)	total: 117ms	remaining: 462ms
200:	learn: 1.0000000	test: 0.8400000	best: 0.8627451 (2)	total: 236ms	remaining: 351ms
300:	learn: 1.0000000	test: 0.8400000	best: 0.8627451 (2)	total: 368ms	remaining: 243ms
400:	learn: 1.0000000	test: 0.8400000	best: 0.8627451 (2)	total: 490ms	remaining: 121ms
499:	learn: 1.0000000	test: 0.8400000	best: 0.8627451 (2)	total: 622ms	remaining: 0us

bestTest = 0.862745098
bestIteration = 2

Shrink model to first 3 iterations.


,threshold,accuracy,precision,recall,f1,positive_rate_pred,positive_rate_true
0,0.49,0.78125,0.758621,1.000000,0.862745,0.90625,0.6875
1,0.50,0.78125,0.758621,1.000000,0.862745,0.90625,0.6875
2,0.51,0.78125,0.758621,1.000000,0.862745,0.90625,0.6875
3,0.52,0.75000,0.750000,0.954545,0.840000,0.87500,0.6875
4,0.53,0.75000,0.750000,0.954545,0.840000,0.87500,0.6875
5,0.48,0.71875,0.709677,1.000000,0.830189,0.96875,0.6875
6,0.20,0.68750,0.687500,1.000000,0.814815,1.00000,0.6875
7,0.21,0.68750,0.687500,1.000000,0.814815,1.00000,0.6875
8,0.22,0.68750,0.687500,1.000000,0.814815,1.00000,0.6875
9,0.23,0.68750,0.687500,1.000000,0.814815,1.00000,0.6875


Best threshold: 0.49000000000000005

Validation report:
                 precision    recall  f1-score   support

not_significant       1.00      0.30      0.46        10
    significant       0.76      1.00      0.86        22

       accuracy                           0.78        32
      macro avg       0.88      0.65      0.66        32
   weighted avg       0.83      0.78      0.74        32



In [32]:
# ---------------------------------------------------------------------
# Train final exp_06 model and evaluate on golden
# ---------------------------------------------------------------------

import model.features as features_module
import model.significance_model as significance_model_module

# Финальная модель обучается на всём train_frame_expanded.
catboost_06_final = CatBoostClassifier(**catboost_06_params)
catboost_06_final.fit(X_full_expanded, y_full_expanded)

catboost_06_model = CatBoostSignificanceModel(
    config=SignificanceModelConfig(
        threshold=best_threshold_06,
        duplicate_threshold=0.90,
        review_margin=0.10,
        feature_columns=EXPANDED_FEATURE_COLUMNS,
    ),
    model=catboost_06_final,
)
catboost_06_model.feature_columns = list(EXPANDED_FEATURE_COLUMNS)


def build_legacy_significance_features_with_derived(*args, **kwargs):
    legacy_features = features_module.build_legacy_significance_features(*args, **kwargs)
    return add_derived_novelty_features(legacy_features)


# Временно патчим feature-builder только для prediction exp_06.
original_significance_builder = significance_model_module.build_legacy_significance_features

try:
    significance_model_module.build_legacy_significance_features = (
        build_legacy_significance_features_with_derived
    )

    pred_06 = catboost_06_model.predict_clustered_with_fallback(
        news_df=old_pipeline_clustered_news,
        embeddings=old_pipeline_embeddings,
    )

finally:
    significance_model_module.build_legacy_significance_features = original_significance_builder


pred_06_path = paths.predictions_dir / f"{experiment_name}.csv"
save_prediction_csv(pred_06, pred_06_path)

metrics_06 = evaluate_predictions(golden, pred_06)

results_table = remove_same_experiment_row(results_table, experiment_name)

results_table = add_experiment_result(
    results_table,
    experiment_name=experiment_name,
    metrics=metrics_06,
    prediction_path=str(pred_06_path),
    embedding_variant="BAAI/bge-m3 id-aligned",
    clustering="fixed LegacyBaselineGraphClusterer from exp_00b",
    novelty_variant="CatBoost + derived features + first-item fallback",
    comment=(
        "CatBoost на legacy + derived novelty features; "
        "кластеризация фиксирована от exp_00b"
    ),
)

print("pred_06:", pred_06.shape)
print("pred_06 clusters:", pred_06["cluster_id"].nunique())
print("pred_06 novelty distribution:")
print(pred_06["novelty_label"].replace("", "<first_item>").value_counts(dropna=False))

compact_metrics_table(results_table)

0:	learn: 0.9051095	total: 1.8ms	remaining: 900ms
100:	learn: 0.9602888	total: 111ms	remaining: 439ms
200:	learn: 0.9815498	total: 222ms	remaining: 331ms
300:	learn: 0.9962547	total: 335ms	remaining: 222ms
400:	learn: 1.0000000	total: 450ms	remaining: 111ms
499:	learn: 1.0000000	total: 584ms	remaining: 0us
pred_06: (3176, 10)
pred_06 clusters: 2607
pred_06 novelty distribution:
novelty_label
<first_item>    2273
significant      836
minor             46
duplicate         21
Name: count, dtype: int64


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Переобучение CatBoost на том же feature pipeli...
6,exp_04_mlp_final_step,0.000000,1.000000,0.793103,0.839506,0.931507,0.883117,Нейросетевая замена финальной CatBoost-модели;...
7,exp_03b_catboost_retrained_fixed_old_clustering,0.769737,0.000000,0.712644,0.875000,0.767123,0.817518,"CatBoost обучен с нуля на legacy features, пос..."
8,exp_04b_mlp_retrained_fixed_old_clustering,0.769737,0.000000,0.724138,0.826667,0.849315,0.837838,"MLP обучен с нуля на legacy features, построен..."
9,exp_04c_mlp_constrained_threshold_fixed_old_cl...,0.769737,0.000000,0.586207,0.824561,0.643836,0.723077,Порог MLP выбран по validation как компромисс ...


In [33]:
valid_fixed_clustering_experiments = [
    "exp_00b_reproduced_old_pipeline",
    "exp_03b_catboost_retrained_fixed_old_clustering",
    "exp_04b_mlp_retrained_fixed_old_clustering",
    "exp_04c_mlp_constrained_threshold_fixed_old_clustering",
    "exp_05_logreg_fixed_old_clustering",
    "exp_06_catboost_derived_features_fixed_old_clustering",
]

valid_fixed_table = compact_metrics_table(
    results_table[
        results_table["experiment"].isin(valid_fixed_clustering_experiments)
    ]
).copy()

baseline_row = valid_fixed_table[
    valid_fixed_table["experiment"] == "exp_00b_reproduced_old_pipeline"
].iloc[0]

for column in [
    "significance_accuracy",
    "significant_precision",
    "significant_recall",
    "significant_f1",
]:
    valid_fixed_table[f"{column}_delta"] = (
        valid_fixed_table[column] - baseline_row[column]
    )

valid_fixed_table

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment,significance_accuracy_delta,significant_precision_delta,significant_recall_delta,significant_f1_delta
0,exp_00b_reproduced_old_pipeline,0.769737,0.0,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...,0.000000,0.000000,0.000000,0.000000
1,exp_03b_catboost_retrained_fixed_old_clustering,0.769737,0.0,0.712644,0.875000,0.767123,0.817518,"CatBoost обучен с нуля на legacy features, пос...",-0.011494,0.039384,-0.068493,-0.018098
2,exp_04b_mlp_retrained_fixed_old_clustering,0.769737,0.0,0.724138,0.826667,0.849315,0.837838,"MLP обучен с нуля на legacy features, построен...",0.000000,-0.008950,0.013699,0.002221
3,exp_04c_mlp_constrained_threshold_fixed_old_cl...,0.769737,0.0,0.586207,0.824561,0.643836,0.723077,Порог MLP выбран по validation как компромисс ...,-0.137931,-0.011055,-0.191781,-0.112540
4,exp_05_logreg_fixed_old_clustering,0.769737,0.0,0.689655,0.838235,0.780822,0.808511,Линейная модель на legacy features; threshold ...,-0.034483,0.002619,-0.054795,-0.027106
5,exp_06_catboost_derived_features_fixed_old_clu...,0.769737,0.0,0.689655,0.848485,0.767123,0.805755,CatBoost на legacy + derived novelty features;...,-0.034483,0.012868,-0.068493,-0.029861


In [34]:
# ---------------------------------------------------------------------
# Experiment 07: CatBoost trained on silver + weighted bronze
# ---------------------------------------------------------------------
# Goal:
#   Check whether bronze weak labels improve final novelty classifier.
#
# Training:
#   - silver train_part: weight 1.0
#   - bronze high confidence: weight 0.5
#   - bronze medium confidence: weight 0.2
#
# Validation threshold:
#   - silver val_part only
#
# Final evaluation:
#   - human golden only
#
# Required existing objects from notebook 06:
#   paths
#   encoder
#   current_model
#   golden
#   old_pipeline_clustered_news
#   old_pipeline_embeddings
#   train_frame
#   train_part
#   val_part
#   results_table
# ---------------------------------------------------------------------

import json
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

from model.legacy_pipeline import LegacyNewsNoveltyPipeline, LegacyNewsPipelineConfig
from model.features import (
    LEGACY_SIGNIFICANCE_FEATURE_COLUMNS,
    build_legacy_significance_features,
)
from model.config import SignificanceModelConfig
from model.significance_model import CatBoostSignificanceModel


# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

experiment_name = "exp_07_catboost_silver_bronze_weighted_fixed_old_clustering"

# Положи сюда путь к размеченному bronze zip.
# Если zip лежит в data/artifacts/bronze_annotation_large_annotated.zip,
# оставь как есть. Иначе поменяй путь.
BRONZE_ANNOTATED_ZIP_PATH = (
    paths.artifacts_dir / "bronze_annotation_large" / "bronze_annotation_large_annotated.zip"
)

# Если распаковал руками, можно указать csv напрямую.
BRONZE_LABELS_CSV_PATH = (
    paths.artifacts_dir / "bronze_annotated" / "bronze_large_labels.csv"
)

# Должно совпадать с тем, как строился bronze export.
LARGE_BRONZE_INPUT_PATH = paths.prepared_dir / "lenta_clean_news.csv"
LARGE_BRONZE_YEARS = [2002, 2003]
MAX_NEWS_PER_YEAR = 12000
LARGE_BRONZE_RANDOM_STATE = 42

FEATURE_COLUMNS = list(LEGACY_SIGNIFICANCE_FEATURE_COLUMNS)

BRONZE_CONFIDENCE_WEIGHTS = {
    "high": 0.05,
    "medium": 0.02,
}

catboost_07_params = {
    "iterations": 500,
    "depth": 5,
    "learning_rate": 0.05,
    "l2_leaf_reg": 5.0,
    "loss_function": "Logloss",
    "eval_metric": "F1",
    "random_seed": 42,
    "verbose": 100,
}


# ---------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------

def normalize_news_id_for_join(series: pd.Series) -> pd.Series:
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def remove_same_experiment_row(
    results_table: pd.DataFrame,
    experiment_name: str,
) -> pd.DataFrame:
    if results_table is None or results_table.empty:
        return pd.DataFrame()

    if "experiment" not in results_table.columns:
        return results_table.copy()

    return (
        results_table[results_table["experiment"] != experiment_name]
        .copy()
        .reset_index(drop=True)
    )


def normalize_needs_review(series: pd.Series) -> pd.Series:
    normalized = series.fillna("").astype(str).str.strip().str.lower()

    return normalized.isin([
        "true",
        "1",
        "yes",
        "y",
        "да",
    ])


def load_bronze_labels() -> pd.DataFrame:
    if BRONZE_LABELS_CSV_PATH.exists():
        print("Loading bronze labels from CSV:", BRONZE_LABELS_CSV_PATH)
        return pd.read_csv(BRONZE_LABELS_CSV_PATH)

    if BRONZE_ANNOTATED_ZIP_PATH.exists():
        print("Loading bronze labels from ZIP:", BRONZE_ANNOTATED_ZIP_PATH)

        with zipfile.ZipFile(BRONZE_ANNOTATED_ZIP_PATH, "r") as archive:
            names = archive.namelist()

            preferred_names = [
                "bronze_large_labels.csv",
                "bronze_large_candidates_all_annotated.csv",
            ]

            selected_name = None

            for preferred_name in preferred_names:
                matches = [
                    name
                    for name in names
                    if name.endswith(preferred_name)
                ]

                if matches:
                    selected_name = matches[0]
                    break

            if selected_name is None:
                raise FileNotFoundError(
                    "Could not find bronze_large_labels.csv or "
                    "bronze_large_candidates_all_annotated.csv inside zip. "
                    f"Files: {names[:20]}"
                )

            print("Selected file from ZIP:", selected_name)

            with archive.open(selected_name) as file:
                return pd.read_csv(file)

    raise FileNotFoundError(
        "Bronze labels not found. Set BRONZE_ANNOTATED_ZIP_PATH or "
        "BRONZE_LABELS_CSV_PATH correctly."
    )


def collect_silver_ids() -> set[str]:
    silver_ids = set()

    for frame_name in [
        "silver_labels",
        "silver_df",
        "train_frame",
        "silver_train_features",
    ]:
        if frame_name in globals():
            frame = globals()[frame_name]

            if isinstance(frame, pd.DataFrame) and "news_id" in frame.columns:
                ids = set(normalize_news_id_for_join(frame["news_id"]))
                silver_ids |= ids
                print(f"{frame_name}: collected {len(ids)} silver ids")

    known_silver_paths = [
        paths.prepared_dir / "lenta_silver_set_llm.csv",
        paths.prepared_dir / "lenta_silver_set.csv",
        paths.prepared_dir / "lenta_silver_annotations.csv",
        paths.prepared_dir / "lenta_llm_silver_set.csv",
        paths.significance_model_dir / "silver_significance_features.csv",
    ]

    for silver_path in known_silver_paths:
        if silver_path.exists():
            try:
                silver_frame = pd.read_csv(silver_path, usecols=["news_id"])
                ids = set(normalize_news_id_for_join(silver_frame["news_id"]))
                silver_ids |= ids
                print(f"{silver_path.name}: collected {len(ids)} silver ids")
            except Exception as error:
                print(f"[WARN] failed to read silver ids from {silver_path}: {error}")

    print("Total silver ids:", len(silver_ids))

    return silver_ids


def build_large_bronze_clustered_news_and_embeddings() -> tuple[pd.DataFrame, np.ndarray]:
    """Rebuild the same large bronze pool used for annotation.

    This may reuse id-aware embedding caches from 07_bronze_annotation_export.ipynb.
    """

    if (
        "large_bronze_clustered_news" in globals()
        and "large_bronze_embeddings" in globals()
    ):
        print("Using large_bronze_clustered_news / large_bronze_embeddings from memory.")
        return large_bronze_clustered_news.copy(), np.asarray(large_bronze_embeddings)

    if not LARGE_BRONZE_INPUT_PATH.exists():
        raise FileNotFoundError(f"Not found: {LARGE_BRONZE_INPUT_PATH}")

    golden_ids = set(normalize_news_id_for_join(golden["news_id"]))
    silver_ids = collect_silver_ids()

    if "old_pipeline_clustered_news" in globals():
        current_candidate_ids = set(
            normalize_news_id_for_join(old_pipeline_clustered_news["news_id"])
        )
    else:
        current_candidate_pool_path = paths.prepared_dir / "lenta_golden_candidate_pool.csv"

        if not current_candidate_pool_path.exists():
            raise FileNotFoundError(f"Not found: {current_candidate_pool_path}")

        current_candidate_pool = pd.read_csv(current_candidate_pool_path)
        current_candidate_ids = set(
            normalize_news_id_for_join(current_candidate_pool["news_id"])
        )

    exclude_ids = golden_ids | silver_ids | current_candidate_ids

    print("\nRebuilding large bronze pool:")
    print("  golden ids:", len(golden_ids))
    print("  silver ids:", len(silver_ids))
    print("  current candidate ids:", len(current_candidate_ids))
    print("  total excluded ids:", len(exclude_ids))

    raw_all = pd.read_csv(LARGE_BRONZE_INPUT_PATH)

    raw_all["news_id"] = normalize_news_id_for_join(raw_all["news_id"])

    raw_all["published_at"] = pd.to_datetime(
        raw_all["published_at"],
        errors="coerce",
    )

    raw_all = raw_all[raw_all["published_at"].notna()].copy()
    raw_all["year"] = raw_all["published_at"].dt.year

    raw = raw_all[raw_all["year"].isin(LARGE_BRONZE_YEARS)].copy()

    before_exclude = len(raw)

    raw = raw[~raw["news_id"].isin(exclude_ids)].copy()

    after_exclude = len(raw)

    print("  rows before exclude:", before_exclude)
    print("  rows after exclude:", after_exclude)
    print("  years:")
    display(raw["year"].value_counts().sort_index())

    clustered_parts = []
    embedding_parts = []

    for year in LARGE_BRONZE_YEARS:
        year_raw = raw[raw["year"] == year].copy()

        if year_raw.empty:
            print(f"\nYear {year}: empty, skipped")
            continue

        if MAX_NEWS_PER_YEAR is not None and len(year_raw) > MAX_NEWS_PER_YEAR:
            year_raw = (
                year_raw
                .sample(n=MAX_NEWS_PER_YEAR, random_state=LARGE_BRONZE_RANDOM_STATE)
                .sort_values(["published_at", "news_id"], kind="mergesort")
                .reset_index(drop=True)
            )

        print(f"\nYear {year}: rows = {len(year_raw)}")

        year_cache_path = (
            paths.artifacts_dir
            / "embeddings"
            / f"bge_m3_large_bronze_{year}_id_aligned.npz"
        )

        year_pipeline = LegacyNewsNoveltyPipeline(
            encoder=encoder,
            novelty_model=current_model,
            config=LegacyNewsPipelineConfig(
                embeddings_cache_path=year_cache_path,
                story_threshold=0.82,
                story_window_days=14,
                use_topic_blocking=True,
                normalize_embeddings_for_clustering=True,
            ),
        )

        year_result = year_pipeline.predict(year_raw)

        year_clustered = year_result.clustered_news.copy()
        year_embeddings = year_result.embeddings

        year_clustered["news_id"] = normalize_news_id_for_join(year_clustered["news_id"])
        year_clustered["cluster_id"] = (
            "bronze_"
            + str(year)
            + "_"
            + year_clustered["cluster_id"].astype(str)
        )

        cluster_sizes = year_clustered.groupby("cluster_id").size()

        print("  clustered rows:", year_clustered.shape)
        print("  clusters:", year_clustered["cluster_id"].nunique())
        print("  max non-first candidates:", int((cluster_sizes - 1).clip(lower=0).sum()))

        clustered_parts.append(year_clustered)
        embedding_parts.append(year_embeddings)

    if not clustered_parts:
        raise RuntimeError("No bronze clustered parts were produced.")

    clustered_news = pd.concat(clustered_parts, ignore_index=True)
    embeddings = np.vstack(embedding_parts)

    print("\nLarge bronze clustered news:", clustered_news.shape)
    print("Large bronze embeddings:", embeddings.shape)

    return clustered_news, embeddings


def threshold_sweep(
    y_true: np.ndarray,
    y_proba: np.ndarray,
    thresholds: np.ndarray | None = None,
) -> pd.DataFrame:
    if thresholds is None:
        thresholds = np.linspace(0.20, 0.80, 61)

    rows = []

    for threshold in thresholds:
        y_pred = (y_proba >= threshold).astype(int)

        rows.append({
            "threshold": float(threshold),
            "accuracy": accuracy_score(y_true, y_pred),
            "precision": precision_score(y_true, y_pred, zero_division=0),
            "recall": recall_score(y_true, y_pred, zero_division=0),
            "f1": f1_score(y_true, y_pred, zero_division=0),
            "positive_rate_pred": float(np.mean(y_pred)),
            "positive_rate_true": float(np.mean(y_true)),
        })

    return (
        pd.DataFrame(rows)
        .sort_values(["f1", "recall", "precision"], ascending=False)
        .reset_index(drop=True)
    )


# ---------------------------------------------------------------------
# Validate required notebook state
# ---------------------------------------------------------------------

required_objects = [
    "paths",
    "encoder",
    "current_model",
    "golden",
    "old_pipeline_clustered_news",
    "old_pipeline_embeddings",
    "train_frame",
    "train_part",
    "val_part",
    "results_table",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(f"Missing required objects: {missing_objects}")


# ---------------------------------------------------------------------
# Load and filter bronze labels
# ---------------------------------------------------------------------

bronze_labels_raw = load_bronze_labels()

bronze_labels = bronze_labels_raw.copy()
bronze_labels["news_id"] = normalize_news_id_for_join(bronze_labels["news_id"])

if "cluster_id" in bronze_labels.columns:
    bronze_labels["cluster_id"] = bronze_labels["cluster_id"].astype(str)

if "bronze_label" not in bronze_labels.columns:
    raise ValueError("Bronze labels must contain bronze_label column.")

if "confidence" not in bronze_labels.columns:
    raise ValueError("Bronze labels must contain confidence column.")

if "needs_review" not in bronze_labels.columns:
    raise ValueError("Bronze labels must contain needs_review column.")

bronze_labels["bronze_label"] = (
    bronze_labels["bronze_label"]
    .fillna("")
    .astype(str)
    .str.strip()
)

bronze_labels["confidence"] = (
    bronze_labels["confidence"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)

bronze_labels["needs_review_bool"] = normalize_needs_review(
    bronze_labels["needs_review"]
)

valid_bronze_labels = {"significant", "minor", "duplicate"}
valid_bronze_confidence = set(BRONZE_CONFIDENCE_WEIGHTS.keys())

bronze_train_labels = bronze_labels[
    bronze_labels["bronze_label"].isin(valid_bronze_labels)
    & bronze_labels["confidence"].isin(valid_bronze_confidence)
    & (~bronze_labels["needs_review_bool"])
].copy()

bronze_train_labels["is_significant"] = (
    bronze_train_labels["bronze_label"].eq("significant").astype(int)
)

bronze_train_labels["sample_weight"] = (
    bronze_train_labels["confidence"].map(BRONZE_CONFIDENCE_WEIGHTS).astype(float)
)

print("Bronze raw rows:", len(bronze_labels))
print("Bronze train rows:", len(bronze_train_labels))
print("\nBronze train labels:")
display(bronze_train_labels["bronze_label"].value_counts())
print("\nBronze train confidence:")
display(bronze_train_labels["confidence"].value_counts())


# ---------------------------------------------------------------------
# Rebuild bronze features
# ---------------------------------------------------------------------

bronze_clustered_news, bronze_embeddings = build_large_bronze_clustered_news_and_embeddings()

bronze_features = build_legacy_significance_features(
    news_df=bronze_clustered_news,
    embeddings=bronze_embeddings,
)

bronze_features["news_id"] = normalize_news_id_for_join(bronze_features["news_id"])

if "cluster_id" in bronze_features.columns:
    bronze_features["cluster_id"] = bronze_features["cluster_id"].astype(str)

missing_features = [
    column
    for column in FEATURE_COLUMNS
    if column not in bronze_features.columns
]

if missing_features:
    raise ValueError(f"Bronze features missing columns: {missing_features}")

# Merge by news_id. Cluster id is checked diagnostically below.
bronze_train_frame = bronze_features.merge(
    bronze_train_labels[
        [
            "news_id",
            "cluster_id",
            "bronze_label",
            "confidence",
            "is_significant",
            "sample_weight",
        ]
    ].rename(columns={"cluster_id": "label_cluster_id"}),
    on="news_id",
    how="inner",
)

if "cluster_id" in bronze_train_frame.columns:
    cluster_match_rate = (
        bronze_train_frame["cluster_id"].astype(str)
        == bronze_train_frame["label_cluster_id"].astype(str)
    ).mean()

    print("Bronze cluster_id match rate:", cluster_match_rate)

print("Bronze features matched rows:", len(bronze_train_frame))
print("Expected bronze train rows:", len(bronze_train_labels))

if len(bronze_train_frame) < 0.8 * len(bronze_train_labels):
    print(
        "[WARN] Many bronze labels did not match rebuilt features. "
        "Check LARGE_BRONZE_YEARS / MAX_NEWS_PER_YEAR / random_state."
    )

# Exclude golden ids defensively.
golden_ids = set(normalize_news_id_for_join(golden["news_id"]))

before_golden_filter = len(bronze_train_frame)

bronze_train_frame = bronze_train_frame[
    ~bronze_train_frame["news_id"].isin(golden_ids)
].copy()

after_golden_filter = len(bronze_train_frame)

print("Bronze rows before golden filter:", before_golden_filter)
print("Bronze rows after golden filter:", after_golden_filter)


# ---------------------------------------------------------------------
# Build silver + bronze train pools
# ---------------------------------------------------------------------

silver_train = train_part.copy()
silver_val = val_part.copy()
silver_full = train_frame.copy()

silver_train["source"] = "silver"
silver_val["source"] = "silver"
silver_full["source"] = "silver"

silver_train["sample_weight"] = 1.0
silver_full["sample_weight"] = 1.0

bronze_train_frame["source"] = "bronze"

X_silver_train = silver_train[FEATURE_COLUMNS]
y_silver_train = silver_train["is_significant"].astype(int).to_numpy()
w_silver_train = silver_train["sample_weight"].astype(float).to_numpy()

X_bronze = bronze_train_frame[FEATURE_COLUMNS]
y_bronze = bronze_train_frame["is_significant"].astype(int).to_numpy()
w_bronze = bronze_train_frame["sample_weight"].astype(float).to_numpy()

X_train_07 = pd.concat([X_silver_train, X_bronze], ignore_index=True)
y_train_07 = np.concatenate([y_silver_train, y_bronze])
w_train_07 = np.concatenate([w_silver_train, w_bronze])

X_val_07 = silver_val[FEATURE_COLUMNS]
y_val_07 = silver_val["is_significant"].astype(int).to_numpy()

X_silver_full = silver_full[FEATURE_COLUMNS]
y_silver_full = silver_full["is_significant"].astype(int).to_numpy()
w_silver_full = silver_full["sample_weight"].astype(float).to_numpy()

X_full_07 = pd.concat([X_silver_full, X_bronze], ignore_index=True)
y_full_07 = np.concatenate([y_silver_full, y_bronze])
w_full_07 = np.concatenate([w_silver_full, w_bronze])

print("\nTraining pool:")
print("  silver train rows:", len(X_silver_train))
print("  bronze train rows:", len(X_bronze))
print("  combined train rows:", len(X_train_07))
print("  combined full rows:", len(X_full_07))
print("  bronze effective weight sum:", float(w_bronze.sum()))
print("  silver full weight sum:", float(w_silver_full.sum()))


# ---------------------------------------------------------------------
# Train candidate model and tune threshold on silver validation
# ---------------------------------------------------------------------

catboost_07_candidate = CatBoostClassifier(**catboost_07_params)

catboost_07_candidate.fit(
    X_train_07,
    y_train_07,
    sample_weight=w_train_07,
    eval_set=(X_val_07, y_val_07),
    use_best_model=True,
)

val_proba_07 = catboost_07_candidate.predict_proba(X_val_07)[:, 1]

threshold_table_07 = threshold_sweep(
    y_true=y_val_07,
    y_proba=val_proba_07,
)

display(threshold_table_07.head(15))

best_threshold_07 = float(threshold_table_07.iloc[0]["threshold"])

print("Best threshold:", best_threshold_07)
print("\nValidation report:")

val_pred_07 = (val_proba_07 >= best_threshold_07).astype(int)

print(
    classification_report(
        y_val_07,
        val_pred_07,
        target_names=["not_significant", "significant"],
        zero_division=0,
    )
)


# ---------------------------------------------------------------------
# Train final model on full silver + bronze and evaluate on golden
# ---------------------------------------------------------------------

catboost_07_final = CatBoostClassifier(**catboost_07_params)

catboost_07_final.fit(
    X_full_07,
    y_full_07,
    sample_weight=w_full_07,
)

catboost_07_model = CatBoostSignificanceModel(
    config=SignificanceModelConfig(
        threshold=best_threshold_07,
        duplicate_threshold=0.90,
        review_margin=0.10,
        feature_columns=FEATURE_COLUMNS,
    ),
    model=catboost_07_final,
)
catboost_07_model.feature_columns = FEATURE_COLUMNS

pred_07 = catboost_07_model.predict_clustered_with_fallback(
    news_df=old_pipeline_clustered_news,
    embeddings=old_pipeline_embeddings,
)

pred_07_path = paths.predictions_dir / f"{experiment_name}.csv"
save_prediction_csv(pred_07, pred_07_path)

metrics_07 = evaluate_predictions(golden, pred_07)

results_table = remove_same_experiment_row(results_table, experiment_name)

results_table = add_experiment_result(
    results_table,
    experiment_name=experiment_name,
    metrics=metrics_07,
    prediction_path=str(pred_07_path),
    embedding_variant="BAAI/bge-m3 id-aligned",
    clustering="fixed LegacyBaselineGraphClusterer from exp_00b",
    novelty_variant="CatBoost trained on silver + weighted bronze + first-item fallback",
    comment=(
        "CatBoost trained on silver plus bronze weak labels with confidence-based "
        "sample weights; threshold tuned on silver validation; clustering fixed from exp_00b"
    ),
)

print("pred_07:", pred_07.shape)
print("pred_07 clusters:", pred_07["cluster_id"].nunique())
print("pred_07 novelty distribution:")
print(pred_07["novelty_label"].replace("", "<first_item>").value_counts(dropna=False))

compact_metrics_table(results_table)

Loading bronze labels from ZIP: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\bronze_annotation_large\bronze_annotation_large_annotated.zip
Selected file from ZIP: bronze_large_labels.csv
Bronze raw rows: 2194
Bronze train rows: 2044

Bronze train labels:


bronze_label
significant    1593
minor           432
duplicate        19
Name: count, dtype: int64


Bronze train confidence:


confidence
high      1211
medium     833
Name: count, dtype: int64

silver_labels: collected 2036 silver ids
train_frame: collected 179 silver ids
lenta_silver_set_llm.csv: collected 2036 silver ids
silver_significance_features.csv: collected 266 silver ids
Total silver ids: 2036

Rebuilding large bronze pool:
  golden ids: 121
  silver ids: 2036
  current candidate ids: 3176
  total excluded ids: 3176
  rows before exclude: 43727
  rows after exclude: 43727
  years:


year
2002    22173
2003    21554
Name: count, dtype: int64


Year 2002: rows = 12000
Рёбер в графе похожести: 1332
Количество кластеров: 10975
  clustered rows: (12000, 16)
  clusters: 10975
  max non-first candidates: 1025

Year 2003: rows = 12000
Рёбер в графе похожести: 1543
Количество кластеров: 10831
  clustered rows: (12000, 16)
  clusters: 10831
  max non-first candidates: 1169

Large bronze clustered news: (24000, 16)
Large bronze embeddings: (24000, 1024)
Bronze cluster_id match rate: 0.0
Bronze features matched rows: 2044
Expected bronze train rows: 2044
Bronze rows before golden filter: 2044
Bronze rows after golden filter: 2044

Training pool:
  silver train rows: 147
  bronze train rows: 2044
  combined train rows: 2191
  combined full rows: 2223
  bronze effective weight sum: 77.21000000000001
  silver full weight sum: 179.0
0:	learn: 0.9266037	test: 0.8400000	best: 0.8400000 (0)	total: 2.19ms	remaining: 1.09s
100:	learn: 0.9811972	test: 0.8800000	best: 0.8800000 (58)	total: 166ms	remaining: 654ms
200:	learn: 0.9848780	test: 0.880

,threshold,accuracy,precision,recall,f1,positive_rate_pred,positive_rate_true
0,0.50,0.81250,0.785714,1.0,0.880000,0.87500,0.6875
1,0.51,0.81250,0.785714,1.0,0.880000,0.87500,0.6875
2,0.52,0.81250,0.785714,1.0,0.880000,0.87500,0.6875
3,0.53,0.81250,0.785714,1.0,0.880000,0.87500,0.6875
4,0.54,0.81250,0.785714,1.0,0.880000,0.87500,0.6875
5,0.55,0.81250,0.785714,1.0,0.880000,0.87500,0.6875
6,0.56,0.81250,0.785714,1.0,0.880000,0.87500,0.6875
7,0.57,0.81250,0.785714,1.0,0.880000,0.87500,0.6875
8,0.58,0.81250,0.785714,1.0,0.880000,0.87500,0.6875
9,0.59,0.81250,0.785714,1.0,0.880000,0.87500,0.6875


Best threshold: 0.5

Validation report:
                 precision    recall  f1-score   support

not_significant       1.00      0.40      0.57        10
    significant       0.79      1.00      0.88        22

       accuracy                           0.81        32
      macro avg       0.89      0.70      0.73        32
   weighted avg       0.85      0.81      0.78        32

0:	learn: 0.9150349	total: 2.31ms	remaining: 1.16s
100:	learn: 0.9660594	total: 167ms	remaining: 658ms
200:	learn: 0.9833425	total: 333ms	remaining: 495ms
300:	learn: 0.9872998	total: 497ms	remaining: 328ms
400:	learn: 0.9881719	total: 662ms	remaining: 163ms
499:	learn: 0.9884739	total: 817ms	remaining: 0us
pred_07: (3176, 10)
pred_07 clusters: 2607
pred_07 novelty distribution:
novelty_label
<first_item>    2273
significant      823
duplicate         41
minor             39
Name: count, dtype: int64


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Переобучение CatBoost на том же feature pipeli...
6,exp_04_mlp_final_step,0.000000,1.000000,0.793103,0.839506,0.931507,0.883117,Нейросетевая замена финальной CatBoost-модели;...
7,exp_03b_catboost_retrained_fixed_old_clustering,0.769737,0.000000,0.712644,0.875000,0.767123,0.817518,"CatBoost обучен с нуля на legacy features, пос..."
8,exp_04b_mlp_retrained_fixed_old_clustering,0.769737,0.000000,0.724138,0.826667,0.849315,0.837838,"MLP обучен с нуля на legacy features, построен..."
9,exp_04c_mlp_constrained_threshold_fixed_old_cl...,0.769737,0.000000,0.586207,0.824561,0.643836,0.723077,Порог MLP выбран по validation как компромисс ...


In [35]:
# ---------------------------------------------------------------------
# Experiment 08: downstream check for fine-tuned BGE-M3
# ---------------------------------------------------------------------
# Goal:
#   Проверить, улучшает ли дообученная BGE-M3 downstream-кластеризацию сюжетов.
#
# Важно:
#   Это диагностический эксперимент по embedding space.
#   Сначала смотрим pairwise_f1 / false_merge_rate.
#   Если fine-tuned embeddings не улучшают clustering без роста false merge,
#   тащить их дальше в novelty model нет смысла.
#
# Required existing objects:
#   paths
#   golden
#   old_pipeline_clustered_news
#   old_pipeline_embeddings
#   encoder/current_model are not required for this diagnostic cell.
# ---------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd

from model.embeddings import SentenceTransformerEncoder


# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

FINE_TUNED_BGE_M3_DIR = Path(
    r"E:\MLCache\news-flow-analysis\models\bge_m3_ru_paraphrase_mnrl"
)

EXP08_THRESHOLDS = [0.80, 0.81, 0.82, 0.83, 0.84, 0.85, 0.86]
EXP08_STORY_WINDOW_DAYS = 14

FINE_TUNED_EMBEDDINGS_CACHE_PATH = (
    paths.artifacts_dir / "embeddings" / "exp_08_finetuned_bge_m3_candidate_pool_embeddings.npz"
)


# ---------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------

def normalize_news_id_for_join(series: pd.Series) -> pd.Series:
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def compute_pairwise_clustering_metrics_simple(
    reference_cluster_ids,
    candidate_cluster_ids,
) -> dict:
    ref = np.array(list(reference_cluster_ids), dtype=object)
    cand = np.array(list(candidate_cluster_ids), dtype=object)

    tp = fp = fn = tn = 0
    n = len(ref)

    for i in range(n):
        for j in range(i + 1, n):
            ref_same = ref[i] == ref[j]
            cand_same = cand[i] == cand[j]

            if ref_same and cand_same:
                tp += 1
            elif not ref_same and cand_same:
                fp += 1
            elif ref_same and not cand_same:
                fn += 1
            else:
                tn += 1

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    false_merge_rate = fp / (tp + fp) if (tp + fp) else 0.0

    return {
        "n_items": n,
        "total_pairs": n * (n - 1) // 2,
        "tp_same_pairs": tp,
        "fp_false_merge_pairs": fp,
        "fn_missed_same_pairs": fn,
        "pairwise_precision": precision,
        "pairwise_recall": recall,
        "pairwise_f1": f1,
        "false_merge_rate": false_merge_rate,
    }


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]

        return x

    def union(self, a: int, b: int):
        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return

        if self.rank[root_a] < self.rank[root_b]:
            self.parent[root_a] = root_b
        elif self.rank[root_a] > self.rank[root_b]:
            self.parent[root_b] = root_a
        else:
            self.parent[root_b] = root_a
            self.rank[root_a] += 1


def threshold_graph_cluster_news(
    news_df: pd.DataFrame,
    embeddings: np.ndarray,
    *,
    story_threshold: float,
    story_window_days: int,
    cluster_prefix: str,
) -> tuple[pd.Series, int]:
    """Threshold graph clustering with topic blocking and temporal window.

    Edges:
      same topic
      abs(published_at_i - published_at_j) <= story_window_days
      cosine_similarity >= story_threshold

    Embeddings are expected to be normalized.
    """

    work = news_df[["news_id", "topic", "published_at"]].copy()
    work["published_at"] = pd.to_datetime(work["published_at"], errors="coerce")
    work["topic"] = work["topic"].fillna("<missing>").astype(str)

    n = len(work)
    union_find = UnionFind(n)
    edge_count = 0

    for _, topic_part in work.groupby("topic", sort=False):
        idx = topic_part.index.to_numpy()
        local_dates = topic_part["published_at"].to_numpy()
        local_embeddings = embeddings[idx]

        # cosine sim because embeddings are normalized
        sim_matrix = local_embeddings @ local_embeddings.T

        for local_i in range(len(idx)):
            global_i = idx[local_i]
            date_i = local_dates[local_i]

            for local_j in range(local_i + 1, len(idx)):
                global_j = idx[local_j]

                days_diff = abs((local_dates[local_j] - date_i) / np.timedelta64(1, "D"))

                if days_diff > story_window_days:
                    continue

                if sim_matrix[local_i, local_j] >= story_threshold:
                    union_find.union(global_i, global_j)
                    edge_count += 1

    roots = [union_find.find(i) for i in range(n)]

    root_to_cluster = {}
    cluster_ids = []

    for root in roots:
        if root not in root_to_cluster:
            root_to_cluster[root] = f"{cluster_prefix}_{len(root_to_cluster):05d}"

        cluster_ids.append(root_to_cluster[root])

    return pd.Series(cluster_ids, index=news_df.index, name="cluster_id"), edge_count


def evaluate_cluster_ids_on_golden(
    golden_df: pd.DataFrame,
    candidate_news_df: pd.DataFrame,
    candidate_cluster_ids: pd.Series,
) -> dict:
    candidate = candidate_news_df[["news_id"]].copy()
    candidate["news_id"] = normalize_news_id_for_join(candidate["news_id"])
    candidate["cluster_id"] = candidate_cluster_ids.astype(str).values

    reference = golden_df[["news_id", "cluster_id"]].copy()
    reference["news_id"] = normalize_news_id_for_join(reference["news_id"])
    reference["cluster_id"] = reference["cluster_id"].astype(str)

    merged = reference.merge(
        candidate,
        on="news_id",
        how="inner",
        suffixes=("_ref", "_cand"),
    )

    metrics = compute_pairwise_clustering_metrics_simple(
        merged["cluster_id_ref"],
        merged["cluster_id_cand"],
    )

    metrics["coverage"] = len(merged) / len(reference) if len(reference) else 0.0

    return metrics


def run_exp08_threshold_sweep(
    *,
    model_name: str,
    news_df: pd.DataFrame,
    embeddings: np.ndarray,
    thresholds: list[float],
    story_window_days: int,
) -> tuple[pd.DataFrame, dict[float, pd.Series]]:
    rows = []
    cluster_ids_by_threshold = {}

    for threshold in thresholds:
        cluster_ids, edge_count = threshold_graph_cluster_news(
            news_df=news_df,
            embeddings=embeddings,
            story_threshold=threshold,
            story_window_days=story_window_days,
            cluster_prefix=f"{model_name}_{threshold:.2f}",
        )

        cluster_ids_by_threshold[threshold] = cluster_ids

        metrics = evaluate_cluster_ids_on_golden(
            golden_df=golden,
            candidate_news_df=news_df,
            candidate_cluster_ids=cluster_ids,
        )

        rows.append({
            "model": model_name,
            "story_threshold": threshold,
            "edge_count": edge_count,
            "n_clusters": int(cluster_ids.nunique()),
            **metrics,
        })

    return pd.DataFrame(rows), cluster_ids_by_threshold


# ---------------------------------------------------------------------
# Prepare candidate pool
# ---------------------------------------------------------------------

if not FINE_TUNED_BGE_M3_DIR.exists():
    raise FileNotFoundError(
        f"Fine-tuned model not found: {FINE_TUNED_BGE_M3_DIR}"
    )

candidate_news_exp08 = old_pipeline_clustered_news.copy()

# Убираем старые предсказательные колонки, чтобы не спутать старые cluster_id с новыми.
candidate_news_exp08 = candidate_news_exp08.drop(
    columns=[
        "cluster_id",
        "novelty_label",
        "needs_review",
        "comment",
    ],
    errors="ignore",
).reset_index(drop=True)

candidate_news_exp08["news_id"] = normalize_news_id_for_join(candidate_news_exp08["news_id"])

print("candidate_news_exp08:", candidate_news_exp08.shape)
print("old_pipeline_embeddings:", old_pipeline_embeddings.shape)


# ---------------------------------------------------------------------
# Encode with fine-tuned BGE-M3
# ---------------------------------------------------------------------

finetuned_encoder = SentenceTransformerEncoder(
    model_name=str(FINE_TUNED_BGE_M3_DIR),
    batch_size=16,
    normalize_embeddings=True,
    show_progress_bar=True,
)

finetuned_embeddings_exp08 = finetuned_encoder.encode_dataframe(
    candidate_news_exp08,
    text_column="model_text",
    id_column="news_id",
    cache_path=FINE_TUNED_EMBEDDINGS_CACHE_PATH,
    force_recompute=False,
)

print("finetuned_embeddings_exp08:", finetuned_embeddings_exp08.shape)


# ---------------------------------------------------------------------
# Threshold sweep: base BGE-M3 vs fine-tuned BGE-M3
# ---------------------------------------------------------------------

base_sweep_exp08, base_cluster_ids_exp08 = run_exp08_threshold_sweep(
    model_name="base_bge_m3",
    news_df=candidate_news_exp08,
    embeddings=old_pipeline_embeddings,
    thresholds=EXP08_THRESHOLDS,
    story_window_days=EXP08_STORY_WINDOW_DAYS,
)

finetuned_sweep_exp08, finetuned_cluster_ids_exp08 = run_exp08_threshold_sweep(
    model_name="finetuned_bge_m3",
    news_df=candidate_news_exp08,
    embeddings=finetuned_embeddings_exp08,
    thresholds=EXP08_THRESHOLDS,
    story_window_days=EXP08_STORY_WINDOW_DAYS,
)

exp08_sweep = pd.concat(
    [base_sweep_exp08, finetuned_sweep_exp08],
    ignore_index=True,
)

exp08_sweep_sorted = (
    exp08_sweep
    .sort_values(
        ["false_merge_rate", "pairwise_f1", "pairwise_recall"],
        ascending=[True, False, False],
    )
    .reset_index(drop=True)
)

display(exp08_sweep_sorted)

exp08_sweep_path = paths.artifacts_dir / "exp_08_finetuned_bge_m3_threshold_sweep.csv"
exp08_sweep.to_csv(exp08_sweep_path, index=False)

print("Saved:", exp08_sweep_path)

candidate_news_exp08: (3176, 16)
old_pipeline_embeddings: (3176, 1024)


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

finetuned_embeddings_exp08: (3176, 1024)


,model,story_threshold,edge_count,n_clusters,n_items,total_pairs,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate,coverage
0,base_bge_m3,0.82,914,2607,121,7260,117,0,70,1.000000,0.625668,0.769737,0.000000,1.0
1,finetuned_bge_m3,0.82,828,2639,121,7260,102,0,85,1.000000,0.545455,0.705882,0.000000,1.0
2,base_bge_m3,0.83,748,2695,121,7260,94,0,93,1.000000,0.502674,0.669039,0.000000,1.0
3,finetuned_bge_m3,0.83,685,2727,121,7260,80,0,107,1.000000,0.427807,0.599251,0.000000,1.0
4,base_bge_m3,0.84,608,2774,121,7260,77,0,110,1.000000,0.411765,0.583333,0.000000,1.0
5,finetuned_bge_m3,0.84,568,2795,121,7260,71,0,116,1.000000,0.379679,0.550388,0.000000,1.0
6,base_bge_m3,0.85,489,2842,121,7260,60,0,127,1.000000,0.320856,0.485830,0.000000,1.0
7,finetuned_bge_m3,0.85,464,2864,121,7260,47,0,140,1.000000,0.251337,0.401709,0.000000,1.0
8,base_bge_m3,0.86,396,2908,121,7260,41,0,146,1.000000,0.219251,0.359649,0.000000,1.0
9,finetuned_bge_m3,0.86,366,2927,121,7260,33,0,154,1.000000,0.176471,0.300000,0.000000,1.0


Saved: E:\ML\Projects\Git\news-flow-analysis\data\artifacts\exp_08_finetuned_bge_m3_threshold_sweep.csv


In [36]:
# ---------------------------------------------------------------------
# Experiment 08 summary
# ---------------------------------------------------------------------

safe_exp08 = exp08_sweep[exp08_sweep["false_merge_rate"] == 0.0].copy()

print("Best rows with false_merge_rate == 0:")
display(
    safe_exp08
    .sort_values(["pairwise_f1", "pairwise_recall"], ascending=False)
    .reset_index(drop=True)
    .head(10)
)

print("Best rows overall:")
display(
    exp08_sweep
    .sort_values(["pairwise_f1", "false_merge_rate"], ascending=[False, True])
    .reset_index(drop=True)
    .head(10)
)

Best rows with false_merge_rate == 0:


,model,story_threshold,edge_count,n_clusters,n_items,total_pairs,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate,coverage
0,base_bge_m3,0.82,914,2607,121,7260,117,0,70,1.0,0.625668,0.769737,0.0,1.0
1,finetuned_bge_m3,0.82,828,2639,121,7260,102,0,85,1.0,0.545455,0.705882,0.0,1.0
2,base_bge_m3,0.83,748,2695,121,7260,94,0,93,1.0,0.502674,0.669039,0.0,1.0
3,finetuned_bge_m3,0.83,685,2727,121,7260,80,0,107,1.0,0.427807,0.599251,0.0,1.0
4,base_bge_m3,0.84,608,2774,121,7260,77,0,110,1.0,0.411765,0.583333,0.0,1.0
5,finetuned_bge_m3,0.84,568,2795,121,7260,71,0,116,1.0,0.379679,0.550388,0.0,1.0
6,base_bge_m3,0.85,489,2842,121,7260,60,0,127,1.0,0.320856,0.485830,0.0,1.0
7,finetuned_bge_m3,0.85,464,2864,121,7260,47,0,140,1.0,0.251337,0.401709,0.0,1.0
8,base_bge_m3,0.86,396,2908,121,7260,41,0,146,1.0,0.219251,0.359649,0.0,1.0
9,finetuned_bge_m3,0.86,366,2927,121,7260,33,0,154,1.0,0.176471,0.300000,0.0,1.0


Best rows overall:


,model,story_threshold,edge_count,n_clusters,n_items,total_pairs,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate,coverage
0,base_bge_m3,0.81,1091,2529,121,7260,127,8,60,0.940741,0.679144,0.788820,0.059259,1.0
1,base_bge_m3,0.82,914,2607,121,7260,117,0,70,1.000000,0.625668,0.769737,0.000000,1.0
2,finetuned_bge_m3,0.80,1175,2473,121,7260,118,8,69,0.936508,0.631016,0.753994,0.063492,1.0
3,base_bge_m3,0.80,1316,2441,121,7260,144,53,43,0.730964,0.770053,0.750000,0.269036,1.0
4,finetuned_bge_m3,0.81,978,2563,121,7260,112,8,75,0.933333,0.598930,0.729642,0.066667,1.0
5,finetuned_bge_m3,0.82,828,2639,121,7260,102,0,85,1.000000,0.545455,0.705882,0.000000,1.0
6,base_bge_m3,0.83,748,2695,121,7260,94,0,93,1.000000,0.502674,0.669039,0.000000,1.0
7,finetuned_bge_m3,0.83,685,2727,121,7260,80,0,107,1.000000,0.427807,0.599251,0.000000,1.0
8,base_bge_m3,0.84,608,2774,121,7260,77,0,110,1.000000,0.411765,0.583333,0.000000,1.0
9,finetuned_bge_m3,0.84,568,2795,121,7260,71,0,116,1.000000,0.379679,0.550388,0.000000,1.0


In [57]:
# ---------------------------------------------------------------------
# exp_09: second-pass attach clustering
# ---------------------------------------------------------------------
# Цель:
#   Улучшить conservative clustering без роста false merge.
#
# Базовый кластеризатор:
#   same topic
#   days_diff <= 14
#   cosine_similarity >= 0.82
#
# Second-pass attach:
#   пробуем добавить связи ниже 0.82, но только если есть дополнительные
#   признаки связи: близкая дата, title overlap, shared numbers или similarity
#   близко к текущему threshold.
#
# Важно:
#   Это diagnostic experiment. Параметры подбираются по golden, поэтому
#   если найдём улучшение, его надо трактовать как гипотезу для дальнейшей
#   проверки, а не как fully validated production result.
# ---------------------------------------------------------------------

from itertools import product
import re
import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Config
# ---------------------------------------------------------------------

EXP09_BASE_THRESHOLD = 0.82
EXP09_STORY_WINDOW_DAYS = 14

EXP09_MIN_RESCUE_SIMILARITIES = [0.75, 0.78, 0.80]
EXP09_MAX_RESCUE_DAYS = [1, 3, 7, 14]
EXP09_TITLE_JACCARD_THRESHOLDS = [0.10, 0.15, 0.20]

# Если similarity >= 0.80 и дата близкая, разрешаем rescue даже без title_jaccard/shared_numbers.
EXP09_STRONG_RESCUE_SIMILARITY = 0.80

# Если в двух новостях встречается одно и то же число, это дополнительный сигнал связи.
EXP09_MIN_SHARED_NUMBERS = 1

EXP09_EXPERIMENT_NAME = "exp_09_second_pass_attach_clustering"


# ---------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------

def normalize_news_id_for_join(series: pd.Series) -> pd.Series:
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_embedding_matrix(embeddings: np.ndarray) -> np.ndarray:
    embeddings = np.asarray(embeddings, dtype=np.float32)
    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)

    return embeddings / norms


def tokenize_for_jaccard(text: str) -> set[str]:
    if pd.isna(text):
        return set()

    text = str(text).lower()
    tokens = re.findall(r"[a-zа-яё0-9]+", text)

    return {
        token
        for token in tokens
        if len(token) >= 3
    }


def extract_numbers(text: str) -> set[str]:
    if pd.isna(text):
        return set()

    text = str(text).lower().replace(",", ".")

    return set(re.findall(r"\d+(?:\.\d+)?", text))


def jaccard(left: set[str], right: set[str]) -> float:
    if not left and not right:
        return 0.0

    union = left | right

    if not union:
        return 0.0

    return len(left & right) / len(union)


class UnionFind:
    def __init__(self, n: int):
        self.parent = list(range(n))
        self.rank = [0] * n

    def find(self, x: int) -> int:
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]

        return x

    def union(self, a: int, b: int):
        root_a = self.find(a)
        root_b = self.find(b)

        if root_a == root_b:
            return

        if self.rank[root_a] < self.rank[root_b]:
            self.parent[root_a] = root_b
        elif self.rank[root_a] > self.rank[root_b]:
            self.parent[root_b] = root_a
        else:
            self.parent[root_b] = root_a
            self.rank[root_a] += 1


def compute_pairwise_clustering_metrics(
    reference_cluster_ids,
    candidate_cluster_ids,
) -> dict:
    ref = np.asarray(list(reference_cluster_ids), dtype=object)
    cand = np.asarray(list(candidate_cluster_ids), dtype=object)

    tp = fp = fn = tn = 0
    n = len(ref)

    for i in range(n):
        for j in range(i + 1, n):
            ref_same = ref[i] == ref[j]
            cand_same = cand[i] == cand[j]

            if ref_same and cand_same:
                tp += 1
            elif not ref_same and cand_same:
                fp += 1
            elif ref_same and not cand_same:
                fn += 1
            else:
                tn += 1

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall)
        else 0.0
    )
    false_merge_rate = fp / (tp + fp) if (tp + fp) else 0.0

    return {
        "n_items": n,
        "total_pairs": n * (n - 1) // 2,
        "tp_same_pairs": tp,
        "fp_false_merge_pairs": fp,
        "fn_missed_same_pairs": fn,
        "pairwise_precision": precision,
        "pairwise_recall": recall,
        "pairwise_f1": f1,
        "false_merge_rate": false_merge_rate,
    }


def evaluate_cluster_ids_on_golden(
    *,
    golden_df: pd.DataFrame,
    candidate_news_df: pd.DataFrame,
    candidate_cluster_ids: pd.Series,
) -> dict:
    reference = golden_df[["news_id", "cluster_id"]].copy()
    reference["news_id"] = normalize_news_id_for_join(reference["news_id"])
    reference["cluster_id"] = reference["cluster_id"].astype(str)

    candidate = candidate_news_df[["news_id"]].copy()
    candidate["news_id"] = normalize_news_id_for_join(candidate["news_id"])
    candidate["cluster_id"] = candidate_cluster_ids.astype(str).values

    merged = reference.merge(
        candidate,
        on="news_id",
        how="inner",
        suffixes=("_ref", "_cand"),
    )

    metrics = compute_pairwise_clustering_metrics(
        reference_cluster_ids=merged["cluster_id_ref"],
        candidate_cluster_ids=merged["cluster_id_cand"],
    )

    metrics["coverage"] = len(merged) / len(reference) if len(reference) else 0.0

    return metrics


def prepare_exp09_news_df(clustered_news_df: pd.DataFrame) -> pd.DataFrame:
    news_df = clustered_news_df.copy()

    news_df = news_df.drop(
        columns=[
            "cluster_id",
            "novelty_label",
            "needs_review",
            "comment",
            "p_significant",
            "max_prev_similarity",
        ],
        errors="ignore",
    ).reset_index(drop=True)

    news_df["news_id"] = normalize_news_id_for_join(news_df["news_id"])
    news_df["topic"] = news_df["topic"].fillna("<missing>").astype(str)
    news_df["published_at"] = pd.to_datetime(news_df["published_at"], errors="coerce")
    news_df["title"] = news_df["title"].fillna("").astype(str)

    if "text" not in news_df.columns:
        news_df["text"] = ""

    news_df["text"] = news_df["text"].fillna("").astype(str)

    return news_df


def precompute_candidate_pairs(
    *,
    news_df: pd.DataFrame,
    embeddings: np.ndarray,
    max_window_days: int,
) -> pd.DataFrame:
    """Precompute same-topic pairs inside max time window with extra pair features."""

    embeddings_norm = normalize_embedding_matrix(embeddings)

    title_tokens = [
        tokenize_for_jaccard(title)
        for title in news_df["title"]
    ]

    number_sets = [
        extract_numbers(title + " " + text)
        for title, text in zip(news_df["title"], news_df["text"])
    ]

    rows = []

    for topic, topic_part in news_df.groupby("topic", sort=False):
        topic_part = topic_part.sort_values(["published_at", "news_id"], kind="mergesort")
        idx = topic_part.index.to_numpy()

        if len(idx) <= 1:
            continue

        local_dates = topic_part["published_at"].to_numpy(dtype="datetime64[ns]")
        local_embeddings = embeddings_norm[idx]

        # Для candidate pool это нормально по памяти.
        sim_matrix = local_embeddings @ local_embeddings.T

        for local_i in range(len(idx)):
            global_i = int(idx[local_i])
            date_i = local_dates[local_i]

            if pd.isna(date_i):
                continue

            for local_j in range(local_i + 1, len(idx)):
                global_j = int(idx[local_j])
                date_j = local_dates[local_j]

                if pd.isna(date_j):
                    continue

                days_diff = float((date_j - date_i) / np.timedelta64(1, "D"))

                if days_diff > max_window_days:
                    break

                title_jaccard = jaccard(
                    title_tokens[global_i],
                    title_tokens[global_j],
                )

                shared_numbers_count = len(
                    number_sets[global_i] & number_sets[global_j]
                )

                rows.append({
                    "left_idx": global_i,
                    "right_idx": global_j,
                    "topic": topic,
                    "days_diff": days_diff,
                    "similarity": float(sim_matrix[local_i, local_j]),
                    "title_jaccard": title_jaccard,
                    "shared_numbers_count": shared_numbers_count,
                    "left_news_id": news_df.loc[global_i, "news_id"],
                    "right_news_id": news_df.loc[global_j, "news_id"],
                    "left_title": news_df.loc[global_i, "title"],
                    "right_title": news_df.loc[global_j, "title"],
                })

    return pd.DataFrame(rows)


def build_clusters_from_edges(
    *,
    news_df: pd.DataFrame,
    candidate_pairs: pd.DataFrame,
    base_threshold: float,
    base_window_days: int,
    enable_second_pass: bool,
    min_rescue_similarity: float | None = None,
    max_rescue_days: int | None = None,
    title_jaccard_threshold: float | None = None,
    strong_rescue_similarity: float | None = None,
    min_shared_numbers: int | None = None,
    cluster_prefix: str = "exp09",
) -> tuple[pd.Series, dict]:
    n = len(news_df)
    union_find = UnionFind(n)

    base_edges = candidate_pairs[
        (candidate_pairs["days_diff"] <= base_window_days)
        & (candidate_pairs["similarity"] >= base_threshold)
    ].copy()

    for row in base_edges.itertuples(index=False):
        union_find.union(int(row.left_idx), int(row.right_idx))

    second_pass_edges = candidate_pairs.iloc[0:0].copy()

    if enable_second_pass:
        if min_rescue_similarity is None:
            raise ValueError("min_rescue_similarity is required for second pass")

        if max_rescue_days is None:
            raise ValueError("max_rescue_days is required for second pass")

        if title_jaccard_threshold is None:
            raise ValueError("title_jaccard_threshold is required for second pass")

        if strong_rescue_similarity is None:
            raise ValueError("strong_rescue_similarity is required for second pass")

        if min_shared_numbers is None:
            raise ValueError("min_shared_numbers is required for second pass")

        rescue_candidate_mask = (
            (candidate_pairs["days_diff"] <= max_rescue_days)
            & (candidate_pairs["similarity"] >= min_rescue_similarity)
            & (candidate_pairs["similarity"] < base_threshold)
        )

        rescue_evidence_mask = (
            (candidate_pairs["similarity"] >= strong_rescue_similarity)
            | (candidate_pairs["title_jaccard"] >= title_jaccard_threshold)
            | (candidate_pairs["shared_numbers_count"] >= min_shared_numbers)
        )

        second_pass_edges = candidate_pairs[
            rescue_candidate_mask & rescue_evidence_mask
        ].copy()

        for row in second_pass_edges.itertuples(index=False):
            union_find.union(int(row.left_idx), int(row.right_idx))

    roots = [union_find.find(i) for i in range(n)]

    root_to_cluster = {}
    cluster_ids = []

    for root in roots:
        if root not in root_to_cluster:
            root_to_cluster[root] = f"{cluster_prefix}_{len(root_to_cluster):05d}"

        cluster_ids.append(root_to_cluster[root])

    diagnostics = {
        "base_edge_count": len(base_edges),
        "second_pass_edge_count": len(second_pass_edges),
        "total_edge_count": len(base_edges) + len(second_pass_edges),
        "n_clusters": len(set(cluster_ids)),
    }

    return pd.Series(cluster_ids, index=news_df.index, name="cluster_id"), diagnostics


# ---------------------------------------------------------------------
# Prepare data and candidate pairs
# ---------------------------------------------------------------------

required_objects = [
    "golden",
    "old_pipeline_clustered_news",
    "old_pipeline_embeddings",
]

missing_objects = [
    name
    for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(f"Missing required objects: {missing_objects}")

exp09_news_df = prepare_exp09_news_df(old_pipeline_clustered_news)

print("exp09_news_df:", exp09_news_df.shape)
print("old_pipeline_embeddings:", old_pipeline_embeddings.shape)

if len(exp09_news_df) != len(old_pipeline_embeddings):
    raise ValueError(
        "News dataframe and embeddings must be aligned and have the same number of rows."
    )

candidate_pairs_exp09 = precompute_candidate_pairs(
    news_df=exp09_news_df,
    embeddings=old_pipeline_embeddings,
    max_window_days=EXP09_STORY_WINDOW_DAYS,
)

print("candidate_pairs_exp09:", candidate_pairs_exp09.shape)

display(
    candidate_pairs_exp09[
        [
            "similarity",
            "days_diff",
            "title_jaccard",
            "shared_numbers_count",
        ]
    ].describe()
)


# ---------------------------------------------------------------------
# Baseline clustering row
# ---------------------------------------------------------------------

baseline_cluster_ids_exp09, baseline_diag_exp09 = build_clusters_from_edges(
    news_df=exp09_news_df,
    candidate_pairs=candidate_pairs_exp09,
    base_threshold=EXP09_BASE_THRESHOLD,
    base_window_days=EXP09_STORY_WINDOW_DAYS,
    enable_second_pass=False,
    cluster_prefix="exp09_baseline",
)

baseline_metrics_exp09 = evaluate_cluster_ids_on_golden(
    golden_df=golden,
    candidate_news_df=exp09_news_df,
    candidate_cluster_ids=baseline_cluster_ids_exp09,
)

baseline_row_exp09 = {
    "variant": "baseline_strict_0.82",
    "enable_second_pass": False,
    "min_rescue_similarity": np.nan,
    "max_rescue_days": np.nan,
    "title_jaccard_threshold": np.nan,
    "strong_rescue_similarity": np.nan,
    "min_shared_numbers": np.nan,
    **baseline_diag_exp09,
    **baseline_metrics_exp09,
}


# ---------------------------------------------------------------------
# Second-pass sweep
# ---------------------------------------------------------------------

rows = [baseline_row_exp09]
cluster_ids_by_variant_exp09 = {
    "baseline_strict_0.82": baseline_cluster_ids_exp09,
}

for min_rescue_similarity, max_rescue_days, title_jaccard_threshold in product(
    EXP09_MIN_RESCUE_SIMILARITIES,
    EXP09_MAX_RESCUE_DAYS,
    EXP09_TITLE_JACCARD_THRESHOLDS,
):
    variant_name = (
        f"second_pass"
        f"_sim{min_rescue_similarity:.2f}"
        f"_days{max_rescue_days}"
        f"_tj{title_jaccard_threshold:.2f}"
    )

    cluster_ids, diagnostics = build_clusters_from_edges(
        news_df=exp09_news_df,
        candidate_pairs=candidate_pairs_exp09,
        base_threshold=EXP09_BASE_THRESHOLD,
        base_window_days=EXP09_STORY_WINDOW_DAYS,
        enable_second_pass=True,
        min_rescue_similarity=min_rescue_similarity,
        max_rescue_days=max_rescue_days,
        title_jaccard_threshold=title_jaccard_threshold,
        strong_rescue_similarity=EXP09_STRONG_RESCUE_SIMILARITY,
        min_shared_numbers=EXP09_MIN_SHARED_NUMBERS,
        cluster_prefix=variant_name,
    )

    metrics = evaluate_cluster_ids_on_golden(
        golden_df=golden,
        candidate_news_df=exp09_news_df,
        candidate_cluster_ids=cluster_ids,
    )

    rows.append({
        "variant": variant_name,
        "enable_second_pass": True,
        "min_rescue_similarity": min_rescue_similarity,
        "max_rescue_days": max_rescue_days,
        "title_jaccard_threshold": title_jaccard_threshold,
        "strong_rescue_similarity": EXP09_STRONG_RESCUE_SIMILARITY,
        "min_shared_numbers": EXP09_MIN_SHARED_NUMBERS,
        **diagnostics,
        **metrics,
    })

    cluster_ids_by_variant_exp09[variant_name] = cluster_ids

exp09_clustering_sweep = pd.DataFrame(rows)

exp09_clustering_sweep_sorted = (
    exp09_clustering_sweep
    .sort_values(
        [
            "fp_false_merge_pairs",
            "false_merge_rate",
            "pairwise_f1",
            "pairwise_recall",
            "second_pass_edge_count",
        ],
        ascending=[True, True, False, False, True],
    )
    .reset_index(drop=True)
)

display(exp09_clustering_sweep_sorted)


# ---------------------------------------------------------------------
# Select best safe variant
# ---------------------------------------------------------------------

baseline_pairwise_f1 = float(baseline_metrics_exp09["pairwise_f1"])

safe_variants_exp09 = exp09_clustering_sweep[
    exp09_clustering_sweep["fp_false_merge_pairs"].eq(0)
].copy()

safe_variants_exp09 = (
    safe_variants_exp09
    .sort_values(
        ["pairwise_f1", "pairwise_recall", "second_pass_edge_count"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

print("Best safe variants:")
display(safe_variants_exp09.head(15))

best_safe_variant_exp09 = safe_variants_exp09.iloc[0].to_dict()

print("Baseline pairwise_f1:", baseline_pairwise_f1)
print("Best safe variant:", best_safe_variant_exp09["variant"])
print("Best safe pairwise_f1:", best_safe_variant_exp09["pairwise_f1"])
print("Best safe false_merge_rate:", best_safe_variant_exp09["false_merge_rate"])

exp09_has_safe_clustering_improvement = (
    best_safe_variant_exp09["variant"] != "baseline_strict_0.82"
    and best_safe_variant_exp09["pairwise_f1"] > baseline_pairwise_f1
    and best_safe_variant_exp09["fp_false_merge_pairs"] == 0
)

print("Has safe clustering improvement:", exp09_has_safe_clustering_improvement)


# ---------------------------------------------------------------------
# Optional: evaluate existing novelty model on best safe clustering
# ---------------------------------------------------------------------
# Если нашли safe-улучшение кластеризации, пробуем прогнать текущую novelty-модель
# поверх новых кластеров. Это НЕ переобучение classifier, а быстрый sanity-check:
# помогает ли улучшенный контекст уже существующей модели.
# ---------------------------------------------------------------------

if exp09_has_safe_clustering_improvement:
    exp09_best_variant_name = str(best_safe_variant_exp09["variant"])
    exp09_best_cluster_ids = cluster_ids_by_variant_exp09[exp09_best_variant_name]

    exp09_best_clustered_news = exp09_news_df.copy()
    exp09_best_clustered_news["cluster_id"] = exp09_best_cluster_ids.values

    print("exp09_best_clustered_news:", exp09_best_clustered_news.shape)
    print("clusters:", exp09_best_clustered_news["cluster_id"].nunique())

    if (
        "current_model" in globals()
        and hasattr(current_model, "predict_clustered_with_fallback")
        and "evaluate_predictions" in globals()
    ):
        pred_09 = current_model.predict_clustered_with_fallback(
            news_df=exp09_best_clustered_news,
            embeddings=old_pipeline_embeddings,
        )

        exp09_prediction_path = None

        if "paths" in globals() and "save_prediction_csv" in globals():
            exp09_prediction_path = (
                paths.predictions_dir / f"{EXP09_EXPERIMENT_NAME}.csv"
            )

            save_prediction_csv(pred_09, exp09_prediction_path)

            print("Saved prediction:", exp09_prediction_path)

        metrics_09 = evaluate_predictions(golden, pred_09)

        print("exp_09 metrics:")
        display(pd.DataFrame([metrics_09]))

        if (
            "results_table" in globals()
            and "add_experiment_result" in globals()
        ):
            results_table = results_table[
                results_table["experiment"] != EXP09_EXPERIMENT_NAME
            ].copy()

            results_table = add_experiment_result(
                results_table,
                experiment_name=EXP09_EXPERIMENT_NAME,
                metrics=metrics_09,
                prediction_path=(
                    str(exp09_prediction_path)
                    if exp09_prediction_path is not None
                    else ""
                ),
                embedding_variant="BAAI/bge-m3 id-aligned",
                clustering=(
                    "Strict threshold graph + diagnostic second-pass attach "
                    f"({exp09_best_variant_name})"
                ),
                novelty_variant=(
                    "Existing current_model novelty classifier, no retraining"
                ),
                comment=(
                    "Diagnostic second-pass attach clustering. "
                    "Improves clustering only if false_merge remains zero; "
                    "novelty classifier is reused without retraining."
                ),
            )

            display(compact_metrics_table(results_table))

    else:
        print(
            "current_model/evaluate_predictions not available; "
            "skipping novelty evaluation."
        )

else:
    print(
        "No safe clustering improvement over baseline. "
        "Skipping novelty evaluation for exp_09."
    )

exp09_news_df: (3176, 16)
old_pipeline_embeddings: (3176, 1024)
candidate_pairs_exp09: (881465, 11)


,similarity,days_diff,title_jaccard,shared_numbers_count
count,881465.000000,881465.000000,881465.000000,881465.000000
mean,0.345971,6.915592,0.002472,0.216520
std,0.087004,4.220944,0.016368,0.540544
min,0.072605,0.000000,0.000000,0.000000
25%,0.287910,3.000000,0.000000,0.000000
50%,0.335327,7.000000,0.000000,0.000000
75%,0.390309,10.000000,0.000000,0.000000
max,0.988150,14.000000,1.000000,18.000000


,variant,enable_second_pass,min_rescue_similarity,max_rescue_days,title_jaccard_threshold,strong_rescue_similarity,min_shared_numbers,base_edge_count,second_pass_edge_count,total_edge_count,...,n_items,total_pairs,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate,coverage
0,baseline_strict_0.82,False,NaN,NaN,NaN,NaN,NaN,914,0,914,...,121,7260,117,0,70,1.000000,0.625668,0.769737,0.000000,1.0
1,second_pass_sim0.80_days14_tj0.10,True,0.80,14.0,0.10,0.8,1.0,914,402,1316,...,121,7260,144,53,43,0.730964,0.770053,0.750000,0.269036,1.0
2,second_pass_sim0.80_days14_tj0.15,True,0.80,14.0,0.15,0.8,1.0,914,402,1316,...,121,7260,144,53,43,0.730964,0.770053,0.750000,0.269036,1.0
3,second_pass_sim0.80_days14_tj0.20,True,0.80,14.0,0.20,0.8,1.0,914,402,1316,...,121,7260,144,53,43,0.730964,0.770053,0.750000,0.269036,1.0
4,second_pass_sim0.80_days7_tj0.10,True,0.80,7.0,0.10,0.8,1.0,914,338,1252,...,121,7260,139,53,48,0.723958,0.743316,0.733509,0.276042,1.0
5,second_pass_sim0.80_days7_tj0.15,True,0.80,7.0,0.15,0.8,1.0,914,338,1252,...,121,7260,139,53,48,0.723958,0.743316,0.733509,0.276042,1.0
6,second_pass_sim0.80_days7_tj0.20,True,0.80,7.0,0.20,0.8,1.0,914,338,1252,...,121,7260,139,53,48,0.723958,0.743316,0.733509,0.276042,1.0
7,second_pass_sim0.80_days3_tj0.10,True,0.80,3.0,0.10,0.8,1.0,914,267,1181,...,121,7260,133,53,54,0.715054,0.711230,0.713137,0.284946,1.0
8,second_pass_sim0.80_days3_tj0.15,True,0.80,3.0,0.15,0.8,1.0,914,267,1181,...,121,7260,133,53,54,0.715054,0.711230,0.713137,0.284946,1.0
9,second_pass_sim0.80_days3_tj0.20,True,0.80,3.0,0.20,0.8,1.0,914,267,1181,...,121,7260,133,53,54,0.715054,0.711230,0.713137,0.284946,1.0


Best safe variants:


,variant,enable_second_pass,min_rescue_similarity,max_rescue_days,title_jaccard_threshold,strong_rescue_similarity,min_shared_numbers,base_edge_count,second_pass_edge_count,total_edge_count,...,n_items,total_pairs,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate,coverage
0,baseline_strict_0.82,False,NaN,NaN,NaN,NaN,NaN,914,0,914,...,121,7260,117,0,70,1.0,0.625668,0.769737,0.0,1.0


Baseline pairwise_f1: 0.7697368421052632
Best safe variant: baseline_strict_0.82
Best safe pairwise_f1: 0.7697368421052632
Best safe false_merge_rate: 0.0
Has safe clustering improvement: False
No safe clustering improvement over baseline. Skipping novelty evaluation for exp_09.


Вывод: эксперимент не удался, если пытаться досклеивать по threshold 0.78 и ниже - начинаются false merge, что точно нам не надо (не надо склеек разных по смысле сюжетов). Просто "влоб" задача не решается.

In [ ]:
# Пробуем более консервативную, но умную склейку второго прохода.
# ---------------------------------------------------------------------
# exp_09b: conservative best-candidate attach clustering
# ---------------------------------------------------------------------
# Goal:
#   More careful second-pass clustering than exp_09.
#
# Difference from exp_09:
#   exp_09 added many rescue edges directly into union-find.
#   This caused false merges.
#
# exp_09b сливает кластеры малые кластеры первого этапа, если:
#   - исходный кластер размера 1, 2
#   - целевой кластер больше исходного
#   - одна тема и временное окно
#   - два лучших кандидата отличаются минимум на X
#
# ---------------------------------------------------------------------

from itertools import product

import numpy as np
import pandas as pd


EXP09B_EXPERIMENT_NAME = "exp_09b_best_candidate_attach_clustering"

EXP09B_MIN_SIMILARITIES = [0.78, 0.80, 0.81]
EXP09B_MAX_DAYS = [1, 3, 7, 14]
EXP09B_MIN_MARGINS = [0.00, 0.02, 0.03, 0.05]

# source cluster can be a singleton or a small fragment;
# target must be strictly larger than source.
EXP09B_SOURCE_MAX_CLUSTER_SIZES = [1, 2]

# Optional evidence filter.
# If enabled, require either title overlap or shared numbers.
EXP09B_REQUIRE_EVIDENCE_OPTIONS = [False, True]
EXP09B_TITLE_JACCARD_THRESHOLD = 0.10
EXP09B_MIN_SHARED_NUMBERS = 1


# ---------------------------------------------------------------------
# Dependency checks
# ---------------------------------------------------------------------

required_exp09_objects = [
    "exp09_news_df",
    "candidate_pairs_exp09",
    "baseline_cluster_ids_exp09",
    "baseline_metrics_exp09",
    "evaluate_cluster_ids_on_golden",
    "golden",
    "old_pipeline_embeddings",
]

missing_exp09_objects = [
    name
    for name in required_exp09_objects
    if name not in globals()
]

if missing_exp09_objects:
    raise RuntimeError(
        "Run exp_09 cell first. Missing objects: "
        f"{missing_exp09_objects}"
    )


# ---------------------------------------------------------------------
# Helper
# ---------------------------------------------------------------------

def relabel_cluster_ids(cluster_ids: pd.Series, prefix: str) -> pd.Series:
    cluster_ids = cluster_ids.astype(str)

    mapping = {}
    relabeled = []

    for cluster_id in cluster_ids:
        if cluster_id not in mapping:
            mapping[cluster_id] = f"{prefix}_{len(mapping):05d}"

        relabeled.append(mapping[cluster_id])

    return pd.Series(relabeled, index=cluster_ids.index, name="cluster_id")


def build_best_candidate_attach_clusters(
    *,
    news_df: pd.DataFrame,
    candidate_pairs: pd.DataFrame,
    base_cluster_ids: pd.Series,
    min_similarity: float,
    max_days: int,
    min_margin: float,
    source_max_cluster_size: int,
    require_evidence: bool,
    title_jaccard_threshold: float,
    min_shared_numbers: int,
    cluster_prefix: str,
) -> tuple[pd.Series, dict, pd.DataFrame]:
    """Attach small stage-1 clusters to larger clusters by best candidate.

    No arbitrary union of all rescue edges:
    each small source cluster can be attached to at most one target cluster.
    """

    work_pairs = candidate_pairs.copy()

    base_cluster_ids = base_cluster_ids.astype(str).reset_index(drop=True)

    cluster_sizes = (
        base_cluster_ids
        .value_counts()
        .rename_axis("cluster_id")
        .reset_index(name="cluster_size")
    )

    cluster_size_map = dict(
        zip(cluster_sizes["cluster_id"], cluster_sizes["cluster_size"])
    )

    idx_to_cluster = base_cluster_ids.to_dict()

    work_pairs["left_cluster"] = work_pairs["left_idx"].map(idx_to_cluster)
    work_pairs["right_cluster"] = work_pairs["right_idx"].map(idx_to_cluster)

    # Only cross-cluster pairs can attach.
    work_pairs = work_pairs[
        work_pairs["left_cluster"].ne(work_pairs["right_cluster"])
    ].copy()

    work_pairs["left_cluster_size"] = work_pairs["left_cluster"].map(cluster_size_map)
    work_pairs["right_cluster_size"] = work_pairs["right_cluster"].map(cluster_size_map)

    base_mask = (
        (work_pairs["days_diff"] <= max_days)
        & (work_pairs["similarity"] >= min_similarity)
    )

    if require_evidence:
        evidence_mask = (
            (work_pairs["title_jaccard"] >= title_jaccard_threshold)
            | (work_pairs["shared_numbers_count"] >= min_shared_numbers)
        )
    else:
        evidence_mask = pd.Series(True, index=work_pairs.index)

    work_pairs = work_pairs[base_mask & evidence_mask].copy()

    oriented_rows = []

    for row in work_pairs.itertuples(index=False):
        left_size = int(row.left_cluster_size)
        right_size = int(row.right_cluster_size)

        # left small fragment -> right larger cluster
        if (
            left_size <= source_max_cluster_size
            and right_size > left_size
        ):
            oriented_rows.append({
                "source_cluster": row.left_cluster,
                "target_cluster": row.right_cluster,
                "source_size": left_size,
                "target_size": right_size,
                "similarity": float(row.similarity),
                "days_diff": float(row.days_diff),
                "title_jaccard": float(row.title_jaccard),
                "shared_numbers_count": int(row.shared_numbers_count),
                "source_news_id": row.left_news_id,
                "target_news_id": row.right_news_id,
                "source_title": row.left_title,
                "target_title": row.right_title,
            })

        # right small fragment -> left larger cluster
        if (
            right_size <= source_max_cluster_size
            and left_size > right_size
        ):
            oriented_rows.append({
                "source_cluster": row.right_cluster,
                "target_cluster": row.left_cluster,
                "source_size": right_size,
                "target_size": left_size,
                "similarity": float(row.similarity),
                "days_diff": float(row.days_diff),
                "title_jaccard": float(row.title_jaccard),
                "shared_numbers_count": int(row.shared_numbers_count),
                "source_news_id": row.right_news_id,
                "target_news_id": row.left_news_id,
                "source_title": row.right_title,
                "target_title": row.left_title,
            })

    oriented_candidates = pd.DataFrame(oriented_rows)

    if oriented_candidates.empty:
        diagnostics = {
            "candidate_attach_edges": 0,
            "attached_source_clusters": 0,
            "attached_rows": 0,
        }

        return (
            relabel_cluster_ids(base_cluster_ids, cluster_prefix),
            diagnostics,
            oriented_candidates,
        )

    # Aggregate source -> target by strongest evidence.
    source_target_scores = (
        oriented_candidates
        .sort_values(
            ["source_cluster", "target_cluster", "similarity", "title_jaccard"],
            ascending=[True, True, False, False],
        )
        .groupby(["source_cluster", "target_cluster"], as_index=False)
        .agg(
            best_similarity=("similarity", "max"),
            min_days_diff=("days_diff", "min"),
            max_title_jaccard=("title_jaccard", "max"),
            max_shared_numbers_count=("shared_numbers_count", "max"),
            source_size=("source_size", "max"),
            target_size=("target_size", "max"),
            source_news_id=("source_news_id", "first"),
            target_news_id=("target_news_id", "first"),
            source_title=("source_title", "first"),
            target_title=("target_title", "first"),
        )
    )

    selected_rows = []

    for source_cluster, part in source_target_scores.groupby("source_cluster"):
        part = (
            part
            .sort_values(
                ["best_similarity", "max_title_jaccard", "max_shared_numbers_count"],
                ascending=[False, False, False],
            )
            .reset_index(drop=True)
        )

        best = part.iloc[0]

        if len(part) > 1:
            second_best_similarity = float(part.iloc[1]["best_similarity"])
        else:
            second_best_similarity = -np.inf

        margin = float(best["best_similarity"]) - second_best_similarity

        if margin >= min_margin:
            selected = best.to_dict()
            selected["second_best_similarity"] = second_best_similarity
            selected["margin"] = margin
            selected_rows.append(selected)

    selected_attachments = pd.DataFrame(selected_rows)

    if selected_attachments.empty:
        diagnostics = {
            "candidate_attach_edges": len(oriented_candidates),
            "attached_source_clusters": 0,
            "attached_rows": 0,
        }

        return (
            relabel_cluster_ids(base_cluster_ids, cluster_prefix),
            diagnostics,
            selected_attachments,
        )

    attach_map = dict(
        zip(
            selected_attachments["source_cluster"],
            selected_attachments["target_cluster"],
        )
    )

    final_cluster_ids_raw = base_cluster_ids.map(
        lambda cluster_id: attach_map.get(cluster_id, cluster_id)
    )

    final_cluster_ids = relabel_cluster_ids(
        final_cluster_ids_raw,
        cluster_prefix,
    )

    attached_source_clusters = set(selected_attachments["source_cluster"])
    attached_rows = int(
        base_cluster_ids
        .isin(attached_source_clusters)
        .sum()
    )

    diagnostics = {
        "candidate_attach_edges": len(oriented_candidates),
        "attached_source_clusters": len(attached_source_clusters),
        "attached_rows": attached_rows,
    }

    return final_cluster_ids, diagnostics, selected_attachments


# ---------------------------------------------------------------------
# Sweep
# ---------------------------------------------------------------------

rows = []
cluster_ids_by_variant_exp09b = {}
attachments_by_variant_exp09b = {}

baseline_pairwise_f1 = float(baseline_metrics_exp09["pairwise_f1"])

# Add baseline row.
cluster_ids_by_variant_exp09b["baseline_strict_0.82"] = baseline_cluster_ids_exp09

rows.append({
    "variant": "baseline_strict_0.82",
    "min_similarity": np.nan,
    "max_days": np.nan,
    "min_margin": np.nan,
    "source_max_cluster_size": np.nan,
    "require_evidence": np.nan,
    "candidate_attach_edges": 0,
    "attached_source_clusters": 0,
    "attached_rows": 0,
    **baseline_metrics_exp09,
})

for (
    min_similarity,
    max_days,
    min_margin,
    source_max_cluster_size,
    require_evidence,
) in product(
    EXP09B_MIN_SIMILARITIES,
    EXP09B_MAX_DAYS,
    EXP09B_MIN_MARGINS,
    EXP09B_SOURCE_MAX_CLUSTER_SIZES,
    EXP09B_REQUIRE_EVIDENCE_OPTIONS,
):
    variant_name = (
        f"exp09b"
        f"_src{source_max_cluster_size}"
        f"_sim{min_similarity:.2f}"
        f"_days{max_days}"
        f"_m{min_margin:.2f}"
        f"_ev{int(require_evidence)}"
    )

    cluster_ids, diagnostics, selected_attachments = build_best_candidate_attach_clusters(
        news_df=exp09_news_df,
        candidate_pairs=candidate_pairs_exp09,
        base_cluster_ids=baseline_cluster_ids_exp09,
        min_similarity=min_similarity,
        max_days=max_days,
        min_margin=min_margin,
        source_max_cluster_size=source_max_cluster_size,
        require_evidence=require_evidence,
        title_jaccard_threshold=EXP09B_TITLE_JACCARD_THRESHOLD,
        min_shared_numbers=EXP09B_MIN_SHARED_NUMBERS,
        cluster_prefix=variant_name,
    )

    metrics = evaluate_cluster_ids_on_golden(
        golden_df=golden,
        candidate_news_df=exp09_news_df,
        candidate_cluster_ids=cluster_ids,
    )

    rows.append({
        "variant": variant_name,
        "min_similarity": min_similarity,
        "max_days": max_days,
        "min_margin": min_margin,
        "source_max_cluster_size": source_max_cluster_size,
        "require_evidence": require_evidence,
        **diagnostics,
        **metrics,
    })

    cluster_ids_by_variant_exp09b[variant_name] = cluster_ids
    attachments_by_variant_exp09b[variant_name] = selected_attachments

exp09b_sweep = pd.DataFrame(rows)

exp09b_sweep_sorted = (
    exp09b_sweep
    .sort_values(
        [
            "fp_false_merge_pairs",
            "false_merge_rate",
            "pairwise_f1",
            "pairwise_recall",
            "attached_source_clusters",
        ],
        ascending=[True, True, False, False, True],
    )
    .reset_index(drop=True)
)

display(exp09b_sweep_sorted)


# ---------------------------------------------------------------------
# Select best safe variant
# ---------------------------------------------------------------------

safe_exp09b = exp09b_sweep[
    exp09b_sweep["fp_false_merge_pairs"].eq(0)
].copy()

safe_exp09b = (
    safe_exp09b
    .sort_values(
        ["pairwise_f1", "pairwise_recall", "attached_source_clusters"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)

print("Best safe exp_09b variants:")
display(safe_exp09b.head(20))

best_safe_exp09b = safe_exp09b.iloc[0].to_dict()

print("Baseline pairwise_f1:", baseline_pairwise_f1)
print("Best safe variant:", best_safe_exp09b["variant"])
print("Best safe pairwise_f1:", best_safe_exp09b["pairwise_f1"])
print("Best safe false_merge_rate:", best_safe_exp09b["false_merge_rate"])

exp09b_has_safe_clustering_improvement = (
    best_safe_exp09b["variant"] != "baseline_strict_0.82"
    and best_safe_exp09b["pairwise_f1"] > baseline_pairwise_f1
    and best_safe_exp09b["fp_false_merge_pairs"] == 0
)

print("Has safe clustering improvement:", exp09b_has_safe_clustering_improvement)


# ---------------------------------------------------------------------
# Inspect selected attachments for best safe variant
# ---------------------------------------------------------------------

best_variant_name_exp09b = str(best_safe_exp09b["variant"])

if best_variant_name_exp09b in attachments_by_variant_exp09b:
    best_attachments_exp09b = attachments_by_variant_exp09b[best_variant_name_exp09b]

    print("Best variant attachments:", best_attachments_exp09b.shape)

    if not best_attachments_exp09b.empty:
        display(
            best_attachments_exp09b
            .sort_values(["best_similarity", "margin"], ascending=[False, False])
            .head(30)
        )


# ---------------------------------------------------------------------
# Optional novelty evaluation with existing current_model
# ---------------------------------------------------------------------

if exp09b_has_safe_clustering_improvement:
    exp09b_best_cluster_ids = cluster_ids_by_variant_exp09b[best_variant_name_exp09b]

    exp09b_best_clustered_news = exp09_news_df.copy()
    exp09b_best_clustered_news["cluster_id"] = exp09b_best_cluster_ids.values

    print("exp09b_best_clustered_news:", exp09b_best_clustered_news.shape)
    print("clusters:", exp09b_best_clustered_news["cluster_id"].nunique())

    if (
        "current_model" in globals()
        and hasattr(current_model, "predict_clustered_with_fallback")
        and "evaluate_predictions" in globals()
    ):
        pred_09b = current_model.predict_clustered_with_fallback(
            news_df=exp09b_best_clustered_news,
            embeddings=old_pipeline_embeddings,
        )

        exp09b_prediction_path = None

        if "paths" in globals() and "save_prediction_csv" in globals():
            exp09b_prediction_path = (
                paths.predictions_dir / f"{EXP09B_EXPERIMENT_NAME}.csv"
            )

            save_prediction_csv(pred_09b, exp09b_prediction_path)

            print("Saved prediction:", exp09b_prediction_path)

        metrics_09b = evaluate_predictions(golden, pred_09b)

        print("exp_09b metrics:")
        display(pd.DataFrame([metrics_09b]))

        if (
            "results_table" in globals()
            and "add_experiment_result" in globals()
        ):
            results_table = results_table[
                results_table["experiment"] != EXP09B_EXPERIMENT_NAME
            ].copy()

            results_table = add_experiment_result(
                results_table,
                experiment_name=EXP09B_EXPERIMENT_NAME,
                metrics=metrics_09b,
                prediction_path=(
                    str(exp09b_prediction_path)
                    if exp09b_prediction_path is not None
                    else ""
                ),
                embedding_variant="BAAI/bge-m3 id-aligned",
                clustering=(
                    "Strict threshold graph + conservative best-candidate attach "
                    f"({best_variant_name_exp09b})"
                ),
                novelty_variant=(
                    "Existing current_model novelty classifier, no retraining"
                ),
                comment=(
                    "Diagnostic conservative attach clustering. "
                    "Only small stage-1 clusters can attach to larger clusters; "
                    "best-candidate margin is required."
                ),
            )

            display(compact_metrics_table(results_table))

    else:
        print(
            "current_model/evaluate_predictions not available; "
            "skipping novelty evaluation."
        )

else:
    print(
        "No safe clustering improvement over baseline. "
        "Skipping novelty evaluation for exp_09b."
    )

,variant,min_similarity,max_days,min_margin,source_max_cluster_size,require_evidence,candidate_attach_edges,attached_source_clusters,attached_rows,n_items,total_pairs,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate,coverage
0,exp09b_src2_sim0.78_days14_m0.03_ev1,0.78,14.0,0.03,2.0,True,192,117,129,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
1,exp09b_src2_sim0.78_days14_m0.05_ev1,0.78,14.0,0.05,2.0,True,192,117,129,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
2,exp09b_src2_sim0.78_days14_m0.02_ev1,0.78,14.0,0.02,2.0,True,192,118,130,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
3,exp09b_src2_sim0.78_days14_m0.00_ev1,0.78,14.0,0.00,2.0,True,192,122,134,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
4,exp09b_src2_sim0.78_days14_m0.05_ev0,0.78,14.0,0.05,2.0,False,306,148,160,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
188,exp09b_src2_sim0.81_days3_m0.02_ev0,0.81,3.0,0.02,2.0,False,29,22,25,121,7260,120,0,67,1.0,0.641711,0.781759,0.0,1.0
189,exp09b_src2_sim0.81_days3_m0.03_ev0,0.81,3.0,0.03,2.0,False,29,22,25,121,7260,120,0,67,1.0,0.641711,0.781759,0.0,1.0
190,exp09b_src2_sim0.81_days3_m0.05_ev0,0.81,3.0,0.05,2.0,False,29,22,25,121,7260,120,0,67,1.0,0.641711,0.781759,0.0,1.0
191,exp09b_src2_sim0.81_days3_m0.00_ev0,0.81,3.0,0.00,2.0,False,29,24,27,121,7260,120,0,67,1.0,0.641711,0.781759,0.0,1.0


Best safe exp_09b variants:


,variant,min_similarity,max_days,min_margin,source_max_cluster_size,require_evidence,candidate_attach_edges,attached_source_clusters,attached_rows,n_items,total_pairs,tp_same_pairs,fp_false_merge_pairs,fn_missed_same_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate,coverage
0,exp09b_src2_sim0.78_days14_m0.03_ev1,0.78,14.0,0.03,2.0,True,192,117,129,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
1,exp09b_src2_sim0.78_days14_m0.05_ev1,0.78,14.0,0.05,2.0,True,192,117,129,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
2,exp09b_src2_sim0.78_days14_m0.02_ev1,0.78,14.0,0.02,2.0,True,192,118,130,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
3,exp09b_src2_sim0.78_days14_m0.00_ev1,0.78,14.0,0.00,2.0,True,192,122,134,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
4,exp09b_src2_sim0.78_days14_m0.05_ev0,0.78,14.0,0.05,2.0,False,306,148,160,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
5,exp09b_src2_sim0.78_days14_m0.03_ev0,0.78,14.0,0.03,2.0,False,306,149,161,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
6,exp09b_src2_sim0.78_days14_m0.02_ev0,0.78,14.0,0.02,2.0,False,306,151,163,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
7,exp09b_src2_sim0.78_days14_m0.00_ev0,0.78,14.0,0.00,2.0,False,306,160,174,121,7260,154,0,33,1.0,0.823529,0.903226,0.0,1.0
8,exp09b_src2_sim0.78_days7_m0.02_ev1,0.78,7.0,0.02,2.0,True,155,93,103,121,7260,152,0,35,1.0,0.812834,0.896755,0.0,1.0
9,exp09b_src2_sim0.78_days7_m0.03_ev1,0.78,7.0,0.03,2.0,True,155,93,103,121,7260,152,0,35,1.0,0.812834,0.896755,0.0,1.0


Baseline pairwise_f1: 0.7697368421052632
Best safe variant: exp09b_src2_sim0.78_days14_m0.03_ev1
Best safe pairwise_f1: 0.9032258064516129
Best safe false_merge_rate: 0.0
Has safe clustering improvement: True
Best variant attachments: (117, 14)


,source_cluster,target_cluster,best_similarity,min_days_diff,max_title_jaccard,max_shared_numbers_count,source_size,target_size,source_news_id,target_news_id,source_title,target_title,second_best_similarity,margin
64,exp09_baseline_01478,exp09_baseline_01195,0.819889,5.0,0.071429,2,1,4,90802,90491,Буш отредактирует доклад комиссии по расследов...,Кондолизза Райс даст публичные показания об об...,-inf,inf
12,exp09_baseline_00335,exp09_baseline_00116,0.819675,0.0,0.230769,0,2,5,89032,89022,Лидер шиитов считает временную конституцию Ира...,Лидер иракских шиитов согласился подписать кон...,-inf,inf
62,exp09_baseline_01382,exp09_baseline_00570,0.819279,8.0,0.083333,2,1,5,90667,90109,Главным подозреваемым по делу о терактах в Мад...,В организации терактов в Мадриде обвиняются уж...,-inf,inf
14,exp09_baseline_00417,exp09_baseline_00336,0.819260,1.0,0.333333,1,2,6,89379,89296,Прокуратура проверит все приказы Хамбиева и пр...,Хамбиев опроверг сообщения о захвате его семьи...,-inf,inf
52,exp09_baseline_01196,exp09_baseline_01246,0.819159,1.0,0.000000,3,2,6,90452,90519,Серия терактов в Узбекистане: три взрыва произ...,Версия узбекских властей: 20 террористов сами ...,-inf,inf
67,exp09_baseline_01523,exp09_baseline_00552,0.818937,14.0,0.222222,0,1,20,90871,89977,"Саакашвили предложил Аджарии ""новые правила игры""",Саакашвили пригрозил возобновить блокаду Аджарии,-inf,inf
73,exp09_baseline_01603,exp09_baseline_01583,0.818785,0.0,0.100000,4,1,5,90991,91034,У Масхадова осталось два взвода боевиков,Кадыров по телефону уговаривает Масхадова сдат...,-inf,inf
29,exp09_baseline_00687,exp09_baseline_00552,0.818728,0.0,0.100000,0,1,20,89593,89568,Национальный банк Грузии выводит деньги из Адж...,Грузия начала экономическую блокаду Аджарии,-inf,inf
1,exp09_baseline_00002,exp09_baseline_00116,0.818310,2.0,0.444444,1,1,5,88528,89067,Правительственный совет Ирака одобрил проект в...,Временная конституция Ирака подписана,-inf,inf
37,exp09_baseline_00799,exp09_baseline_00769,0.818254,1.0,0.066667,1,1,3,89771,89821,Миротворцы НАТО останавливают беспорядки в Кос...,Немецкие и французские солдаты отправятся в Ко...,-inf,inf


exp09b_best_clustered_news: (3176, 17)
clusters: 2492
Saved prediction: E:\ML\Projects\Git\news-flow-analysis\data\predictions\exp_09b_best_candidate_attach_clustering.csv
exp_09b metrics:


,reference_rows,candidate_rows,overlap_rows,coverage,cluster_eval_items,total_ref_pairs,total_pred_pairs,tp_same_pairs,fp_false_merge_pairs,fn_missed_pairs,...,false_merge_rate,novelty_eval_rows,significance_accuracy,significant_precision,significant_recall,significant_f1,significant_tp,significant_fp,significant_fn,significant_tn
0,121,3176,121,1.0,121,187,154,154,0,33,...,0.0,87,0.747126,0.84,0.863014,0.851351,63,12,10,2


,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,comment
0,exp_00b_reproduced_old_pipeline,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Переобучение CatBoost на том же feature pipeli...
6,exp_04_mlp_final_step,0.000000,1.000000,0.793103,0.839506,0.931507,0.883117,Нейросетевая замена финальной CatBoost-модели;...
7,exp_03b_catboost_retrained_fixed_old_clustering,0.769737,0.000000,0.712644,0.875000,0.767123,0.817518,"CatBoost обучен с нуля на legacy features, пос..."
8,exp_04b_mlp_retrained_fixed_old_clustering,0.769737,0.000000,0.724138,0.826667,0.849315,0.837838,"MLP обучен с нуля на legacy features, построен..."
9,exp_04c_mlp_constrained_threshold_fixed_old_cl...,0.769737,0.000000,0.586207,0.824561,0.643836,0.723077,Порог MLP выбран по validation как компромисс ...


## 11. Анализ ошибок

После всех экспериментов нужно посмотреть не только агрегированные метрики, но и конкретные ошибки.

Особенно важны:

- false negative для `significant`: модель пропустила важное обновление;
- false positive для `significant`: модель пометила повтор/малое обновление как важное;
- false merge в кластерах: разные сюжеты склеены;
- missed merge: один сюжет разбит на несколько кластеров.


In [37]:
def compare_errors(reference_df: pd.DataFrame, prediction_df: pd.DataFrame) -> pd.DataFrame:
    ref = reference_df[["news_id", "title", "topic", "cluster_id", "novelty_label"]].copy()
    pred = prediction_df[["news_id", "cluster_id", "novelty_label"]].copy()
    merged = ref.merge(pred, on="news_id", how="inner", suffixes=("_ref", "_pred"))
    merged["ref_is_significant"] = merged["novelty_label_ref"].eq("significant")
    merged["pred_is_significant"] = merged["novelty_label_pred"].eq("significant")
    merged["error_type"] = "ok"
    merged.loc[merged["ref_is_significant"] & ~merged["pred_is_significant"], "error_type"] = "FN_significant"
    merged.loc[~merged["ref_is_significant"] & merged["pred_is_significant"], "error_type"] = "FP_significant"
    return merged

# Заменить pred_04 на prediction-фрейм лучшего эксперимента.
error_df = compare_errors(golden, pred_00)
error_df["error_type"].value_counts()


error_type
ok                78
FP_significant    25
FN_significant    18
Name: count, dtype: int64

In [38]:
# Самые важные ошибки для ручного просмотра.
error_df[error_df["error_type"] != "ok"].sort_values(["error_type", "topic", "title"]).head(10)


,news_id,title,topic,cluster_id_ref,novelty_label_ref,cluster_id_pred,novelty_label_pred,ref_is_significant,pred_is_significant,error_type
100,89950,"""Бригады мученников Аль-Аксы"" обещают отомстит...",Мир,prelim_0977,significant,baseline_cluster_00908,,True,False,FN_significant
117,90802,Буш отредактирует доклад комиссии по расследов...,Мир,prelim_1143,significant,baseline_cluster_01478,,True,False,FN_significant
20,88970,Вслед за Opportunity следы воды на Марсе нашел...,Мир,prelim_0068,significant,baseline_cluster_00297,,True,False,FN_significant
55,89494,Израиль запретил хоронить Абу Аббаса в Рамалле,Мир,prelim_0471,significant,baseline_cluster_00622,,True,False,FN_significant
113,90390,Кондолизза Райс вновь отказалась дать публичны...,Мир,prelim_1143,significant,baseline_cluster_01195,,True,False,FN_significant
109,90544,Конституционный суд Литвы признал президента П...,Мир,prelim_1139,significant,baseline_cluster_01296,,True,False,FN_significant
3,88827,На родине культа вуду введено чрезвычайное пол...,Мир,prelim_0013,significant,baseline_cluster_00194,,True,False,FN_significant
61,90078,Первый американский эсминец-компонент ПРО заст...,Мир,prelim_0565,significant,baseline_cluster_00997,,True,False,FN_significant
64,89344,"""Аль-Каеда"" взяла на себя ответственность за в...",Россия,prelim_0572,significant,baseline_cluster_00544,,True,False,FN_significant
43,89194,"""Росгидромет"" ищет льдину для новой полярной с...",Россия,prelim_0312,significant,baseline_cluster_00448,,True,False,FN_significant


In [46]:
# ---------------------------------------------------------------------
# Error analysis for valid fixed-clustering experiments
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd


def normalize_news_id_for_join(series: pd.Series) -> pd.Series:
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def build_error_frame(
    model_name: str,
    pred_df: pd.DataFrame,
    golden_df: pd.DataFrame,
    feature_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    """Build row-level error table for significant / not significant prediction.

    This analysis is meaningful only for experiments with valid clustering.
    """

    ref_columns = [
        "news_id",
        "published_at",
        "topic",
        "title",
        "text",
        "cluster_id",
        "novelty_label",
    ]

    ref_columns = [column for column in ref_columns if column in golden_df.columns]

    ref = golden_df[ref_columns].copy()
    ref["news_id"] = normalize_news_id_for_join(ref["news_id"])

    ref = ref.rename(
        columns={
            "cluster_id": "cluster_id_true",
            "novelty_label": "novelty_label_true",
        }
    )

    pred_columns = [
        "news_id",
        "cluster_id",
        "novelty_label",
        "p_significant",
        "max_prev_similarity",
        "comment",
        "needs_review",
    ]

    pred_columns = [column for column in pred_columns if column in pred_df.columns]

    pred = pred_df[pred_columns].copy()
    pred["news_id"] = normalize_news_id_for_join(pred["news_id"])

    pred = pred.rename(
        columns={
            "cluster_id": "cluster_id_pred",
            "novelty_label": "novelty_label_pred",
            "comment": "comment_pred",
            "needs_review": "needs_review_pred",
        }
    )

    if "p_significant" not in pred.columns:
        pred["p_significant"] = np.nan

    if "max_prev_similarity" not in pred.columns:
        pred["max_prev_similarity"] = np.nan

    merged = ref.merge(
        pred,
        on="news_id",
        how="inner",
    )

    # Evaluation only for non-first items where golden novelty_label exists.
    eval_mask = (
        merged["novelty_label_true"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    merged = merged[eval_mask].copy()

    merged["model"] = model_name

    merged["true_is_significant"] = (
        merged["novelty_label_true"].eq("significant")
    )

    merged["pred_is_significant"] = (
        merged["novelty_label_pred"].eq("significant")
    )

    conditions = [
        merged["true_is_significant"] & merged["pred_is_significant"],
        ~merged["true_is_significant"] & ~merged["pred_is_significant"],
        ~merged["true_is_significant"] & merged["pred_is_significant"],
        merged["true_is_significant"] & ~merged["pred_is_significant"],
    ]

    choices = [
        "TP_significant",
        "TN_not_significant",
        "FP_significant",
        "FN_significant",
    ]

    merged["error_type"] = np.select(
        conditions,
        choices,
        default="unknown",
    )

    merged["is_error"] = merged["error_type"].isin([
        "FP_significant",
        "FN_significant",
    ])

    # Add legacy features if available.
    if feature_df is not None:
        feature_part = feature_df.copy()
        feature_part["news_id"] = normalize_news_id_for_join(feature_part["news_id"])

        feature_columns = [
            "news_id",
            "position_in_cluster",
            "cluster_size_so_far",
            "days_since_previous",
            "days_since_cluster_start",
            "max_prev_similarity",
            "mean_prev_similarity",
            "min_prev_similarity",
            "top2_mean_similarity",
            "top3_mean_similarity",
            "last_prev_similarity",
            "previous_centroid_similarity",
            "previous_centroid_distance",
            "title_jaccard_max",
            "text_jaccard_max",
            "shared_numbers_count",
            "new_numbers_count",
            "title_length",
            "text_length",
        ]

        feature_columns = [
            column
            for column in feature_columns
            if column in feature_part.columns
        ]

        feature_part = feature_part[feature_columns].copy()

        merged = merged.merge(
            feature_part,
            on="news_id",
            how="left",
            suffixes=("", "_feature"),
        )

    return merged


# Берём только эксперименты с валидной кластеризацией.
# exp_04_mlp_final_step сюда специально НЕ включаем:
# у него pairwise_f1=0 и false_merge_rate=1, поэтому novelty-метрики диагностические.
prediction_registry = {}

if "pred_00b" in globals():
    prediction_registry["exp_00b_reference"] = pred_00b

if "pred_04b" in globals():
    prediction_registry["exp_04b_mlp_fixed"] = pred_04b

if "pred_03b" in globals():
    prediction_registry["exp_03b_catboost_fixed"] = pred_03b

if "pred_05" in globals():
    prediction_registry["exp_05_logreg_fixed"] = pred_05

if "pred_06" in globals():
    prediction_registry["exp_06_catboost_derived_fixed"] = pred_06

if "pred_07" in globals():
    prediction_registry["exp_07_catboost_bronze_fixed"] = pred_07

feature_df_for_errors = fixed_features if "fixed_features" in globals() else None

error_frames = []

for model_name, pred_df in prediction_registry.items():
    error_frames.append(
        build_error_frame(
            model_name=model_name,
            pred_df=pred_df,
            golden_df=golden,
            feature_df=feature_df_for_errors,
        )
    )

error_analysis_df = pd.concat(error_frames, ignore_index=True)

print("Models:", list(prediction_registry.keys()))
print("Rows:", error_analysis_df.shape)

display(
    error_analysis_df[
        [
            "model",
            "error_type",
            "news_id",
            "topic",
            "published_at",
            "novelty_label_true",
            "novelty_label_pred",
            "p_significant",
            "title",
        ]
    ].head()
)

Models: ['exp_00b_reference', 'exp_04b_mlp_fixed', 'exp_03b_catboost_fixed', 'exp_05_logreg_fixed', 'exp_06_catboost_derived_fixed', 'exp_07_catboost_bronze_fixed']
Rows: (534, 36)


,model,error_type,news_id,topic,published_at,novelty_label_true,novelty_label_pred,p_significant,title
0,exp_00b_reference,FN_significant,88596,Мир,2004-03-01,significant,,NaN,Изгнанный президент Гаити прилетел в Центральн...
1,exp_00b_reference,TP_significant,88643,Мир,2004-03-02,significant,significant,0.764207,"Жан-Бертран Аристид: ""США вынудили меня бежать..."
2,exp_00b_reference,FN_significant,88827,Мир,2004-03-04,significant,,NaN,На родине культа вуду введено чрезвычайное пол...
3,exp_00b_reference,TP_significant,88851,Мир,2004-03-04,significant,significant,0.691424,За бывшего президента Гаити вступились соседни...
4,exp_00b_reference,TP_significant,88861,Россия,2004-03-05,significant,significant,0.726134,Некоторые граждане РФ жертвуют кандидатам в пр...


In [47]:
# ---------------------------------------------------------------------
# Error counts and metrics from row-level errors
# ---------------------------------------------------------------------

error_counts = (
    error_analysis_df
    .groupby(["model", "error_type"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for column in [
    "TP_significant",
    "TN_not_significant",
    "FP_significant",
    "FN_significant",
]:
    if column not in error_counts.columns:
        error_counts[column] = 0

error_counts["precision"] = (
    error_counts["TP_significant"]
    / (error_counts["TP_significant"] + error_counts["FP_significant"])
).replace([np.inf, -np.inf], np.nan).fillna(0.0)

error_counts["recall"] = (
    error_counts["TP_significant"]
    / (error_counts["TP_significant"] + error_counts["FN_significant"])
).replace([np.inf, -np.inf], np.nan).fillna(0.0)

error_counts["f1"] = (
    2 * error_counts["precision"] * error_counts["recall"]
    / (error_counts["precision"] + error_counts["recall"])
).replace([np.inf, -np.inf], np.nan).fillna(0.0)

error_counts["n_errors"] = (
    error_counts["FP_significant"]
    + error_counts["FN_significant"]
)

error_counts["n_rows"] = (
    error_counts["TP_significant"]
    + error_counts["TN_not_significant"]
    + error_counts["FP_significant"]
    + error_counts["FN_significant"]
)

error_counts = (
    error_counts
    .sort_values(["f1", "recall", "precision"], ascending=False)
    .reset_index(drop=True)
)

display(error_counts)

error_type,model,FN_significant,FP_significant,TN_not_significant,TP_significant,precision,recall,f1,n_errors,n_rows
0,exp_04b_mlp_fixed,11,13,3,62,0.826667,0.849315,0.837838,24,89
1,exp_00b_reference,12,12,4,61,0.835616,0.835616,0.835616,24,89
2,exp_03b_catboost_fixed,17,8,8,56,0.875000,0.767123,0.817518,25,89
3,exp_07_catboost_bronze_fixed,17,9,7,56,0.861538,0.767123,0.811594,26,89
4,exp_05_logreg_fixed,16,11,5,57,0.838235,0.780822,0.808511,27,89
5,exp_06_catboost_derived_fixed,17,10,6,56,0.848485,0.767123,0.805755,27,89


In [48]:
# ---------------------------------------------------------------------
# Direct comparison: exp_00b vs exp_04b
# ---------------------------------------------------------------------

base_name = "exp_00b_reference"
mlp_name = "exp_04b_mlp_fixed"

base_errors = error_analysis_df[
    error_analysis_df["model"].eq(base_name)
].copy()

mlp_errors = error_analysis_df[
    error_analysis_df["model"].eq(mlp_name)
].copy()

compare_00b_04b = base_errors.merge(
    mlp_errors[
        [
            "news_id",
            "novelty_label_pred",
            "p_significant",
            "error_type",
            "is_error",
        ]
    ].rename(
        columns={
            "novelty_label_pred": "novelty_label_pred_04b",
            "p_significant": "p_significant_04b",
            "error_type": "error_type_04b",
            "is_error": "is_error_04b",
        }
    ),
    on="news_id",
    how="inner",
)

compare_00b_04b = compare_00b_04b.rename(
    columns={
        "novelty_label_pred": "novelty_label_pred_00b",
        "p_significant": "p_significant_00b",
        "error_type": "error_type_00b",
        "is_error": "is_error_00b",
    }
)

compare_00b_04b["transition"] = (
    compare_00b_04b["error_type_00b"]
    + " -> "
    + compare_00b_04b["error_type_04b"]
)

compare_00b_04b["delta_p_significant"] = (
    compare_00b_04b["p_significant_04b"]
    - compare_00b_04b["p_significant_00b"]
)

transition_summary = (
    compare_00b_04b
    .groupby("transition")
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)

display(transition_summary)

# Ошибки, которые 04b исправил относительно 00b.
fixed_by_04b = compare_00b_04b[
    compare_00b_04b["is_error_00b"]
    & ~compare_00b_04b["is_error_04b"]
].copy()

# Новые ошибки, которые появились у 04b.
new_errors_04b = compare_00b_04b[
    ~compare_00b_04b["is_error_00b"]
    & compare_00b_04b["is_error_04b"]
].copy()

review_columns = [
    "news_id",
    "topic",
    "published_at",
    "novelty_label_true",
    "novelty_label_pred_00b",
    "novelty_label_pred_04b",
    "error_type_00b",
    "error_type_04b",
    "p_significant_00b",
    "p_significant_04b",
    "delta_p_significant",
    "max_prev_similarity",
    "days_since_previous",
    "position_in_cluster",
    "cluster_size_so_far",
    "title",
]

review_columns = [
    column
    for column in review_columns
    if column in compare_00b_04b.columns
]

print("Fixed by 04b:", len(fixed_by_04b))
display(fixed_by_04b[review_columns].sort_values("delta_p_significant", ascending=False))

print("New errors in 04b:", len(new_errors_04b))
display(new_errors_04b[review_columns].sort_values("delta_p_significant", ascending=False))

,transition,rows
5,TP_significant -> TP_significant,61
2,FP_significant -> FP_significant,12
0,FN_significant -> FN_significant,11
4,TN_not_significant -> TN_not_significant,3
1,FN_significant -> TP_significant,1
3,TN_not_significant -> FP_significant,1


Fixed by 04b: 1


,news_id,topic,published_at,novelty_label_true,novelty_label_pred_00b,novelty_label_pred_04b,error_type_00b,error_type_04b,p_significant_00b,p_significant_04b,delta_p_significant,max_prev_similarity,days_since_previous,position_in_cluster,cluster_size_so_far,title
57,89493,Россия,2004-03-15,significant,minor,significant,FN_significant,TP_significant,0.411581,0.540558,0.128977,NaN,0.0,8,8,Аслан Абашидзе заявил о готовности к переговор...


New errors in 04b: 1


,news_id,topic,published_at,novelty_label_true,novelty_label_pred_00b,novelty_label_pred_04b,error_type_00b,error_type_04b,p_significant_00b,p_significant_04b,delta_p_significant,max_prev_similarity,days_since_previous,position_in_cluster,cluster_size_so_far,title
51,89423,Россия,2004-03-14,minor,minor,significant,TN_not_significant,FP_significant,0.403847,0.516766,0.112919,NaN,0.0,4,4,На Камчатке за два часа выборов проголосовали ...


In [49]:
# ---------------------------------------------------------------------
# Error analysis by topic and feature values
# ---------------------------------------------------------------------

topic_error_summary = (
    error_analysis_df
    .groupby(["model", "topic", "error_type"])
    .size()
    .reset_index(name="rows")
    .sort_values(["model", "rows"], ascending=[True, False])
)

display(topic_error_summary.head(50))


feature_columns_for_diagnostics = [
    "position_in_cluster",
    "cluster_size_so_far",
    "days_since_previous",
    "days_since_cluster_start",
    "max_prev_similarity",
    "mean_prev_similarity",
    "last_prev_similarity",
    "previous_centroid_similarity",
    "previous_centroid_distance",
    "title_jaccard_max",
    "text_jaccard_max",
    "shared_numbers_count",
    "new_numbers_count",
    "title_length",
    "text_length",
]

feature_columns_for_diagnostics = [
    column
    for column in feature_columns_for_diagnostics
    if column in error_analysis_df.columns
]

feature_error_summary = (
    error_analysis_df
    .groupby(["model", "error_type"])[feature_columns_for_diagnostics]
    .agg(["count", "mean", "median"])
)

display(feature_error_summary)

,model,topic,error_type,rows
2,exp_00b_reference,Мир,TP_significant,26
6,exp_00b_reference,Россия,TP_significant,25
4,exp_00b_reference,Россия,FP_significant,12
9,exp_00b_reference,Экономика,TP_significant,10
0,exp_00b_reference,Мир,FN_significant,5
3,exp_00b_reference,Россия,FN_significant,4
7,exp_00b_reference,Экономика,FN_significant,3
5,exp_00b_reference,Россия,TN_not_significant,2
1,exp_00b_reference,Мир,TN_not_significant,1
8,exp_00b_reference,Экономика,TN_not_significant,1


position_in_cluster  \
                                                               count   
model                         error_type                               
exp_00b_reference             FN_significant                      12   
                              FP_significant                      12   
                              TN_not_significant                   4   
                              TP_significant                      61   
exp_03b_catboost_fixed        FN_significant                      17   
                              FP_significant                       8   
                              TN_not_significant                   8   
                              TP_significant                      56   
exp_04b_mlp_fixed             FN_significant                      11   
                              FP_significant                      13   
                              TN_not_significant                   3   
                              TP_significant                      62   
exp_05_logreg_fixed           FN_significant                      16   
                              FP_significant                      11   
                              TN_not_significant                   5   
                              TP_significant                      57   
exp_06_catboost_derived_fixed FN_significant                      17   
                              FP_significant                      10   
                              TN_not_significant                   6   
                              TP_significant                      56   
exp_07_catboost_bronze_fixed  FN_significant                      17   
                              FP_significant                       9   
                              TN_not_significant                   7   
                              TP_significant                      56   

                                                                   \
                                                      mean median   
model                         error_type                            
exp_00b_reference             FN_significant      0.666667    0.0   
                              FP_significant      3.250000    2.5   
                              TN_not_significant  1.000000    0.0   
                              TP_significant      1.819672    2.0   
exp_03b_catboost_fixed        FN_significant      0.882353    0.0   
                              FP_significant      4.000000    4.5   
                              TN_not_significant  1.375000    1.0   
                              TP_significant      1.857143    2.0   
exp_04b_mlp_fixed             FN_significant      0.000000    0.0   
                              FP_significant      3.307692    3.0   
                              TN_not_significant  0.000000    0.0   
                              TP_significant      1.919355    2.0   
exp_05_logreg_fixed           FN_significant      0.500000    0.0   
                              FP_significant      3.454545    3.0   
                              TN_not_significant  1.000000    0.0   
                              TP_significant      1.947368    2.0   
exp_06_catboost_derived_fixed FN_significant      0.823529    0.0   
                              FP_significant      3.500000    3.0   
                              TN_not_significant  1.333333    0.5   
                              TP_significant      1.875000    2.0   
exp_07_catboost_bronze_fixed  FN_significant      1.000000    0.0   
                              FP_significant      4.222222    4.0   
                              TN_not_significant  0.714286    1.0   
                              TP_significant      1.821429    1.0   

                                                 cluster_size_so_far  \
                                                               count   
model                         error_type                               
exp_00b_reference       

In [50]:
# ---------------------------------------------------------------------
# Manual review tables: false negatives and false positives
# ---------------------------------------------------------------------

def show_model_errors(
    error_df: pd.DataFrame,
    model_name: str,
    error_type: str,
    n: int = 20,
) -> pd.DataFrame:
    part = error_df[
        error_df["model"].eq(model_name)
        & error_df["error_type"].eq(error_type)
    ].copy()

    columns = [
        "news_id",
        "topic",
        "published_at",
        "novelty_label_true",
        "novelty_label_pred",
        "p_significant",
        "max_prev_similarity",
        "mean_prev_similarity",
        "last_prev_similarity",
        "days_since_previous",
        "position_in_cluster",
        "cluster_size_so_far",
        "title",
        "comment_pred",
    ]

    columns = [
        column
        for column in columns
        if column in part.columns
    ]

    return (
        part[columns]
        .sort_values(
            [
                "topic",
                "published_at",
                "p_significant",
            ],
            ascending=[True, True, False],
        )
        .head(n)
    )


for model_name in ["exp_00b_reference", "exp_04b_mlp_fixed"]:
    print(f"\n{model_name}: false negatives significant")
    display(show_model_errors(error_analysis_df, model_name, "FN_significant", n=20))

    print(f"\n{model_name}: false positives significant")
    display(show_model_errors(error_analysis_df, model_name, "FP_significant", n=20))


exp_00b_reference: false negatives significant


,news_id,topic,published_at,novelty_label_true,novelty_label_pred,p_significant,max_prev_similarity,mean_prev_similarity,last_prev_similarity,days_since_previous,position_in_cluster,cluster_size_so_far,title,comment_pred
0,88596,Мир,2004-03-01,significant,,NaN,NaN,0.000000,0.000000,-1.0,0,0,Изгнанный президент Гаити прилетел в Центральн...,first item in cluster; no fallback candidates
2,88827,Мир,2004-03-04,significant,,NaN,NaN,0.000000,0.000000,-1.0,0,0,На родине культа вуду введено чрезвычайное пол...,first item in cluster; no fallback candidates
14,88970,Мир,2004-03-06,significant,,NaN,NaN,0.000000,0.000000,-1.0,0,0,Вслед за Opportunity следы воды на Марсе нашел...,first item in cluster; no fallback candidates
43,90078,Мир,2004-03-23,significant,,NaN,NaN,0.000000,0.000000,-1.0,0,0,Первый американский эсминец-компонент ПРО заст...,first item in cluster; no fallback candidates
82,90390,Мир,2004-03-29,significant,,NaN,NaN,0.000000,0.000000,-1.0,0,0,Кондолизза Райс вновь отказалась дать публичны...,first item in cluster; no fallback candidates
32,89000,Россия,2004-03-07,significant,,NaN,NaN,0.000000,0.000000,-1.0,0,0,В жилом доме в Чертаново взорвался газовый бал...,first item in cluster; no fallback candidates
31,89194,Россия,2004-03-10,significant,,NaN,NaN,0.000000,0.000000,-1.0,0,0,"""Росгидромет"" ищет льдину для новой полярной с...",first item in cluster; no fallback candidates
45,89344,Россия,2004-03-12,significant,,NaN,NaN,0.000000,0.000000,-1.0,0,0,"""Аль-Каеда"" взяла на себя ответственность за в...",first item in cluster; no fallback candidates
57,89493,Россия,2004-03-15,significant,minor,0.411581,NaN,0.832183,0.877182,0.0,8,8,Аслан Абашидзе заявил о готовности к переговор...,in-cluster previous context; position=8; p_sig...
12,88687,Экономика,2004-03-02,significant,,NaN,NaN,0.000000,0.000000,-1.0,0,0,"Кредитный рейтинг ""ЮКОСа"" сделают еще негативнее",first item in cluster; no fallback candidates



exp_00b_reference: false positives significant


,news_id,topic,published_at,novelty_label_true,novelty_label_pred,p_significant,max_prev_similarity,mean_prev_similarity,last_prev_similarity,days_since_previous,position_in_cluster,cluster_size_so_far,title,comment_pred
6,88614,Россия,2004-03-01,duplicate,significant,0.496805,NaN,0.911870,0.911870,0.0,1,1,Наряд дагестанских пограничников еще раз уничт...,in-cluster previous context; position=1; p_sig...
29,88968,Россия,2004-03-06,duplicate,significant,0.538082,NaN,0.797408,0.795180,0.0,4,4,Ми-26 подобрал полярников с разрушенной станци...,in-cluster previous context; position=4; p_sig...
66,89467,Россия,2004-03-14,minor,significant,0.613798,NaN,0.819484,0.777739,0.0,5,5,К 12.00 в России проголосовали почти 15 процен...,in-cluster previous context; position=5; p_sig...
53,89426,Россия,2004-03-14,minor,significant,0.599502,NaN,0.708399,0.743657,0.0,5,5,На Чукотке проголосовали 22 процента. Или пять,in-cluster previous context; position=5; p_sig...
46,89417,Россия,2004-03-14,duplicate,significant,0.561110,NaN,0.877622,0.877622,2.0,1,1,"""Аль-Каеда"" вновь взяла на себя ответственност...",in-cluster previous context; position=1; p_sig...
54,89439,Россия,2004-03-14,minor,significant,0.557912,NaN,0.777372,0.593601,0.0,7,7,На Чукотке за четыре часа проголосовали 51 про...,in-cluster previous context; position=7; p_sig...
59,89435,Россия,2004-03-14,duplicate,significant,0.557709,NaN,0.860142,0.860142,0.0,1,1,В центре Москвы горит здание Манежа,in-cluster previous context; position=1; p_sig...
55,89464,Россия,2004-03-14,minor,significant,0.499921,NaN,0.752681,0.859796,0.0,8,8,"В Якутии за шесть часов проголосовали 14,65 п...",in-cluster previous context; position=8; p_sig...
63,89443,Россия,2004-03-14,duplicate,significant,0.495329,NaN,0.902464,0.897821,0.0,2,2,Выборы президента России состоялись,in-cluster previous context; position=2; p_sig...
50,89421,Россия,2004-03-14,minor,significant,0.474018,NaN,0.771905,0.764418,0.0,3,3,"Явка на Сахалине и в Приморье в 2 раза выше, ч...",in-cluster previous context; position=3; p_sig...



exp_04b_mlp_fixed: false negatives significant


,news_id,topic,published_at,novelty_label_true,novelty_label_pred,p_significant,max_prev_similarity,mean_prev_similarity,last_prev_similarity,days_since_previous,position_in_cluster,cluster_size_so_far,title,comment_pred
89,88596,Мир,2004-03-01,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,Изгнанный президент Гаити прилетел в Центральн...,first item in cluster; no fallback candidates
91,88827,Мир,2004-03-04,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,На родине культа вуду введено чрезвычайное пол...,first item in cluster; no fallback candidates
103,88970,Мир,2004-03-06,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,Вслед за Opportunity следы воды на Марсе нашел...,first item in cluster; no fallback candidates
132,90078,Мир,2004-03-23,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,Первый американский эсминец-компонент ПРО заст...,first item in cluster; no fallback candidates
171,90390,Мир,2004-03-29,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,Кондолизза Райс вновь отказалась дать публичны...,first item in cluster; no fallback candidates
121,89000,Россия,2004-03-07,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,В жилом доме в Чертаново взорвался газовый бал...,first item in cluster; no fallback candidates
120,89194,Россия,2004-03-10,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,"""Росгидромет"" ищет льдину для новой полярной с...",first item in cluster; no fallback candidates
134,89344,Россия,2004-03-12,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,"""Аль-Каеда"" взяла на себя ответственность за в...",first item in cluster; no fallback candidates
101,88687,Экономика,2004-03-02,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,"Кредитный рейтинг ""ЮКОСа"" сделают еще негативнее",first item in cluster; no fallback candidates
107,89180,Экономика,2004-03-10,significant,,NaN,NaN,0.0,0.0,-1.0,0,0,СИБУР продал Лукашенко миллиард кубометров газа,first item in cluster; no fallback candidates



exp_04b_mlp_fixed: false positives significant


,news_id,topic,published_at,novelty_label_true,novelty_label_pred,p_significant,max_prev_similarity,mean_prev_similarity,last_prev_similarity,days_since_previous,position_in_cluster,cluster_size_so_far,title,comment_pred
95,88614,Россия,2004-03-01,duplicate,significant,0.506589,NaN,0.911870,0.911870,0.0,1,1,Наряд дагестанских пограничников еще раз уничт...,in-cluster previous context; position=1; p_sig...
118,88968,Россия,2004-03-06,duplicate,significant,0.446358,NaN,0.797408,0.795180,0.0,4,4,Ми-26 подобрал полярников с разрушенной станци...,in-cluster previous context; position=4; p_sig...
144,89464,Россия,2004-03-14,minor,significant,0.655878,NaN,0.752681,0.859796,0.0,8,8,"В Якутии за шесть часов проголосовали 14,65 п...",in-cluster previous context; position=8; p_sig...
142,89426,Россия,2004-03-14,minor,significant,0.573804,NaN,0.708399,0.743657,0.0,5,5,На Чукотке проголосовали 22 процента. Или пять,in-cluster previous context; position=5; p_sig...
139,89421,Россия,2004-03-14,minor,significant,0.567402,NaN,0.771905,0.764418,0.0,3,3,"Явка на Сахалине и в Приморье в 2 раза выше, ч...",in-cluster previous context; position=3; p_sig...
143,89439,Россия,2004-03-14,minor,significant,0.539532,NaN,0.777372,0.593601,0.0,7,7,На Чукотке за четыре часа проголосовали 51 про...,in-cluster previous context; position=7; p_sig...
140,89423,Россия,2004-03-14,minor,significant,0.516766,NaN,0.794124,0.786365,0.0,4,4,На Камчатке за два часа выборов проголосовали ...,in-cluster previous context; position=4; p_sig...
155,89467,Россия,2004-03-14,minor,significant,0.516263,NaN,0.819484,0.777739,0.0,5,5,К 12.00 в России проголосовали почти 15 процен...,in-cluster previous context; position=5; p_sig...
135,89417,Россия,2004-03-14,duplicate,significant,0.482569,NaN,0.877622,0.877622,2.0,1,1,"""Аль-Каеда"" вновь взяла на себя ответственност...",in-cluster previous context; position=1; p_sig...
148,89435,Россия,2004-03-14,duplicate,significant,0.457651,NaN,0.860142,0.860142,0.0,1,1,В центре Москвы горит здание Манежа,in-cluster previous context; position=1; p_sig...


In [52]:
# ---------------------------------------------------------------------
# 11.1. Clustering error analysis: missed same-story pairs
# ---------------------------------------------------------------------
# Анализируем не novelty_label, а именно качество story clustering.
#
# Важно:
# - false merge: модель склеила разные golden-кластеры;
# - missed same-story pair: golden считает пару одним сюжетом, а модель разнесла.
#
# Для текущего conservative pipeline ожидаем:
# - false merge ≈ 0;
# - основные проблемы — missed same-story pairs.
# ---------------------------------------------------------------------

from itertools import combinations

import numpy as np
import pandas as pd


STORY_THRESHOLD = 0.82
STORY_WINDOW_DAYS = 14


def normalize_news_id_for_join(series: pd.Series) -> pd.Series:
    return (
        series
        .astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
    )


def normalize_embedding_matrix(embeddings: np.ndarray) -> np.ndarray:
    embeddings = np.asarray(embeddings, dtype=np.float32)

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)

    return embeddings / norms


def edge_blocker_reason(
    *,
    same_topic: bool,
    days_diff: float,
    similarity: float,
    story_threshold: float,
    story_window_days: int,
) -> str:
    reasons = []

    if not same_topic:
        reasons.append("different_topic")

    if pd.isna(days_diff):
        reasons.append("missing_date")
    elif days_diff > story_window_days:
        reasons.append("outside_time_window")

    if pd.isna(similarity):
        reasons.append("missing_embedding")
    elif similarity < story_threshold:
        reasons.append("below_similarity_threshold")

    if not reasons:
        return "passes_current_edge_rule"

    return " + ".join(reasons)


def build_clustering_pair_analysis(
    *,
    golden_df: pd.DataFrame,
    pred_df: pd.DataFrame,
    clustered_news_df: pd.DataFrame,
    embeddings: np.ndarray,
    story_threshold: float = 0.82,
    story_window_days: int = 14,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Build pair-level clustering analysis on golden subset."""

    reference_columns = [
        "news_id",
        "published_at",
        "topic",
        "title",
        "text",
        "cluster_id",
    ]

    reference_columns = [
        column for column in reference_columns
        if column in golden_df.columns
    ]

    reference = golden_df[reference_columns].copy()
    reference["news_id"] = normalize_news_id_for_join(reference["news_id"])
    reference["cluster_id"] = reference["cluster_id"].astype(str)

    reference = reference.rename(
        columns={
            "cluster_id": "cluster_id_true",
        }
    )

    prediction = pred_df[["news_id", "cluster_id"]].copy()
    prediction["news_id"] = normalize_news_id_for_join(prediction["news_id"])
    prediction["cluster_id"] = prediction["cluster_id"].astype(str)

    prediction = prediction.rename(
        columns={
            "cluster_id": "cluster_id_pred",
        }
    )

    merged = reference.merge(
        prediction,
        on="news_id",
        how="inner",
    )

    merged["published_at"] = pd.to_datetime(
        merged["published_at"],
        errors="coerce",
    )

    merged = merged.reset_index(drop=True)

    # embeddings aligned to clustered_news_df
    clustered_index = clustered_news_df[["news_id"]].copy()
    clustered_index["news_id"] = normalize_news_id_for_join(clustered_index["news_id"])

    news_id_to_embedding_index = {
        news_id: idx
        for idx, news_id in enumerate(clustered_index["news_id"])
    }

    embeddings_norm = normalize_embedding_matrix(embeddings)

    rows = []

    for i, j in combinations(range(len(merged)), 2):
        left = merged.iloc[i]
        right = merged.iloc[j]

        true_same = left["cluster_id_true"] == right["cluster_id_true"]
        pred_same = left["cluster_id_pred"] == right["cluster_id_pred"]

        left_embedding_index = news_id_to_embedding_index.get(left["news_id"])
        right_embedding_index = news_id_to_embedding_index.get(right["news_id"])

        if left_embedding_index is not None and right_embedding_index is not None:
            similarity = float(
                embeddings_norm[left_embedding_index]
                @ embeddings_norm[right_embedding_index]
            )
        else:
            similarity = np.nan

        if pd.notna(left["published_at"]) and pd.notna(right["published_at"]):
            days_diff = abs(
                (right["published_at"] - left["published_at"]).total_seconds()
                / 86_400
            )
        else:
            days_diff = np.nan

        same_topic = str(left.get("topic", "")) == str(right.get("topic", ""))

        if true_same and pred_same:
            pair_type = "hit_same_story"
        elif true_same and not pred_same:
            pair_type = "missed_same_story"
        elif not true_same and pred_same:
            pair_type = "false_merge"
        else:
            pair_type = "true_negative"

        blocker_reason = edge_blocker_reason(
            same_topic=same_topic,
            days_diff=days_diff,
            similarity=similarity,
            story_threshold=story_threshold,
            story_window_days=story_window_days,
        )

        rows.append({
            "news_id_left": left["news_id"],
            "news_id_right": right["news_id"],
            "topic_left": left.get("topic", ""),
            "topic_right": right.get("topic", ""),
            "topic_pair": (
                str(left.get("topic", ""))
                if same_topic
                else f"{left.get('topic', '')} / {right.get('topic', '')}"
            ),
            "published_at_left": left["published_at"],
            "published_at_right": right["published_at"],
            "days_diff": days_diff,
            "title_left": left.get("title", ""),
            "title_right": right.get("title", ""),
            "cluster_id_true_left": left["cluster_id_true"],
            "cluster_id_true_right": right["cluster_id_true"],
            "cluster_id_pred_left": left["cluster_id_pred"],
            "cluster_id_pred_right": right["cluster_id_pred"],
            "true_same": true_same,
            "pred_same": pred_same,
            "pair_type": pair_type,
            "same_topic": same_topic,
            "similarity": similarity,
            "edge_blocker_reason": blocker_reason,
            "passes_current_edge_rule": blocker_reason == "passes_current_edge_rule",
        })

    pair_analysis = pd.DataFrame(rows)

    return merged, pair_analysis


# Берём текущую reference/final модель.
# Если pred_00b есть в памяти — используем её.
# Иначе можно подгрузить prediction CSV из artifacts/predictions.
if "pred_00b" in globals():
    reference_pred = pred_00b.copy()
else:
    reference_pred_path = paths.predictions_dir / "exp_00b_reproduced_old_pipeline.csv"
    reference_pred = pd.read_csv(reference_pred_path)

golden_cluster_rows, clustering_pair_analysis = build_clustering_pair_analysis(
    golden_df=golden,
    pred_df=reference_pred,
    clustered_news_df=old_pipeline_clustered_news,
    embeddings=old_pipeline_embeddings,
    story_threshold=STORY_THRESHOLD,
    story_window_days=STORY_WINDOW_DAYS,
)

print("Golden rows in clustering analysis:", golden_cluster_rows.shape)
print("Pair analysis rows:", clustering_pair_analysis.shape)

pair_type_counts = (
    clustering_pair_analysis["pair_type"]
    .value_counts()
    .rename_axis("pair_type")
    .reset_index(name="pairs")
)

display(pair_type_counts)

Golden rows in clustering analysis: (121, 7)
Pair analysis rows: (7260, 21)


,pair_type,pairs
0,true_negative,7073
1,hit_same_story,117
2,missed_same_story,70


In [53]:
# ---------------------------------------------------------------------
# 11.2. Clustering misses summary
# ---------------------------------------------------------------------

tp_same_pairs = int((clustering_pair_analysis["pair_type"] == "hit_same_story").sum())
fn_missed_same_pairs = int((clustering_pair_analysis["pair_type"] == "missed_same_story").sum())
fp_false_merge_pairs = int((clustering_pair_analysis["pair_type"] == "false_merge").sum())

pairwise_precision = (
    tp_same_pairs / (tp_same_pairs + fp_false_merge_pairs)
    if (tp_same_pairs + fp_false_merge_pairs) > 0
    else 0.0
)

pairwise_recall = (
    tp_same_pairs / (tp_same_pairs + fn_missed_same_pairs)
    if (tp_same_pairs + fn_missed_same_pairs) > 0
    else 0.0
)

pairwise_f1 = (
    2 * pairwise_precision * pairwise_recall / (pairwise_precision + pairwise_recall)
    if (pairwise_precision + pairwise_recall) > 0
    else 0.0
)

false_merge_rate = (
    fp_false_merge_pairs / (tp_same_pairs + fp_false_merge_pairs)
    if (tp_same_pairs + fp_false_merge_pairs) > 0
    else 0.0
)

clustering_summary = pd.DataFrame([{
    "tp_same_pairs": tp_same_pairs,
    "fn_missed_same_pairs": fn_missed_same_pairs,
    "fp_false_merge_pairs": fp_false_merge_pairs,
    "pairwise_precision": pairwise_precision,
    "pairwise_recall": pairwise_recall,
    "pairwise_f1": pairwise_f1,
    "false_merge_rate": false_merge_rate,
}])

display(clustering_summary)


missed_pairs = clustering_pair_analysis[
    clustering_pair_analysis["pair_type"].eq("missed_same_story")
].copy()

false_merge_pairs = clustering_pair_analysis[
    clustering_pair_analysis["pair_type"].eq("false_merge")
].copy()

print("Missed same-story pairs:", len(missed_pairs))
print("False merge pairs:", len(false_merge_pairs))

print("\nMissed pair blocker reasons:")
display(
    missed_pairs["edge_blocker_reason"]
    .value_counts()
    .rename_axis("edge_blocker_reason")
    .reset_index(name="pairs")
)

print("\nMissed pairs by topic:")
display(
    missed_pairs
    .groupby("topic_pair")
    .agg(
        missed_pairs=("pair_type", "size"),
        true_clusters=("cluster_id_true_left", "nunique"),
        median_similarity=("similarity", "median"),
        max_similarity=("similarity", "max"),
        median_days_diff=("days_diff", "median"),
        max_days_diff=("days_diff", "max"),
    )
    .sort_values("missed_pairs", ascending=False)
    .reset_index()
)

,tp_same_pairs,fn_missed_same_pairs,fp_false_merge_pairs,pairwise_precision,pairwise_recall,pairwise_f1,false_merge_rate
0,117,70,0,1.0,0.625668,0.769737,0.0


Missed same-story pairs: 70
False merge pairs: 0

Missed pair blocker reasons:


,edge_blocker_reason,pairs
0,below_similarity_threshold,67
1,outside_time_window + below_similarity_threshold,3



Missed pairs by topic:


,topic_pair,missed_pairs,true_clusters,median_similarity,max_similarity,median_days_diff,max_days_diff
0,Мир,30,7,0.702511,0.819889,4.0,24.0
1,Россия,27,7,0.756326,0.810833,1.0,4.0
2,Экономика,13,4,0.740398,0.814988,7.0,19.0


In [54]:
# ---------------------------------------------------------------------
# 11.3. Golden cluster fragmentation analysis
# ---------------------------------------------------------------------

true_cluster_base = (
    golden_cluster_rows
    .groupby("cluster_id_true")
    .agg(
        n_items=("news_id", "size"),
        n_pred_clusters=("cluster_id_pred", "nunique"),
        topic=("topic", lambda values: " / ".join(sorted(set(map(str, values))))),
        first_date=("published_at", "min"),
        last_date=("published_at", "max"),
        titles=("title", lambda values: " | ".join(list(map(str, values))[:3])),
        pred_clusters=("cluster_id_pred", lambda values: " | ".join(sorted(set(map(str, values))))),
    )
    .reset_index()
)

missed_by_true_cluster = (
    missed_pairs
    .groupby("cluster_id_true_left")
    .agg(
        missed_pairs=("pair_type", "size"),
        median_similarity=("similarity", "median"),
        max_similarity=("similarity", "max"),
        median_days_diff=("days_diff", "median"),
        max_days_diff=("days_diff", "max"),
    )
    .reset_index()
    .rename(columns={"cluster_id_true_left": "cluster_id_true"})
)

true_cluster_fragmentation = true_cluster_base.merge(
    missed_by_true_cluster,
    on="cluster_id_true",
    how="left",
)

true_cluster_fragmentation["missed_pairs"] = (
    true_cluster_fragmentation["missed_pairs"]
    .fillna(0)
    .astype(int)
)

true_cluster_fragmentation["true_pairs"] = (
    true_cluster_fragmentation["n_items"]
    * (true_cluster_fragmentation["n_items"] - 1)
    // 2
)

true_cluster_fragmentation["pair_recall_inside_cluster"] = (
    (true_cluster_fragmentation["true_pairs"] - true_cluster_fragmentation["missed_pairs"])
    / true_cluster_fragmentation["true_pairs"].replace(0, np.nan)
)

true_cluster_fragmentation = (
    true_cluster_fragmentation
    .sort_values(
        ["missed_pairs", "n_pred_clusters", "n_items"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

display(true_cluster_fragmentation)

,cluster_id_true,n_items,n_pred_clusters,topic,first_date,last_date,titles,pred_clusters,missed_pairs,median_similarity,max_similarity,median_days_diff,max_days_diff,true_pairs,pair_recall_inside_cluster
0,prelim_1143,6,4,Мир,2004-03-25,2004-04-04,Ричард Кларк: в терактах 11 сентября виноват Б...,baseline_cluster_01061 | baseline_cluster_0119...,12,0.656215,0.819889,5.0,10.0,15,0.200000
1,prelim_0612,9,2,Россия,2004-03-13,2004-03-14,Выборы президента Российской Федерации началис...,baseline_cluster_00573 | baseline_cluster_00592,8,0.745656,0.801734,0.0,1.0,36,0.777778
2,prelim_0013,5,3,Мир,2004-03-01,2004-03-04,На Гаити высадились французские миротворцы | И...,baseline_cluster_00028 | baseline_cluster_0003...,7,0.681393,0.757542,2.0,3.0,10,0.300000
3,prelim_0051,4,3,Экономика,2004-03-01,2004-03-18,Саудовская Аравия пообещала не допустить нехва...,baseline_cluster_00011 | baseline_cluster_0051...,5,0.732352,0.808121,8.0,17.0,6,0.166667
4,prelim_0112,4,3,Экономика,2004-03-02,2004-03-10,"""Газпром"": Минск саботирует подписание газовог...",baseline_cluster_00088 | baseline_cluster_0012...,5,0.749804,0.780418,5.0,8.0,6,0.166667
5,prelim_0572,4,3,Россия,2004-03-12,2004-03-15,"""Аль-Каеда"" обещает Америке ""Ветры черной смер...",baseline_cluster_00536 | baseline_cluster_0054...,5,0.751969,0.791812,2.0,3.0,6,0.166667
6,prelim_0312,5,2,Россия,2004-03-06,2004-03-10,Ми-26 отправился спасать российских полярников...,baseline_cluster_00193 | baseline_cluster_00448,4,0.663153,0.733911,4.0,4.0,10,0.600000
7,prelim_0044,4,2,Россия,2004-03-01,2004-03-03,"Заместитель генпрокурора России подтвердил, чт...",baseline_cluster_00041 | baseline_cluster_00119,3,0.783666,0.803043,2.0,2.0,6,0.500000
8,prelim_0624,4,2,Россия,2004-03-14,2004-03-15,МВД Аджарии: На нас идет грузинская колонна | ...,baseline_cluster_00552 | baseline_cluster_00640,3,0.809316,0.810833,0.0,1.0,6,0.500000
9,prelim_1139,4,2,Мир,2004-03-25,2004-04-07,Российский спонсор президента Литвы помещен по...,baseline_cluster_01056 | baseline_cluster_01296,3,0.750301,0.781795,11.0,13.0,6,0.500000


In [55]:
# ---------------------------------------------------------------------
# 11.4. Missed pair distributions: similarity and time
# ---------------------------------------------------------------------

missed_pairs = missed_pairs.copy()

missed_pairs["similarity_bin"] = pd.cut(
    missed_pairs["similarity"],
    bins=[-np.inf, 0.70, 0.75, 0.78, 0.80, 0.82, 0.84, 0.86, 0.90, np.inf],
    labels=[
        "<0.70",
        "0.70–0.75",
        "0.75–0.78",
        "0.78–0.80",
        "0.80–0.82",
        "0.82–0.84",
        "0.84–0.86",
        "0.86–0.90",
        ">=0.90",
    ],
)

missed_pairs["days_diff_bin"] = pd.cut(
    missed_pairs["days_diff"],
    bins=[-np.inf, 1, 3, 7, 14, 30, 90, np.inf],
    labels=[
        "<=1d",
        "1–3d",
        "3–7d",
        "7–14d",
        "14–30d",
        "30–90d",
        ">90d",
    ],
)

print("Missed pairs by similarity bin:")
display(
    missed_pairs
    .groupby("similarity_bin", observed=False)
    .size()
    .reset_index(name="missed_pairs")
)

print("Missed pairs by days_diff bin:")
display(
    missed_pairs
    .groupby("days_diff_bin", observed=False)
    .size()
    .reset_index(name="missed_pairs")
)

print("Missed pairs by topic and blocker reason:")
display(
    missed_pairs
    .groupby(["topic_pair", "edge_blocker_reason"])
    .size()
    .reset_index(name="missed_pairs")
    .sort_values(["missed_pairs"], ascending=False)
)

Missed pairs by similarity bin:


,similarity_bin,missed_pairs
0,<0.70,22
1,0.70–0.75,18
2,0.75–0.78,11
3,0.78–0.80,10
4,0.80–0.82,9
5,0.82–0.84,0
6,0.84–0.86,0
7,0.86–0.90,0
8,>=0.90,0


Missed pairs by days_diff bin:


,days_diff_bin,missed_pairs
0,<=1d,29
1,1–3d,12
2,3–7d,18
3,7–14d,8
4,14–30d,3
5,30–90d,0
6,>90d,0


Missed pairs by topic and blocker reason:


,topic_pair,edge_blocker_reason,missed_pairs
0,Мир,below_similarity_threshold,29
2,Россия,below_similarity_threshold,27
3,Экономика,below_similarity_threshold,11
4,Экономика,outside_time_window + below_similarity_threshold,2
1,Мир,outside_time_window + below_similarity_threshold,1


In [56]:
# ---------------------------------------------------------------------
# 11.5. Manual review: near-miss same-story pairs
# ---------------------------------------------------------------------

review_columns = [
    "topic_pair",
    "cluster_id_true_left",
    "news_id_left",
    "news_id_right",
    "published_at_left",
    "published_at_right",
    "days_diff",
    "similarity",
    "edge_blocker_reason",
    "cluster_id_pred_left",
    "cluster_id_pred_right",
    "title_left",
    "title_right",
]

print("Highest-similarity missed same-story pairs:")
display(
    missed_pairs[review_columns]
    .sort_values(["similarity", "days_diff"], ascending=[False, True])
    .head(30)
)

print("Missed pairs closest to current threshold:")
display(
    missed_pairs.assign(
        distance_to_threshold=(STORY_THRESHOLD - missed_pairs["similarity"]).abs()
    )[review_columns + ["distance_to_threshold"]]
    .sort_values(["distance_to_threshold", "days_diff"], ascending=[True, True])
    .head(30)
)

print("Missed pairs that would pass current edge rule, if any:")
display(
    missed_pairs[
        missed_pairs["passes_current_edge_rule"]
    ][review_columns]
    .sort_values(["similarity"], ascending=False)
)

Highest-similarity missed same-story pairs:


,topic_pair,cluster_id_true_left,news_id_left,news_id_right,published_at_left,published_at_right,days_diff,similarity,edge_blocker_reason,cluster_id_pred_left,cluster_id_pred_right,title_left,title_right
7250,Мир,prelim_1143,90491,90802,2004-03-30,2004-04-04,5.0,0.819889,below_similarity_threshold,baseline_cluster_01195,baseline_cluster_01478,Кондолизза Райс даст публичные показания об об...,Буш отредактирует доклад комиссии по расследов...
6960,Экономика,prelim_0924,90363,91009,2004-03-28,2004-04-07,10.0,0.814988,below_similarity_threshold,baseline_cluster_00827,baseline_cluster_01615,"""ЮКОСу"" не дали сменить руководство ""Сибнефти""","""ЮКОС"" заблокировал поправки в устав ""Сибнефти"""
6315,Россия,prelim_0624,89473,89521,2004-03-15,2004-03-15,0.0,0.810833,below_similarity_threshold,baseline_cluster_00552,baseline_cluster_00640,Колонна грузинской бронетехники находится в 30...,"Грузия не собирается вводить в Аджарию танки, ..."
6272,Россия,prelim_0624,89413,89521,2004-03-14,2004-03-15,1.0,0.809316,below_similarity_threshold,baseline_cluster_00552,baseline_cluster_00640,МВД Аджарии: На нас идет грузинская колонна,"Грузия не собирается вводить в Аджарию танки, ..."
1589,Экономика,prelim_0051,89300,89751,2004-03-11,2004-03-18,7.0,0.808121,below_similarity_threshold,baseline_cluster_00518,baseline_cluster_00783,ОПЕК временно отменила нефтяные квоты,"ОПЕК обеспокоена ростом цен на нефть, но снизи..."
1155,Россия,prelim_0044,88662,88715,2004-03-02,2004-03-03,1.0,0.803043,below_similarity_threshold,baseline_cluster_00041,baseline_cluster_00119,ФСБ опознала Гелаева по особому кинжалу,Уничтожившие Гелаева пограничники представлены...
4983,Мир,prelim_0471,89219,89494,2004-03-10,2004-03-15,5.0,0.802065,below_similarity_threshold,baseline_cluster_00429,baseline_cluster_00622,Палестинцы обвиняют США в убийстве Абу Аббаса,Израиль запретил хоронить Абу Аббаса в Рамалле
6035,Россия,prelim_0612,89423,89425,2004-03-14,2004-03-14,0.0,0.801734,below_similarity_threshold,baseline_cluster_00573,baseline_cluster_00592,На Камчатке за два часа выборов проголосовали ...,Выборы начались в Западной Сибири и на Урале
6085,Россия,prelim_0612,89425,89439,2004-03-14,2004-03-14,0.0,0.801357,below_similarity_threshold,baseline_cluster_00592,baseline_cluster_00573,Выборы начались в Западной Сибири и на Урале,На Чукотке за четыре часа проголосовали 51 про...
5665,Россия,prelim_0572,89344,89505,2004-03-12,2004-03-15,3.0,0.791812,below_similarity_threshold,baseline_cluster_00544,baseline_cluster_00628,"""Аль-Каеда"" взяла на себя ответственность за в...",Подозреваемые во взрывах поездов в Мадриде свя...


Missed pairs closest to current threshold:


,topic_pair,cluster_id_true_left,news_id_left,news_id_right,published_at_left,published_at_right,days_diff,similarity,edge_blocker_reason,cluster_id_pred_left,cluster_id_pred_right,title_left,title_right,distance_to_threshold
7250,Мир,prelim_1143,90491,90802,2004-03-30,2004-04-04,5.0,0.819889,below_similarity_threshold,baseline_cluster_01195,baseline_cluster_01478,Кондолизза Райс даст публичные показания об об...,Буш отредактирует доклад комиссии по расследов...,0.000111
6960,Экономика,prelim_0924,90363,91009,2004-03-28,2004-04-07,10.0,0.814988,below_similarity_threshold,baseline_cluster_00827,baseline_cluster_01615,"""ЮКОСу"" не дали сменить руководство ""Сибнефти""","""ЮКОС"" заблокировал поправки в устав ""Сибнефти""",0.005012
6315,Россия,prelim_0624,89473,89521,2004-03-15,2004-03-15,0.0,0.810833,below_similarity_threshold,baseline_cluster_00552,baseline_cluster_00640,Колонна грузинской бронетехники находится в 30...,"Грузия не собирается вводить в Аджарию танки, ...",0.009167
6272,Россия,prelim_0624,89413,89521,2004-03-14,2004-03-15,1.0,0.809316,below_similarity_threshold,baseline_cluster_00552,baseline_cluster_00640,МВД Аджарии: На нас идет грузинская колонна,"Грузия не собирается вводить в Аджарию танки, ...",0.010684
1589,Экономика,prelim_0051,89300,89751,2004-03-11,2004-03-18,7.0,0.808121,below_similarity_threshold,baseline_cluster_00518,baseline_cluster_00783,ОПЕК временно отменила нефтяные квоты,"ОПЕК обеспокоена ростом цен на нефть, но снизи...",0.011879
1155,Россия,prelim_0044,88662,88715,2004-03-02,2004-03-03,1.0,0.803043,below_similarity_threshold,baseline_cluster_00041,baseline_cluster_00119,ФСБ опознала Гелаева по особому кинжалу,Уничтожившие Гелаева пограничники представлены...,0.016957
4983,Мир,prelim_0471,89219,89494,2004-03-10,2004-03-15,5.0,0.802065,below_similarity_threshold,baseline_cluster_00429,baseline_cluster_00622,Палестинцы обвиняют США в убийстве Абу Аббаса,Израиль запретил хоронить Абу Аббаса в Рамалле,0.017935
6035,Россия,prelim_0612,89423,89425,2004-03-14,2004-03-14,0.0,0.801734,below_similarity_threshold,baseline_cluster_00573,baseline_cluster_00592,На Камчатке за два часа выборов проголосовали ...,Выборы начались в Западной Сибири и на Урале,0.018266
6085,Россия,prelim_0612,89425,89439,2004-03-14,2004-03-14,0.0,0.801357,below_similarity_threshold,baseline_cluster_00592,baseline_cluster_00573,Выборы начались в Западной Сибири и на Урале,На Чукотке за четыре часа проголосовали 51 про...,0.018643
5665,Россия,prelim_0572,89344,89505,2004-03-12,2004-03-15,3.0,0.791812,below_similarity_threshold,baseline_cluster_00544,baseline_cluster_00628,"""Аль-Каеда"" взяла на себя ответственность за в...",Подозреваемые во взрывах поездов в Мадриде свя...,0.028188


Missed pairs that would pass current edge rule, if any:


,topic_pair,cluster_id_true_left,news_id_left,news_id_right,published_at_left,published_at_right,days_diff,similarity,edge_blocker_reason,cluster_id_pred_left,cluster_id_pred_right,title_left,title_right


## 12. Итоговое сравнение экспериментов

Финальный вывод должен быть не «мы попробовали много всего», а нормальная инженерная сводка:

1. какая точка отсчёта была;
2. какие изменения дали прирост;
3. какие изменения не дали прироста;
4. какой вариант выбран финальным;
5. какие ограничения остаются.


In [39]:
final_results = compact_metrics_table(results_table).copy()
# ---------------------------------------------------------------------
# Final results with clustering guardrail
# ---------------------------------------------------------------------

import numpy as np
import pandas as pd

final_results = compact_metrics_table(results_table).copy()

# Кластеризация — обязательный guardrail для end-to-end pipeline.
# Если она сломана, significant_f1 считаем только диагностической метрикой.
MAX_FALSE_MERGE_RATE_FOR_SELECTION = 1e-9
MIN_PAIRWISE_F1_FOR_SELECTION = 0.50

final_results["clustering_status"] = np.where(
    (
        (final_results["false_merge_rate"] <= MAX_FALSE_MERGE_RATE_FOR_SELECTION)
        & (final_results["pairwise_f1"] >= MIN_PAIRWISE_F1_FOR_SELECTION)
    ),
    "valid",
    "failed_or_not_eligible",
)

final_results["selection_note"] = np.where(
    final_results["clustering_status"].eq("valid"),
    "Can be compared by novelty metrics",
    (
        "Not eligible for final selection: clustering failed or false_merge_rate is too high; "
        "significant metrics are diagnostic only"
    ),
)

# Отдельно пометим финально выбранную модель и сомнительный MLP-прирост.
final_results.loc[
    final_results["experiment"].eq("exp_00b_reproduced_old_pipeline"),
    "selection_note",
] = (
    "Selected final/current-best pipeline: valid clustering, false_merge_rate=0, "
    "balanced significant precision/recall/f1"
)

final_results.loc[
    final_results["experiment"].eq("exp_04b_mlp_retrained_fixed_old_clustering"),
    "selection_note",
] = (
    "Formally slightly higher significant_f1 than reference, "
    "but gain is minimal and not considered robust"
)

# Сначала выводим таблицу, где на первом месте состояние кластеризации.
clustering_first_columns = [
    "experiment",
    "clustering_status",
    "pairwise_f1",
    "false_merge_rate",
    "significance_accuracy",
    "significant_precision",
    "significant_recall",
    "significant_f1",
    "selection_note",
    "comment",
]

clustering_first_columns = [
    column for column in clustering_first_columns
    if column in final_results.columns
]

display(final_results[clustering_first_columns])


,experiment,clustering_status,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,selection_note,comment
0,exp_00b_reproduced_old_pipeline,valid,0.769737,0.000000,0.724138,0.835616,0.835616,0.835616,Selected final/current-best pipeline: valid cl...,Полностью воспроизводимый pipeline: legacy inp...
1,exp_01_embedding_projection,failed_or_not_eligible,0.052427,0.971169,0.839080,0.839080,1.000000,0.912500,Not eligible for final selection: clustering f...,"Проверка, улучшает ли contrastive projection г..."
2,exp_02a_mutual_knn_clustering,failed_or_not_eligible,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Not eligible for final selection: clustering f...,Снижает chaining-effect: связь засчитывается т...
3,exp_02b_temporal_decay_clustering,failed_or_not_eligible,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Not eligible for final selection: clustering f...,Похожесть штрафуется за временную дистанцию
4,exp_02c_online_lifecycle_clustering,failed_or_not_eligible,0.000000,1.000000,0.839080,0.839080,1.000000,0.912500,Not eligible for final selection: clustering f...,Старые кластеры перестают принимать новые ново...
5,exp_03_expanded_features_catboost,failed_or_not_eligible,0.000000,1.000000,0.827586,0.881579,0.917808,0.899329,Not eligible for final selection: clustering f...,Переобучение CatBoost на том же feature pipeli...
6,exp_04_mlp_final_step,failed_or_not_eligible,0.000000,1.000000,0.793103,0.839506,0.931507,0.883117,Not eligible for final selection: clustering f...,Нейросетевая замена финальной CatBoost-модели;...
7,exp_03b_catboost_retrained_fixed_old_clustering,valid,0.769737,0.000000,0.712644,0.875000,0.767123,0.817518,Can be compared by novelty metrics,"CatBoost обучен с нуля на legacy features, пос..."
8,exp_04b_mlp_retrained_fixed_old_clustering,valid,0.769737,0.000000,0.724138,0.826667,0.849315,0.837838,Formally slightly higher significant_f1 than r...,"MLP обучен с нуля на legacy features, построен..."
9,exp_04c_mlp_constrained_threshold_fixed_old_cl...,valid,0.769737,0.000000,0.586207,0.824561,0.643836,0.723077,Can be compared by novelty metrics,Порог MLP выбран по validation как компромисс ...


In [40]:
# ---------------------------------------------------------------------
# Experiments eligible for novelty-model comparison
# ---------------------------------------------------------------------

valid_clustering_results = final_results[
    final_results["clustering_status"].eq("valid")
].copy()

valid_clustering_results = (
    valid_clustering_results
    .sort_values(
        ["significant_f1", "significant_recall", "significant_precision"],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

valid_columns = [
    "experiment",
    "pairwise_f1",
    "false_merge_rate",
    "significance_accuracy",
    "significant_precision",
    "significant_recall",
    "significant_f1",
    "selection_note",
    "comment",
]

valid_columns = [
    column for column in valid_columns
    if column in valid_clustering_results.columns
]

display(valid_clustering_results[valid_columns])

,experiment,pairwise_f1,false_merge_rate,significance_accuracy,significant_precision,significant_recall,significant_f1,selection_note,comment
0,exp_04b_mlp_retrained_fixed_old_clustering,0.769737,0.0,0.724138,0.826667,0.849315,0.837838,Formally slightly higher significant_f1 than r...,"MLP обучен с нуля на legacy features, построен..."
1,exp_00b_reproduced_old_pipeline,0.769737,0.0,0.724138,0.835616,0.835616,0.835616,Selected final/current-best pipeline: valid cl...,Полностью воспроизводимый pipeline: legacy inp...
2,exp_03b_catboost_retrained_fixed_old_clustering,0.769737,0.0,0.712644,0.875000,0.767123,0.817518,Can be compared by novelty metrics,"CatBoost обучен с нуля на legacy features, пос..."
3,exp_07_catboost_silver_bronze_weighted_fixed_o...,0.769737,0.0,0.701149,0.861538,0.767123,0.811594,Can be compared by novelty metrics,CatBoost trained on silver plus bronze weak la...
4,exp_05_logreg_fixed_old_clustering,0.769737,0.0,0.689655,0.838235,0.780822,0.808511,Can be compared by novelty metrics,Линейная модель на legacy features; threshold ...
5,exp_06_catboost_derived_features_fixed_old_clu...,0.769737,0.0,0.689655,0.848485,0.767123,0.805755,Can be compared by novelty metrics,CatBoost на legacy + derived novelty features;...
6,exp_04c_mlp_constrained_threshold_fixed_old_cl...,0.769737,0.0,0.586207,0.824561,0.643836,0.723077,Can be compared by novelty metrics,Порог MLP выбран по validation как компромисс ...


Эксперименты с `clustering_status = failed_or_not_eligible` не используются для выбора финальной end-to-end модели. У них может быть высокий `significant_f1`, но эта метрика становится диагностической, если предварительно сломана кластеризация: новости сравниваются с неправильным контекстом внутри ошибочных кластеров. Поэтому сначала применяется guardrail по кластеризации (`false_merge_rate ≈ 0`, приемлемый `pairwise_f1`), и только затем сравниваются novelty-метрики.

In [41]:
# Сохраняем таблицу экспериментов для отчёта.
results_path = paths.artifacts_dir / "model_improvement_results.csv"
results_table.to_csv(results_path, index=False)
results_path


WindowsPath('E:/ML/Projects/Git/news-flow-analysis/data/artifacts/model_improvement_results.csv')

## 13. Предварительный шаблон вывода

Ниже текст-заготовка, которую нужно будет обновить после фактического запуска экспериментов.


### Выводы

1. В этом ноутбуке текущая CatBoost-модель была вынесена из исследовательского кода в отдельный модуль и использована как воспроизводимая точка отсчёта.
2. Для воспроизведения предыдущего пайплайна кластеризация и prediction выполняются на полном `lenta_golden_candidate_pool.csv`, а не только на human golden subset. Golden-разметка используется только для evaluation по пересечению `news_id`.
3. Для всех экспериментов применялся единый evaluation protocol: модель возвращала prediction CSV в той же схеме, что и golden-разметка, после чего считались cluster-level и novelty-level метрики.
4. Эксперименты с семантической векторизацией проверяли, улучшает ли contrastive projection/adaptor геометрию BGE-M3 embeddings для новостных сюжетов.
5. Эксперименты с кластеризацией проверяли альтернативы простому threshold graph: mutual-kNN, temporal decay и lifecycle clustering. Эти варианты направлены на снижение false merge rate и chaining-effect.
6. Эксперименты финального шага проверяли переобучение CatBoost и нейросетевую MLP-модель как альтернативу финальной CatBoost-модели.
7. Финальная модель должна выбираться не только по `significant_f1`, но и с учётом `false_merge_rate`: для новостных сюжетов ложные склейки особенно опасны, так как ухудшают и кластеризацию, и последующую оценку novelty.


In [42]:
# ---------------------------------------------------------------------
# Diagnostics: train/validation behavior
# ---------------------------------------------------------------------

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)


def binary_diagnostics(
    name: str,
    model,
    X_train,
    y_train,
    X_val,
    y_val,
    threshold: float,
):
    train_proba = model.predict_proba(X_train)[:, 1]
    val_proba = model.predict_proba(X_val)[:, 1]

    train_pred = (train_proba >= threshold).astype(int)
    val_pred = (val_proba >= threshold).astype(int)

    rows = []

    for split_name, y_true, y_pred, y_proba in [
        ("train", y_train, train_pred, train_proba),
        ("validation", y_val, val_pred, val_proba),
    ]:
        rows.append(
            {
                "model": name,
                "split": split_name,
                "threshold": threshold,
                "accuracy": accuracy_score(y_true, y_pred),
                "precision": precision_score(y_true, y_pred, zero_division=0),
                "recall": recall_score(y_true, y_pred, zero_division=0),
                "f1": f1_score(y_true, y_pred, zero_division=0),
                "roc_auc": roc_auc_score(y_true, y_proba),
                "average_precision": average_precision_score(y_true, y_proba),
                "positive_rate_pred": y_pred.mean(),
                "positive_rate_true": y_true.mean(),
                "proba_mean": y_proba.mean(),
                "proba_min": y_proba.min(),
                "proba_max": y_proba.max(),
            }
        )

    return pd.DataFrame(rows)


diagnostics_03b = binary_diagnostics(
    name="exp_03b_catboost",
    model=catboost_candidate,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    threshold=best_threshold_03b,
)

diagnostics_04b = binary_diagnostics(
    name="exp_04b_mlp",
    model=mlp_candidate,
    X_train=X_train,
    y_train=y_train,
    X_val=X_val,
    y_val=y_val,
    threshold=best_threshold_04b,
)

diagnostics_table = pd.concat(
    [diagnostics_03b, diagnostics_04b],
    ignore_index=True,
)

diagnostics_table

,model,split,threshold,accuracy,precision,recall,f1,roc_auc,average_precision,positive_rate_pred,positive_rate_true,proba_mean,proba_min,proba_max
0,exp_03b_catboost,train,0.48,0.870748,0.865079,0.981982,0.919831,0.882132,0.937994,0.857143,0.755102,0.519445,0.474495,0.540978
1,exp_03b_catboost,validation,0.48,0.781250,0.758621,1.000000,0.862745,0.752273,0.839897,0.906250,0.687500,0.525895,0.474495,0.540978
2,exp_04b_mlp,train,0.36,0.761905,0.760274,1.000000,0.863813,0.703203,0.855167,0.993197,0.755102,0.553723,0.350973,0.828392
3,exp_04b_mlp,validation,0.36,0.718750,0.709677,1.000000,0.830189,0.540909,0.762526,0.968750,0.687500,0.556513,0.353370,0.783924


In [43]:
def golden_prediction_diagnostics(
    name: str,
    pred: pd.DataFrame,
    golden: pd.DataFrame,
) -> pd.DataFrame:
    merged = golden[["news_id", "novelty_label"]].copy()
    merged["news_id"] = merged["news_id"].astype(str)

    pred_part = pred[["news_id", "novelty_label", "p_significant"]].copy()
    pred_part["news_id"] = pred_part["news_id"].astype(str)

    merged = merged.merge(
        pred_part,
        on="news_id",
        how="inner",
        suffixes=("_true", "_pred"),
    )

    eval_mask = merged["novelty_label_true"].fillna("").astype(str).str.strip() != ""
    merged = merged[eval_mask].copy()

    merged["is_significant_true"] = (
        merged["novelty_label_true"].eq("significant").astype(int)
    )
    merged["is_significant_pred"] = (
        merged["novelty_label_pred"].eq("significant").astype(int)
    )

    return pd.DataFrame([{
        "model": name,
        "rows": len(merged),
        "positive_rate_true": merged["is_significant_true"].mean(),
        "positive_rate_pred": merged["is_significant_pred"].mean(),
        "p_mean": merged["p_significant"].mean(),
        "p_min": merged["p_significant"].min(),
        "p_max": merged["p_significant"].max(),
        "p_std": merged["p_significant"].std(),
    }])


golden_diag = pd.concat(
    [
        golden_prediction_diagnostics("exp_00b", pred_00b, golden),
        golden_prediction_diagnostics("exp_03b", pred_03b, golden),
        golden_prediction_diagnostics("exp_04b", pred_04b, golden),
    ],
    ignore_index=True,
)

golden_diag

,model,rows,positive_rate_true,positive_rate_pred,p_mean,p_min,p_max,p_std
0,exp_00b,89,0.820225,0.820225,0.612220,0.403847,0.771617,0.102653
1,exp_03b,89,0.820225,0.719101,0.836835,0.018273,0.998273,0.292551
2,exp_04b,89,0.820225,0.842697,0.549185,0.383903,0.849525,0.104686


In [44]:
# ---------------------------------------------------------------------
# Diagnose bronze effective contribution
# ---------------------------------------------------------------------

print("Silver full rows:", len(train_frame))
print("Bronze train rows:", len(bronze_train_frame))

print("\nSilver label distribution:")
display(train_frame["is_significant"].value_counts(normalize=True))

print("\nBronze label distribution:")
display(bronze_train_frame["is_significant"].value_counts(normalize=True))

print("\nBronze labels:")
display(bronze_train_frame["bronze_label"].value_counts())

print("\nBronze confidence:")
display(bronze_train_frame["confidence"].value_counts())

print("\nEffective bronze weight by label:")
display(
    bronze_train_frame
    .groupby(["bronze_label", "confidence"])
    .agg(
        rows=("news_id", "size"),
        weight_sum=("sample_weight", "sum"),
        weight_mean=("sample_weight", "mean"),
    )
    .reset_index()
)

print("\nTotal effective weights:")
print("silver full weight:", float(len(train_frame)))
print("bronze weight:", float(bronze_train_frame["sample_weight"].sum()))
print("bronze / silver weight ratio:", float(bronze_train_frame["sample_weight"].sum() / len(train_frame)))

Silver full rows: 179
Bronze train rows: 2044

Silver label distribution:


is_significant
1    0.743017
0    0.256983
Name: proportion, dtype: float64


Bronze label distribution:


is_significant
1    0.779354
0    0.220646
Name: proportion, dtype: float64


Bronze labels:


...


Bronze confidence:


confidence
high      1211
medium     833
Name: count, dtype: int64


Effective bronze weight by label:


,bronze_label,confidence,rows,weight_sum,weight_mean
0,duplicate,high,10,0.50,0.05
1,duplicate,medium,9,0.18,0.02
2,minor,high,119,5.95,0.05
3,minor,medium,313,6.26,0.02
4,significant,high,1082,54.10,0.05
5,significant,medium,511,10.22,0.02



Total effective weights:
silver full weight: 179.0
bronze weight: 77.21000000000001
bronze / silver weight ratio: 0.43134078212290505


In [45]:
# ---------------------------------------------------------------------
# Final current-best pipeline benchmark from standalone scripts
# ---------------------------------------------------------------------
# This cell runs the notebook-independent final pipeline implementation.
#
# It benchmarks the selected current-best pipeline on 10 000 rows from
# lenta_clean_news.csv:
#
# raw/clean news
#   -> data preparation
#   -> BAAI/bge-m3 embeddings
#   -> temporal graph clustering
#   -> legacy novelty features
#   -> CatBoost/fallback novelty prediction
#   -> CSV + timing JSON
# ---------------------------------------------------------------------

import json
import shutil
import subprocess
import sys
from pathlib import Path

import pandas as pd

# ---------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------

SCRIPTS_PATH = PROJECT_ROOT / "scripts"

DATA_PATH = PROJECT_ROOT / "data"

ARTIFACTS_PATH = DATA_PATH / "artifacts"

FINAL_BENCHMARK_SCRIPT = SCRIPTS_PATH / "benchmark_final_pipeline.py"

FINAL_PIPELINE_SRC_DIR = SRC_DIR / "final_pipeline"

INPUT_CLEAN_NEWS_PATH = DATA_PATH  / "prepared" / "lenta_clean_news.csv"

FINAL_MODEL_DIR = ARTIFACTS_PATH / "models" / "significance_model"
FINAL_MODEL_PATH = FINAL_MODEL_DIR / "catboost_is_significant.joblib"
FINAL_MODEL_CONFIG_PATH = FINAL_MODEL_DIR / "semantic_significance_model_config.json"

FINAL_BENCHMARK_OUTPUT_DIR = (
    ARTIFACTS_PATH / "final_pipeline_benchmark"
)

N_BENCHMARK_ROWS = 10_000
BENCHMARK_RANDOM_STATE = 42


print("PROJECT_ROOT:", PROJECT_ROOT)
print("FINAL_PIPELINE_SRC_DIR exists:", FINAL_PIPELINE_SRC_DIR.exists())
print("FINAL_BENCHMARK_SCRIPT exists:", FINAL_BENCHMARK_SCRIPT.exists())
print("INPUT_CLEAN_NEWS_PATH exists:", INPUT_CLEAN_NEWS_PATH.exists())
print("FINAL_MODEL_PATH exists:", FINAL_MODEL_PATH.exists())
print("FINAL_MODEL_CONFIG_PATH exists:", FINAL_MODEL_CONFIG_PATH.exists())


if not FINAL_PIPELINE_SRC_DIR.exists():
    raise FileNotFoundError(
        "src/final_pipeline not found. "
        "Unzip final_current_best_pipeline_scripts.zip into the project root first."
    )

if not FINAL_BENCHMARK_SCRIPT.exists():
    raise FileNotFoundError(
        "scripts/benchmark_final_pipeline.py not found. "
        "Unzip final_current_best_pipeline_scripts.zip into the project root first."
    )

if not INPUT_CLEAN_NEWS_PATH.exists():
    raise FileNotFoundError(f"Input clean news file not found: {INPUT_CLEAN_NEWS_PATH}")


# ---------------------------------------------------------------------
# Optional safety: export current model from notebook memory if artifact is missing
# ---------------------------------------------------------------------

FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

if not FINAL_MODEL_PATH.exists():
    if "current_model" not in globals():
        raise FileNotFoundError(
            f"Final model artifact not found: {FINAL_MODEL_PATH}. "
            "Also current_model is not available in notebook memory."
        )

    import joblib

    model_object = getattr(current_model, "model", current_model)

    joblib.dump(model_object, FINAL_MODEL_PATH)

    print("Saved current_model to:", FINAL_MODEL_PATH)

if not FINAL_MODEL_CONFIG_PATH.exists():
    # Minimal config compatible with final_pipeline.novelty.FinalNoveltyModel.
    config_payload = {
        "threshold": 0.42,
        "duplicate_threshold": 0.90,
        "review_margin": 0.10,
        "feature_columns": [
            "position_in_cluster",
            "cluster_size_so_far",
            "days_since_previous",
            "days_since_cluster_start",
            "max_prev_similarity",
            "mean_prev_similarity",
            "min_prev_similarity",
            "top2_mean_similarity",
            "top3_mean_similarity",
            "last_prev_similarity",
            "previous_centroid_similarity",
            "previous_centroid_distance",
            "title_jaccard_max",
            "text_jaccard_max",
            "shared_numbers_count",
            "new_numbers_count",
            "title_length",
            "text_length",
        ],
        "first_item_fallback_enabled": True,
        "fallback_window_days": 30,
        "fallback_similarity_threshold": 0.78,
        "fallback_max_previous_candidates": 10,
    }

    FINAL_MODEL_CONFIG_PATH.write_text(
        json.dumps(config_payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print("Saved model config to:", FINAL_MODEL_CONFIG_PATH)


# ---------------------------------------------------------------------
# Run benchmark
# ---------------------------------------------------------------------

cmd = [
    sys.executable,
    str(FINAL_BENCHMARK_SCRIPT),
    "--project-root",
    str(PROJECT_ROOT),
    "--input",
    str(INPUT_CLEAN_NEWS_PATH),
    "--output-dir",
    str(FINAL_BENCHMARK_OUTPUT_DIR),
    "--model-path",
    str(FINAL_MODEL_PATH),
    "--model-config",
    str(FINAL_MODEL_CONFIG_PATH),
    "--n-rows",
    str(N_BENCHMARK_ROWS),
    "--random-state",
    str(BENCHMARK_RANDOM_STATE),
    "--device",
    "cuda",
    "--force-recompute-embeddings",
]

print("Running command:")
print(" ".join(f'"{part}"' if " " in part else part for part in cmd))

completed = subprocess.run(
    cmd,
    cwd=str(PROJECT_ROOT),
    text=True,
    capture_output=True,
)

print("Return code:", completed.returncode)

print("\n--- STDOUT ---")
print(completed.stdout)

print("\n--- STDERR ---")
print(completed.stderr)

if completed.returncode != 0:
    raise RuntimeError("Final pipeline benchmark failed. See STDERR above.")


# ---------------------------------------------------------------------
# Read and display benchmark result
# ---------------------------------------------------------------------

timing_path = FINAL_BENCHMARK_OUTPUT_DIR / f"final_benchmark_{N_BENCHMARK_ROWS}_timing.json"
summary_path = FINAL_BENCHMARK_OUTPUT_DIR / f"final_benchmark_{N_BENCHMARK_ROWS}_summary.md"
predictions_path = FINAL_BENCHMARK_OUTPUT_DIR / f"final_predictions_{N_BENCHMARK_ROWS}.csv"

print("Timing JSON:", timing_path)
print("Summary MD:", summary_path)
print("Predictions CSV:", predictions_path)

benchmark_payload = json.loads(timing_path.read_text(encoding="utf-8"))

print("\nBenchmark summary:")
print("n_rows:", benchmark_payload["n_rows"])
print("total_seconds:", round(benchmark_payload["total_seconds"], 2))
print("rows_per_second:", round(benchmark_payload["rows_per_second"], 2))
print("rows_per_minute:", round(benchmark_payload["rows_per_second"] * 60, 2))

print("\nStage timings:")
display(
    pd.DataFrame(
        [
            {"stage": stage, "seconds": seconds}
            for stage, seconds in benchmark_payload["stage_seconds"].items()
        ]
    ).sort_values("seconds", ascending=False)
)

print("\nDiagnostics:")
display(pd.DataFrame([benchmark_payload["diagnostics"]]))

print("\nPrediction sample:")
display(pd.read_csv(predictions_path).head())

PROJECT_ROOT: E:\ML\Projects\Git\news-flow-analysis
FINAL_PIPELINE_SRC_DIR exists: True
FINAL_BENCHMARK_SCRIPT exists: True
INPUT_CLEAN_NEWS_PATH exists: True
FINAL_MODEL_PATH exists: True
FINAL_MODEL_CONFIG_PATH exists: True
Running command:
e:\Mamba\envs\ml-gpu\python.exe E:\ML\Projects\Git\news-flow-analysis\scripts\benchmark_final_pipeline.py --project-root E:\ML\Projects\Git\news-flow-analysis --input E:\ML\Projects\Git\news-flow-analysis\data\prepared\lenta_clean_news.csv --output-dir E:\ML\Projects\Git\news-flow-analysis\data\artifacts\final_pipeline_benchmark --model-path E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\significance_model\catboost_is_significant.joblib --model-config E:\ML\Projects\Git\news-flow-analysis\data\artifacts\models\significance_model\semantic_significance_model_config.json --n-rows 10000 --random-state 42 --device cuda --force-recompute-embeddings
Return code: 0

--- STDOUT ---
Benchmark rows: 10000
Predictions: E:\ML\Projects\Git\news-flo

,stage,seconds
4,encode_embeddings,189.377518
6,predict_novelty,21.107206
2,load_models,13.434150
0,load_data,5.917446
5,cluster_news,1.084784
3,prepare_data,0.701861
7,save_outputs,0.277213
1,choose_benchmark_rows,0.046316



Diagnostics:


,n_rows,n_clusters,edge_count,similarity_checks,story_threshold,story_window_days,embedding_model_name,novelty_threshold,duplicate_threshold,review_margin
0,10000,9858,146,222046,0.82,14,BAAI/bge-m3,0.42,0.9,0.1



Prediction sample:


,news_id,published_at,topic,title,text,cluster_id,novelty_label,needs_review,comment,p_significant,max_prev_similarity
0,23,1999-09-07,Россия,Еще одно назначение Путина,Еще одно назначение Владимира Путина: президен...,story_000000,NaN,False,first item in cluster; no fallback candidates,NaN,NaN
1,53,1999-09-08,Мир,"Судебный процесс ""Ford против Интернета"" проиг...",Судебный процесс между компанией Ford Motor Co...,story_000001,NaN,False,first item in cluster; no fallback candidates,NaN,NaN
2,70,1999-09-09,Россия,Якушкин: военной цензуры в России не будет,"""Общественность должна иметьмаксимально полную...",story_000002,NaN,False,first item in cluster; no fallback candidates,NaN,NaN
3,101,1999-09-10,Мир,"Cкандал с ""отмывкой денег"" не отразится на отн...","Сегодня в Окленде (Новая Зеландия), в рамках с...",story_000003,NaN,False,first item in cluster; no fallback candidates,NaN,NaN
4,102,1999-09-10,Россия,С 1 ноября до аэропорта Домодедово можно будет...,Московская железная дорога (МЖД) планирует с 1...,story_000004,NaN,False,first item in cluster; no fallback candidates,NaN,NaN


### Benchmark финального pipeline

Финальная воспроизводимая версия pipeline была вынесена из ноутбуков в отдельный пакет `src/final_pipeline` и запускается через standalone-скрипт `scripts/benchmark_final_pipeline.py`. Это позволяет использовать выбранную конфигурацию не только в исследовательском ноутбуке, но и как основу для последующей интеграции в сервис.

На cold benchmark для 10 000 записей из `lenta_clean_news.csv` полный pipeline от загрузки данных до сохранения predictions занял 321.76 секунды, то есть около 5 минут 22 секунд. Средняя скорость составила 31.08 новости/сек, или около 1 865 новостей/мин.

Основное время занимает построение embeddings через `BAAI/bge-m3`: 273.09 секунды, около 85% общего времени. Остальные этапы значительно дешевле: кластеризация заняла 1.21 секунды, построение признаков и novelty prediction — 23.83 секунды, загрузка моделей — 16.34 секунды.

Этот результат показывает, что bottleneck pipeline — именно embedding-модель, а не temporal graph clustering или novelty classifier. Для production-сценария это означает, что сервис должен держать embedding-модель в памяти и работать инкрементально: считать embedding только для новых документов, после чего сравнивать их с активными story clusters в пределах topic/time window. Batch/backfill-обработка десятков тысяч документов также выглядит реалистичной на одной локальной GPU.

## Итоговое описание проделанной работы и выводы

В этом ноутбуке была проведена серия экспериментов по улучшению уже существующего рабочего pipeline для задачи Semantic News Novelty. Важно, что точкой сравнения была не наивная baseline-модель, а уже доработанная текущая рабочая конфигурация: `BAAI/bge-m3` embeddings, topic-aware temporal graph clustering, временное окно для сюжетов, настроенные thresholds, legacy novelty features и final novelty classifier с fallback-логикой для первых элементов кластеров.

Целью текущей серии экспериментов было проверить, можно ли устойчиво улучшить этот pipeline за счёт изменений в кластеризации, финальном классификаторе, признаках, дополнительной weak supervision-разметке или embedding-модели.

### 1. Воспроизведение текущей рабочей модели

Сначала была воспроизведена текущая рабочая конфигурация pipeline (`exp_00b_reproduced_old_pipeline`). Она стала reference model для дальнейших сравнений.

Эта конфигурация включает:

- построение embeddings через `BAAI/bge-m3`;
- кластеризацию новостей через similarity graph внутри `topic`;
- ограничение связей временным окном `STORY_WINDOW_DAYS = 14`;
- порог сюжетной похожести `STORY_THRESHOLD = 0.82`;
- legacy-набор признаков для novelty detection;
- final CatBoost/fallback novelty classifier;
- fallback-логику для первых элементов кластеров.

Reference model показала устойчивое качество на human golden set:

- `pairwise_f1 ≈ 0.7697`;
- `false_merge_rate = 0.0`;
- `significant_f1 ≈ 0.8356`.

Эта модель была принята как текущий best/current pipeline, который дальнейшие эксперименты пытались улучшить.

### 2. Эксперименты с кластеризацией

Были проверены несколько альтернативных вариантов кластеризации:

- embedding projection;
- mutual kNN clustering;
- temporal decay clustering;
- online lifecycle clustering.

Идея состояла в том, чтобы улучшить выделение сюжетов и уменьшить проблемы threshold graph clustering, например chaining-effect или слишком грубую привязку новостей.

Однако эти варианты не дали устойчивого улучшения. Часть экспериментов резко ухудшила кластеризацию: `pairwise_f1` падал почти до нуля, а `false_merge_rate` становился неприемлемо высоким. Это показало, что текущая topic-aware temporal graph clustering-логика, несмотря на простоту, лучше контролирует риск ложного объединения разных сюжетов.

### 3. Эксперименты с final novelty classifier

Далее кластеризация была зафиксирована, чтобы честно сравнить только final novelty classifier. Были проверены:

- заново обученный CatBoost;
- MLP;
- MLP с более консервативным threshold;
- Logistic Regression;
- CatBoost с дополнительными derived features.

Большинство вариантов ухудшили итоговое качество относительно reference pipeline. MLP формально дал минимальный прирост `significant_f1`, но этот прирост оказался слишком малым и диагностически неубедительным: модель стала немного агрессивнее предсказывать класс `significant`, а не явно лучше различать смысловую новизну. При более осторожном threshold качество MLP резко ухудшилось.

CatBoost с derived features, Logistic Regression и заново обученный CatBoost стали более консервативными и потеряли recall важных обновлений. Поэтому простая замена final classifier не дала устойчивого улучшения.

### 4. Эксперимент с bronze weak labels

Была подготовлена дополнительная bronze-разметка. Для этого были взяты новости из неиспользованных годов Lenta.ru, исключены пересечения с golden/silver/current candidate pool, после чего новости были кластеризованы тем же temporal graph-подходом и размечены как weak labels.

Bronze-разметка была добавлена в обучение final classifier с confidence-based sample weights. Первый запуск показал, что effective weight bronze оказался более чем в 4 раза выше веса silver. В результате weak labels фактически задавили более качественную silver-разметку, и качество на human golden set заметно ухудшилось.

После снижения весов bronze ухудшение стало меньше, но устойчивого прироста относительно reference model всё равно получить не удалось. Это показало, что текущую bronze v1-разметку нельзя напрямую смешивать с silver как полноценный обучающий источник. Для пользы bronze требуется более строгая фильтрация, калибровка confidence или ручная проверка части weak labels.

### 5. Эксперимент с fine-tuning BGE-M3

Также была проверена гипотеза, что `BAAI/bge-m3` можно улучшить для задачи новостных сюжетов через дообучение на русскоязычных парафразах.

Проверка проводилась в два этапа:

1. sanity-check на paraphrase retrieval;
2. downstream threshold sweep на golden candidate pool.

Sanity-check не показал улучшения: у fine-tuned модели немного снизились `Recall@1` и `MRR@10` относительно базовой `BAAI/bge-m3`.

Downstream-проверка также показала ухудшение. При безопасном пороге `story_threshold = 0.82`, где `false_merge_rate = 0`, базовая `BAAI/bge-m3` достигает `pairwise_f1 ≈ 0.7697`, а fine-tuned BGE-M3 — около `0.7059`.

При снижении threshold fine-tuned модель повышала recall связей внутри сюжета, но начинала давать false merge и всё равно не превосходила базовую BGE-M3. Поэтому fine-tuned BGE-M3 не используется как основной encoder.

### 6. Вынос финального pipeline в воспроизводимый код

После выбора финальной конфигурации pipeline был вынесен из ноутбука в отдельный воспроизводимый пакет:

- `src/final_pipeline/`;
- `scripts/run_final_pipeline.py`;
- `scripts/benchmark_final_pipeline.py`.

Это позволяет запускать выбранную модель без ноутбука и использовать её как основу для последующей интеграции в сервис.

Финальный pipeline принимает clean-like news dataset, выполняет подготовку данных, строит embeddings, кластеризует новости, считает novelty features, применяет final novelty classifier и сохраняет predictions в CSV.

Таким образом, выбранная конфигурация стала не только исследовательским результатом в ноутбуке, но и воспроизводимым pipeline, доступным для дальнейшей сервисной интеграции.

### 7. Benchmark скорости финального pipeline

Для оценки производительности был выполнен cold benchmark на 10 000 записей из `lenta_clean_news.csv`.

Полный pipeline от загрузки данных до сохранения predictions занял:

- `321.76` секунды;
- около `5 минут 22 секунд`;
- `31.08` новости/сек;
- около `1 865` новостей/мин.

Разбивка по стадиям:

- `encode_embeddings`: `273.09` сек;
- `predict_novelty`: `23.83` сек;
- `load_models`: `16.34` сек;
- `load_data`: `6.20` сек;
- `cluster_news`: `1.21` сек;
- `prepare_data`: `0.71` сек;
- `save_outputs`: `0.33` сек.

Основной bottleneck — построение embeddings через `BAAI/bge-m3`, которое занимает около 85% общего времени. Сама temporal graph clustering-часть оказалась дешёвой: около 1.2 секунды на 10 000 новостей.

Это важный инженерный вывод: после получения embeddings downstream-часть pipeline работает быстро. Для production-сценария это означает, что embedding-модель должна быть заранее загружена в память, а основной рабочий режим должен быть инкрементальным: для новой новости считается embedding, после чего она сравнивается только с активными story clusters в пределах topic/time window.

Отдельно ранее был получен похожий результат на batch/backfill-сценарии: embeddings для двух годовых блоков по 12 000 новостей были посчитаны примерно за 19 минут суммарно на локальной машине с RTX 4070. Это подтверждает, что batch-обработка десятков тысяч новостей технически реалистична даже на одной локальной GPU.

### Общий итог

Проведённые эксперименты показали, что текущая рабочая модель остаётся наиболее устойчивой конфигурацией. Ни новая кластеризация, ни замена final classifier, ни дополнительные признаки, ни bronze weak labels, ни fine-tuning embedding-модели не дали стабильного улучшения на human golden set при сохранении низкого `false_merge_rate`.

Это не означает, что дальнейшее улучшение невозможно. Скорее, эксперименты показывают, что качество сейчас ограничено не выбором отдельного классификатора, а более фундаментальными факторами:

- малым размером human golden set;
- различием между silver/bronze и human-разметкой;
- сложностью границы между `minor` и `significant`;
- необходимостью отдельно моделировать задачу `attach / new_story`;
- риском false merge при слишком агрессивной семантической кластеризации;
- чувствительностью novelty detection к качеству story clustering.

Поэтому в качестве финальной рабочей конфигурации выбран воспроизведённый current best pipeline. Он показал лучшее устойчивое качество среди проверенных вариантов, был вынесен в отдельный воспроизводимый код и дополнительно проверен через benchmark скорости.

Дальнейшие улучшения стоит искать не в простой замене модели, а в расширении и уточнении human/silver-разметки, более строгой калибровке weak supervision, отдельном обучении attach/new_story-модели и более аккуратной постановке задачи значимости обновлений внутри уже найденного сюжета.